# Know Your Data: Exploring and Preparing Image Datasets at the Wildwood Farm Park

**Objective:** run the setup, meet the park, and see the three things you will be able to do at the end.

<table width="100%"><tr><td bgcolor="#0B6E79"><font color="#FFFFFF">
<font size="5"><b>Know Your Data: Exploring and Preparing Image Datasets
at the Wildwood Farm Park</b></font><br><br>
<b>At the end of this notebook you will be able to:</b>
<ol>
<li>Inspect an image dataset qualitatively and quantitatively, and say whether it fits a task.</li>
<li>Find and fix common dataset errors (duplicates, flipped/rotated copies, broken files, bad labels or
masks) before they leak into training.</li>
<li>Choose a split and a metric that match the use case, and explain how imbalance, object size and leakage
look different for classification, detection and segmentation.</li>
</ol>
Questions are marked 🟢 beginner · 🟡 intermediate · 🔴 advanced (optional).
</font></td></tr></table>

**Before you start:** you need a Google account (this notebook runs in Colab, a CPU runtime is enough),
Python basics, and the lecture on image datasets. No model is trained here; that is next week.
The notebook follows the first part of Karpathy's
[*A Recipe for Training Neural Networks*](https://karpathy.github.io/2019/04/25/recipe/):
know your data before you touch a model.

## 0. Setup & the park

**The Wildwood Farm Park** has cameras in four places, and it wants three things from them.

<table width="100%"><tr>
<td align="center" width="25%"><font size="6">🐑🐴</font><br><b>Farm paddock</b><br>sheep, horses</td>
<td align="center" width="25%"><font size="6">🐦</font><br><b>Aviary pond</b><br>birds</td>
<td align="center" width="25%"><font size="6">🦓</font><br><b>Savanna enclosure</b><br>zebras</td>
<td align="center" width="25%"><font size="6">🐱🐶</font><br><b>Visitor yard</b><br>cats and dogs wander
through</td>
</tr></table>

1. **Recognise** (classification): "who is in front of this camera?" The case that matters is **a dog in the
   sheep paddock**, which triggers an alert. Missing that dog is worse than a false alarm.
2. **Count** (detection): a daily headcount of sheep, zebras and birds. Far-away birds are tiny.
3. **Outline** (segmentation): how much of each enclosure each animal uses. Birds cover few pixels.

The park has no training data yet, so we practise on a **Play Dataset**: COCO 2017 photos of the same six
animals (cat, dog, bird, horse, sheep, zebra), each photo showing one animal class. Everything you learn
here is a checklist you will reuse on your own project dataset in the last section.

**Run the next cell once.** It installs the one package Colab lacks, fixes the random seed, sets the
notebook's colours and defines `use()`, the helper that lets you write small functions without ever
blocking "Run all".

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import os
import random
import sys
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt  # noqa: F401  (every later section plots with plt)
import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated tests: tiny data, CPU, seconds. Never set this by hand.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:  # Jupytext turns the commented magic below into a real `%pip install -q imagehash` (N-021)
    pass
    %pip install -q imagehash

# --- Theme (SPEC L-003): white background, dark teal text, light grid, one colour per class everywhere ---
CLASSES = ["cat", "dog", "bird", "horse", "sheep", "zebra"]
CLASS_COLORS = {
    "cat": "#E07A5F",
    "dog": "#3D405B",
    "bird": "#F2CC8F",
    "horse": "#81B29A",
    "sheep": "#9C89B8",
    "zebra": "#2A9D8F",
}
INK = "#21434A"
mpl.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "text.color": INK,
        "axes.labelcolor": INK,
        "axes.edgecolor": INK,
        "axes.titlecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.grid": True,
        "grid.color": "#DDE5E7",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": mpl.cycler(color=[CLASS_COLORS[c] for c in CLASSES]),
        "figure.dpi": 100,
        "font.size": 11,
    }
)

# --- Results every section writes into (SPEC §9) ---
RESULTS = {}
RESULTS["class_colors"] = dict(CLASS_COLORS)


# --- use(): run a learner's function next to the reference solution (N-029) ---
def _is_stub(fn) -> bool:
    """True if fn's body is only a docstring plus `...` or `pass` (the learner has not written it yet)."""
    try:
        node = ast.parse(textwrap.dedent(inspect.getsource(fn))).body[0]
        body = node.body
    except (OSError, TypeError, SyntaxError, IndexError, AttributeError):
        return False
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, "value", None), str):
        body = body[1:]  # skip the docstring
    return all(
        isinstance(n, ast.Pass) or (isinstance(n, ast.Expr) and getattr(n.value, "value", None) is Ellipsis)
        for n in body
    )


def _same(a, b) -> bool:
    if hasattr(a, "equals"):  # pandas
        return bool(a.equals(b))
    try:
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            return bool(np.array_equal(a, b))
        return bool(a == b)
    except Exception:
        return False


def _short(x, n=60) -> str:
    s = repr(x)
    return s if len(s) <= n else s[: n - 1] + "…"


def _diff(mine, ref) -> str:
    """One short line on how `mine` differs from `ref` (e.g. 'cat: 12 vs 14')."""
    if isinstance(mine, dict) and isinstance(ref, dict):
        keys = list(ref) + [k for k in mine if k not in ref]
        bad = [k for k in keys if not _same(mine.get(k, "—"), ref.get(k, "—"))]
        return ", ".join(f"{k}: {_short(mine.get(k, '—'))} vs {_short(ref.get(k, '—'))}" for k in bad[:4])
    if isinstance(mine, (list, tuple, set)) and isinstance(ref, (list, tuple, set)):
        try:
            extra, missing = set(mine) - set(ref), set(ref) - set(mine)
            parts = []
            if missing:
                parts.append(f"missing {_short(sorted(missing, key=str))}")
            if extra:
                parts.append(f"extra {_short(sorted(extra, key=str))}")
            return " · ".join(parts) or f"same items, different order or length ({len(mine)} vs {len(ref)})"
        except TypeError:
            pass
    return f"{_short(mine)} vs {_short(ref)}"


def use(fn, *args, reference, **kwargs):
    """Run `fn(*args, **kwargs)` next to `reference(*args, **kwargs)`; ALWAYS return the reference's result.

    - stub (body is `...`) or NotImplementedError → "using the reference solution";
    - otherwise ✅ when both agree, ❌ with a short diff when they do not.
    The notebook continues with the reference either way, so "Run all" never breaks on your code.
    """
    expected = reference(*args, **kwargs)
    name = getattr(fn, "__name__", "your function")
    if _is_stub(fn):
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    try:
        mine = fn(*args, **kwargs)
    except NotImplementedError:
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    except Exception as err:  # a learner's bug must not stop the notebook
        print(f"❌ {name} raised {type(err).__name__}: {err} · using the reference solution.")
        return expected
    if _same(mine, expected):
        print(f"✅ {name} matches the reference solution.")
    else:
        print(f"❌ {name} differs from the reference: {_diff(mine, expected)} · using the reference.")
    return expected


print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · seed: {SEED} · imagehash: ", end="")
try:
    import imagehash  # noqa: F401

    print("ok")
except ImportError:
    print("missing (run this cell again)")

### How the ✍️ cells work

About four times in this notebook you write a small function. Each one comes with its signature and a
`use(my_function, …, reference=…)` line that compares your result with the reference solution and
then continues with the reference, so an unfinished or wrong function never stops the notebook.
Run the cell below as is, then fill in the body and run it again to see the message change.

In [ ]:
def double(x):
    """Return twice x."""
    ...


def _ref_double(x):
    return 2 * x


result = use(double, 21, reference=_ref_double)
print("the notebook continues with:", result)

### The Play Dataset

The next cell fetches the annotation subset we prepared (`data/coco_animals_subset.json`, COCO 2017 val,
six animal classes, each image shows exactly one of them) and downloads the photos themselves from
`images.cocodataset.org`. **The first run takes a minute or two** (about 115 MB); afterwards the images
are cached and the cell is instant. The result is two tidy tables, `images` (one row per photo) and
`annotations` (one row per animal: its box and mask).

In [ ]:
# @title Data helpers (download with cache, build the tables) { display-mode: "form" }
import json
import urllib.error
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

# Where the committed manifests live. In Colab they are fetched from the repository; locally they are read
# from ./data. ⬇️ DATA_BASE_URL: change this if the repository moves or you work on a fork.
DATA_BASE_URL = "https://raw.githubusercontent.com/FHNW-VISED/eda-teaching-notebook/main/data/"
DATA_DIR = Path("data")
IMAGE_CACHE = DATA_DIR / "cache" / "coco"
SMOKE_IMAGES_PER_CLASS = 6


def _github_headers() -> dict:
    """Authorization header from the Colab secret GITHUB_TOKEN, if the repository is private (SPEC §5)."""
    if not IN_COLAB:
        return {}
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return {"Authorization": f"token {token}"} if token else {}
    except Exception:
        return {}


def download_file(url: str, dest: Path, headers: dict | None = None, attempts: int = 3) -> Path | None:
    """Download `url` to `dest` once; a half-finished download is written to `.part` first (N-019)."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(attempts):
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as fh:
                fh.write(resp.read())
            part.rename(dest)
            return dest
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == attempts - 1:
                part.unlink(missing_ok=True)
                return None
    return None


def fetch_manifest(name: str) -> Path:
    """Return the local path of a committed manifest, downloading it from DATA_BASE_URL if needed."""
    path = DATA_DIR / name
    if not path.exists():
        if download_file(DATA_BASE_URL + name, path, headers=_github_headers()) is None:
            raise RuntimeError(
                f"Could not download {name} from {DATA_BASE_URL}. Check DATA_BASE_URL, or add a GITHUB_TOKEN "
                "secret in Colab if the repository is private."
            )
    return path


def download_images(rows: list[dict], workers: int = 16) -> dict[int, Path]:
    """Download every image in `rows` ('id', 'coco_url', 'file_name') into IMAGE_CACHE; one progress line."""
    done, paths = 0, {}

    def one(row):
        return row["id"], download_file(row["coco_url"], IMAGE_CACHE / row["file_name"])

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for image_id, path in pool.map(one, rows):
            done += 1
            if path is not None:
                paths[image_id] = path
            print(f"\r  images ready: {done}/{len(rows)}", end="", flush=True)
    print()
    return paths


def build_tables(coco: dict, paths: dict[int, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Tidy tables from a COCO-format dict: `images` (one row per photo), `annotations` (one per animal)."""
    cat_name = {c["id"]: c["name"] for c in coco["categories"]}
    lic_name = {lic["id"]: lic["name"] for lic in coco.get("licenses", [])}
    anns = pd.DataFrame(
        {
            "id": a["id"],
            "image_id": a["image_id"],
            "class": cat_name[a["category_id"]],
            "bbox": a["bbox"],
            "area": a["area"],
            "segmentation": a["segmentation"],
            "iscrowd": a["iscrowd"],
        }
        for a in coco["annotations"]
        if a["image_id"] in paths
    )
    per_image_class = anns.groupby("image_id")["class"].first()
    n_animals = anns.groupby("image_id").size()
    imgs = pd.DataFrame(
        {
            "id": im["id"],
            "file": im["file_name"],
            "path": str(paths[im["id"]]),
            "width": im["width"],
            "height": im["height"],
            "class": per_image_class.get(im["id"]),
            "n_animals": int(n_animals.get(im["id"], 0)),
            "license": lic_name.get(im["license"], str(im["license"])),
        }
        for im in coco["images"]
        if im["id"] in paths
    )
    imgs = imgs.sort_values("id").reset_index(drop=True)
    anns = anns.sort_values("id").reset_index(drop=True)
    return imgs, anns

In [ ]:
# Load the manifest, pick the images (all of them, or 6 per class in smoke mode), download, build the tables.
COCO_RAW = json.loads(fetch_manifest("coco_animals_subset.json").read_text())
_cat_name = {c["id"]: c["name"] for c in COCO_RAW["categories"]}
_image_class = {a["image_id"]: _cat_name[a["category_id"]] for a in COCO_RAW["annotations"]}

_rows = COCO_RAW["images"]
if SMOKE:  # same code path, fewer images (N-014)
    _rng = random.Random(SEED)
    _rows = []
    for c in CLASSES:
        _of_class = [im for im in COCO_RAW["images"] if _image_class[im["id"]] == c]
        _rows += sorted(_rng.sample(_of_class, SMOKE_IMAGES_PER_CLASS), key=lambda im: im["id"])

print(f"{len(_rows)} images to fetch from images.cocodataset.org (cached in {IMAGE_CACHE}/)")
_paths = download_images(_rows)
if len(_paths) < len(_rows):
    print(f"  ⚠️ {len(_rows) - len(_paths)} images could not be downloaded and are left out; run again.")

images, annotations = build_tables(COCO_RAW, _paths)
RESULTS["n_images"] = int(len(images))

_counts = Counter(images["class"])
print(f"images: {len(images)} · animals (annotations): {len(annotations)}")
print("per class: " + " · ".join(f"{c} {_counts[c]}" for c in CLASSES))
images.head()

Each row of `images` is one photo with its size, its class and how many animals it holds; `annotations`
has one row per animal with its box and mask. Look at the `n_animals` column: several photos hold more
than one animal, which already hints at a question we will meet in section 4.

## 1. Image datasets

**Objective:** know where image datasets come from and what makes a good one, then hold the Play Dataset
against that checklist.

The park has no training data yet. Before anyone mounts cameras and hires annotators, the first question is
always: *does a usable dataset already exist?* These are the places to look, and the tools to label your own
images when nothing fits.

**Find a dataset:**
[Kaggle datasets](https://www.kaggle.com/datasets) ·
[Google Dataset Search](https://datasetsearch.research.google.com) ·
[V7 open datasets](https://www.v7labs.com/open-datasets) ·
[Roboflow Universe](https://universe.roboflow.com) ·
[LILA](https://lila.science) (wildlife and camera-trap data) ·
[V7 blog on computer-vision datasets](https://www.v7labs.com/blog) ·
[torchvision datasets](https://pytorch.org/vision/stable/datasets.html) (the classics, one import away).

**Label your own images:**
[Supervisely](https://supervisely.com) ·
[V7](https://www.v7labs.com) (free for education) ·
[Roboflow Annotate](https://roboflow.com/annotate) ·
[Label Studio](https://labelstud.io) (open source).

Our Play Dataset is a slice of [COCO](https://cocodataset.org), and the woodland trail-camera images of
section 6 come from LILA. Part of "finding" a dataset is reading its licence: COCO's annotations are
CC BY 4.0, but every photo keeps its own Flickr licence, which is why this notebook downloads the images at
runtime instead of shipping them.

Having a dataset is not the same as having a *good* one. This is the checklist from the lecture:

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · What makes a good dataset</b><br>
<ul>
<li><b>Does it fit the question?</b> The images, the classes and the setting must match the job you want the
model to do.</li>
<li><b>Enough samples:</b> a few hundred to a few thousand images, depending on how many classes you
have.</li>
<li><b>2–10 classes.</b></li>
<li><b>About 500×500 px.</b> Check the input size of the models you plan to use.</li>
<li><b>Classes, distributions, imbalance, bias.</b> Know them, and take them into account in training
(for example with a weighted loss).</li>
<li><b>Variability matches the target population.</b> Lighting, viewpoints, backgrounds and seasons in the
data should look like the ones the model will meet.</li>
<li><b>Labels are present and validated.</b></li>
<li><b>Objects are identifiable and separable by eye.</b> If you cannot tell them apart, the model will
struggle too.</li>
<li><b>What do experts and the literature say</b> about this data and this task?</li>
<li><b>Get a qualitative and a quantitative sense of the data</b> (Karpathy, 2019): look at many images,
then measure.</li>
</ul>
</font></td></tr></table>

The next cell goes through the checklist for the Play Dataset. Where the tables from section 0 can answer
a bullet it computes the answer (✅ fine, ⚠️ needs attention); where only your eyes can, it says so (👀)
and points to the section that will look.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run: the park wants to recognise, count and outline six animals from fixed cameras.
</font></td></tr></table>

**Q1.** 🟢 Which bullets do you expect the Play Dataset to pass (✅) as it is, and which one worries you
most for the dog-in-the-paddock alert? Write one line, then run the cell.

In [ ]:
# Q1  Apply the checklist to the Play Dataset. Every number comes from the `images` table of section 0.
n_images = len(images)
counts = images["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
n_classes = int((counts > 0).sum())
imbalance = counts.max() / max(counts.min(), 1)
median_w, median_h = int(images["width"].median()), int(images["height"].median())
share_labelled = float((images["n_animals"] > 0).mean())
share_several = float((images["n_animals"] > 1).mean())

checklist = [
    (
        "⚠️",
        "Fits the question?",
        "the park's six animals, but Flickr holiday photos, not fixed cameras → judge in section 2",
    ),
    (
        "✅" if n_images >= 300 else "⚠️",
        "A few hundred to a few thousand samples",
        f"{n_images} images" + ("" if n_images >= 300 else " (a tiny test sample; the full set has ~600)"),
    ),
    ("✅" if 2 <= n_classes <= 10 else "⚠️", "2–10 classes", f"{n_classes}: " + ", ".join(CLASSES)),
    (
        "✅" if min(median_w, median_h) >= 224 else "⚠️",
        "About 500×500 px",
        f"median {median_w}×{median_h} px; most classifiers take 224–512 px, so there is room to resize",
    ),
    (
        "⚠️" if imbalance > 2 else "✅",
        "Classes, distributions, imbalance, bias",
        " · ".join(f"{c} {counts[c]}" for c in CLASSES)
        + f" → largest / smallest = {imbalance:.1f}× (weighted loss? section 4)",
    ),
    ("👀", "Variability matches the target population", "lighting, distance, backgrounds: look in section 2"),
    (
        "✅" if share_labelled == 1 else "⚠️",
        "Labels present",
        f"{share_labelled:.0%} of images have at least one box and mask",
    ),
    (
        "⚠️",
        "Labels validated",
        "COCO-ReM (Singh et al., 2024) found ~2.4 % of val masks poor, plus duplicate masks → sections 5, 9",
    ),
    (
        "👀",
        "Objects identifiable and separable by eye",
        f"{share_several:.0%} of images hold several animals; check the grids in section 2",
    ),
    (
        "📚",
        "What experts and the literature say",
        "Lin et al. 2014 (COCO) and Singh et al. 2024 (COCO-ReM) document the data and its mask errors",
    ),
    ("📓", "Qualitative and quantitative sense (Karpathy 2019)", "sections 2–4 of this notebook"),
]
for status, bullet, finding in checklist:
    print(f"{status} {bullet}: {finding}")

Read the list top to bottom: the size, class count and resolution pass on their own, and everything that
needs attention points to a later section of this notebook, which is the order we will follow.

**Worked answer (Q1).** Sample size, number of classes and resolution pass without any work. The bullets
that matter for the alert are *fit* and *imbalance*: holiday photos of dogs are not a camera pointed at a
paddock, and on the full set the biggest class has more than twice the images of the smallest, so a model
can score well while ignoring a rare animal. Section 4 makes that concrete for the dog and section 7 picks
the metric for it.

## 2. Become one with the data

**Objective:** look at many images before computing anything, and write down what you see.

Section 1 ended with three 👀 bullets that no table can answer: does the data *fit* the park's cameras, does
its *variability* match what the cameras will see, and are the animals *identifiable by eye*? The only tool
for those is your eyes, and this is the step most people skip. Karpathy puts it first in his recipe:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1: Become one with the data</b><br>
<i>"The first step to training a neural net is to not touch any neural net code at all and instead begin by
thoroughly inspecting your data. This step is critical. I like to spend copious amount of time (measured in
units of hours) scanning through thousands of examples, understanding their distribution and looking for
patterns."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

We will not spend hours, but we will spend a few minutes. The first cell draws one row of photos per class,
each framed in its class colour (the same colour every plot in this notebook uses for that class). Scroll
slowly: look at the backgrounds, the distance to the animal, the lighting, and whether anything else is in
the photo.

In [ ]:
# @title Grid helpers { display-mode: "form" }
from IPython.display import display
from PIL import Image


def load_thumb(path, size: int = 256) -> Image.Image:
    """Open an image as a small RGB thumbnail (fast: JPEG decodes at reduced size)."""
    im = Image.open(path)
    im.draft("RGB", (size, size))
    im = im.convert("RGB")
    im.thumbnail((size, size))
    return im


def sample_rows(class_name: str, n: int) -> pd.DataFrame:
    """A fixed random sample of `n` rows of `images` for one class (seeded: re-runs show the same photos)."""
    of_class = images[images["class"] == class_name]
    n = min(n, len(of_class))
    idx = random.Random(SEED).sample(sorted(of_class.index), n)
    return of_class.loc[sorted(idx)]


def frame(ax, color: str, width: float = 4) -> None:
    """Draw a coloured frame around an image axis and hide ticks and grid."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(width)


def show_grid(class_name: str, n: int, cols: int = 6, label_animals: bool = True) -> None:
    """Show `n` photos of one class in a grid, framed in CLASS_COLORS[class_name]."""
    rows_df = sample_rows(class_name, n)
    n = len(rows_df)
    n_rows = max(1, -(-n // cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.2 * cols, 2.4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[class_name])
        if label_animals:
            plural = "s" if row["n_animals"] != 1 else ""
            ax.set_title(f"{row['n_animals']} {class_name}{plural}", fontsize=8)
    fig.suptitle(f"{class_name} · {n} of {int((images['class'] == class_name).sum())} photos", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_all_classes(n_per_class: int) -> None:
    """One row per class, `n_per_class` photos each, every row framed in its class colour."""
    fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(2.0 * n_per_class, 2.1 * len(CLASSES)))
    axes = np.atleast_2d(axes)
    for r, class_name in enumerate(CLASSES):
        rows_df = sample_rows(class_name, n_per_class)
        for c in range(n_per_class):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(rows_df):
                ax.axis("on")
                ax.imshow(load_thumb(rows_df.iloc[c]["path"]))
                frame(ax, CLASS_COLORS[class_name])
            if c == 0:
                ax.set_ylabel(class_name, color=CLASS_COLORS[class_name], fontsize=12, fontweight="bold")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run: the park's cameras are fixed, outdoors, and often far from the animal.
</font></td></tr></table>

**Q2.** 🟢 Which class do you expect to look *least* like a fixed park camera's view, and why? One line,
then run the cell.

In [ ]:
# Q2  One row per class: a fixed sample of photos, framed in the class colour used throughout the notebook.
N_PER_CLASS = 3 if SMOKE else 6  # fewer thumbnails in smoke mode, same code (N-014)
show_all_classes(N_PER_CLASS)

💡 Even six photos per class show that these are holiday snapshots, not camera-trap frames: close-ups,
people, sofas and streets, with the animal usually centred and filling the frame.

**Worked answer (Q2).** Cats and dogs are the farthest from a park camera: they are photographed indoors,
up close, often with people and furniture. Sheep, horses and zebras are at least outdoors and sometimes far
away, which is closer to what the paddock and savanna cameras will see.

Now pick one class and look at more of it. Twelve photos is enough to start noticing patterns; raise it if
you have time.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Pick a class</b><br>
Set <code>CLASS_TO_SHOW</code> to one of <code>cat, dog, bird, horse, sheep, zebra</code> and
<code>N_SHOW</code> to how many photos you want (up to the class size), then run the cell. Try at least
<code>"dog"</code> (the alert class) and <code>"bird"</code> (the tiny-box class).
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
CLASS_TO_SHOW = "dog"
N_SHOW = 12

if CLASS_TO_SHOW not in CLASSES:
    raise SystemExit(f"CLASS_TO_SHOW must be one of {CLASSES}, not {CLASS_TO_SHOW!r}")
if not isinstance(N_SHOW, int) or N_SHOW < 1:
    raise SystemExit(f"N_SHOW must be a positive integer, not {N_SHOW!r}")
n_show = min(N_SHOW, 6) if SMOKE else N_SHOW  # smoke mode caps the grid, same code path (N-014)
show_grid(CLASS_TO_SHOW, n_show)

💡 The title of each photo says how many animals of that class it holds: more than one is common, so "one
label per image" is already a simplification, and section 4 asks which label a photo with several animals
should get.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Karpathy's step ends with writing things down. Do the same: edit the next markdown cell (double-click it).
</font></td></tr></table>

**Q3.** 🟢 Write three observations about the photos you just looked at. Good observations are specific
("many dog photos are indoors with a person's legs in the frame"), and each one hints at something the park
should worry about or check later (fit to the cameras, lighting, distance, several animals, people, odd
photos).

**My three observations** (double-click to edit):

1. …
2. …
3. …

**Worked answer (Q3)**, for the dog grid on the full set. Yours will differ; that is the point.

1. **Setting:** most dog photos are indoors or in a street, taken by the owner from a metre away. A paddock
   camera sees a small dog far away in grass. The model will have to generalise from close-ups to that
   (section 11's question "does it fit?").
2. **People:** a person, a hand or a leash appears in many photos. A model can learn "person + leash = dog"
   and miss the stray dog alone in the paddock.
3. **Several animals and odd photos:** some photos hold two or three dogs, some show a dog in a costume,
   on a skateboard or on a screen. Each of these is a labelling decision (sections 4 and 5) before it is a
   modelling problem.

Keep your three lines: section 11 asks you for the same three on your own dataset, and the rest of this
notebook turns the observations above into numbers, one at a time.

## 3. Image inspection

**Objective:** measure sizes, aspect ratios, colour modes and colour statistics, and learn to read a cluster
of odd values as a clue about where the photos came from.

Your eyes found patterns in section 2. Now the computer looks at every photo once and writes down a handful
of numbers per image: its size, its shape, how its colours are stored and how bright it is. None of these
numbers needs a label, so this is the first check you can run on *any* folder of images, including your own
project dataset in section 11. Karpathy's recipe says why this pays off:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: search, filter, sort</b><br>
<i>"Once you get a qualitative sense it is also a good idea to write some simple code to search/filter/sort
by whatever you can think of (e.g. type of label, size of annotations, number of annotations, etc.) and
visualize their distributions and the outliers along any axis. The outliers especially almost always uncover
some bugs in data quality or preprocessing."</i><br> Andrej Karpathy, <a
href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural Networks</a> (2019).
</font></td></tr></table>

The next cell (collapsed) defines one function that measures a single photo. The cell after it runs that
function on every photo in the Play Dataset and collects the numbers in a table called `image_stats`.

In [ ]:
# @title Measurement helpers { display-mode: "form" }
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
from PIL import ImageDraw

HIST_BINS = 32
CHANNEL_COLORS = {"R": "#C0392B", "G": "#27AE60", "B": "#2980B9"}
COLOUR_MODES = ["RGB", "L", "RGBA", "CMYK"]  # the modes we always report, even when the count is 0


def measure_image(path, thumb: int) -> tuple[dict, np.ndarray]:
    """Open one photo and return (stats, channel_hist).

    stats: width, height, mode (as stored in the file), aspect (width / height), long_side, megapixels,
    brightness (mean grey level 0–255 on a thumbnail), contrast (its standard deviation), colourfulness
    (mean distance of the RGB channels from grey: 0 for a grayscale picture), mean_r / mean_g / mean_b.
    channel_hist: a (3, HIST_BINS) array, the density histogram of the R, G and B channels.
    """
    with Image.open(path) as im:
        mode, (w, h) = im.mode, im.size  # read these before decoding: they describe the file itself
        im.draft("RGB", (thumb, thumb))  # fast reduced-size JPEG decode (no effect on other formats)
        rgb = im.convert("RGB")
        rgb.thumbnail((thumb, thumb))
        arr = np.asarray(rgb, dtype=np.float32)
    grey = arr.mean(axis=2)
    channels = arr.reshape(-1, 3)
    hist = np.stack(
        [np.histogram(channels[:, k], bins=HIST_BINS, range=(0, 256), density=True)[0] for k in range(3)]
    )
    stats = {
        "width": w,
        "height": h,
        "mode": mode,
        "aspect": w / h,
        "long_side": max(w, h),
        "megapixels": w * h / 1e6,
        "brightness": float(grey.mean()),
        "contrast": float(grey.std()),
        "colourfulness": float(np.abs(channels - grey.reshape(-1, 1)).mean()),
        "mean_r": float(channels[:, 0].mean()),
        "mean_g": float(channels[:, 1].mean()),
        "mean_b": float(channels[:, 2].mean()),
    }
    return stats, hist


def show_rows(rows_df: pd.DataFrame, title: str, caption=None, cols: int = 6) -> None:
    """A small gallery of the photos in `rows_df` (rows of image_stats), each framed in its class colour.

    `caption(row) -> str` writes the title above each photo; by default the size and the colour mode.
    """
    if caption is None:

        def caption(row):
            return f"{row['class']} · {row['width']}×{row['height']} {row['mode']}"

    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.4 * cols, 2.7 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row), fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_extremes(stat: str, n: int = 4) -> None:
    """Gallery of the `n` photos with the lowest and the `n` with the highest value of `stat`.

    `stat` is a numeric column of `image_stats` (e.g. "brightness", "aspect", "megapixels", "colourfulness").
    """
    numeric = [c for c in image_stats.columns if pd.api.types.is_numeric_dtype(image_stats[c]) and c != "id"]
    if stat not in numeric:
        raise SystemExit(f"stat must be one of {numeric}, not {stat!r}")
    ordered = image_stats.sort_values(stat)
    n = max(1, min(n, len(ordered) // 2))

    def caption(row):
        return f"{row['class']} · {stat} = {row[stat]:.2f}\n{row['width']}×{row['height']} {row['mode']}"

    show_rows(ordered.head(n), f"lowest {stat}", caption, cols=n)
    show_rows(ordered.tail(n)[::-1], f"highest {stat}", caption, cols=n)

In [ ]:
# Measure every photo once (a few seconds: each JPEG is decoded at thumbnail size) and keep the numbers.
THUMB = 64 if SMOKE else 128  # smaller thumbnails in smoke mode, same code (N-014)

_records, _hists = [], []
for _, _row in images.iterrows():
    _stats, _hist = measure_image(_row["path"], THUMB)
    _records.append({"id": _row["id"], "class": _row["class"], "path": _row["path"], **_stats})
    _hists.append(_hist)
image_stats = pd.DataFrame(_records)
CHANNEL_HISTS = np.stack(_hists)  # (n_images, 3, HIST_BINS)
RESULTS["image_stats_n"] = int(len(image_stats))

_same_size = image_stats[["width", "height"]].to_numpy() == images[["width", "height"]].to_numpy()
_agree = int(_same_size.all(axis=1).sum())
print(f"measured {len(image_stats)} photos · the manifest's width/height matches the file for {_agree}")
STAT_COLUMNS = ["width", "height", "aspect", "megapixels", "brightness", "contrast", "colourfulness"]
image_stats[STAT_COLUMNS].describe().round(2)

💡 `describe()` is the quickest sanity check there is: look at `min` and `max` of every column before you
look at anything else. A `min` width of 0, a `max` of 6000 or a brightness of 254 each tell a story.

### Sizes and shapes

A model sees every photo at one fixed size, so the first question is how far the photos are from that.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run. These photos were uploaded to Flickr by hundreds of people and collected by the COCO
team; the park's photos will come from a few fixed cameras.
</font></td></tr></table>

**Q4.** 🟢 Do you expect all photos to have the same size? Sketch (in your head) where the dots of a
width-vs-height scatter would land: one point, a line, a cloud? Then run the cell.

In [ ]:
# Q4  Left: every photo as a dot (width across, height up) in its class colour; the dashed line is a square
#     photo. Right: the aspect ratio width/height, stacked by class.
fig, (ax_sc, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
for c in CLASSES:
    s = image_stats[image_stats["class"] == c]
    ax_sc.scatter(s["width"], s["height"], s=22, alpha=0.65, color=CLASS_COLORS[c], label=c, edgecolor="none")
_lim = float(max(image_stats["width"].max(), image_stats["height"].max())) * 1.08
ax_sc.plot([0, _lim], [0, _lim], "--", color=INK, lw=1)
ax_sc.set_xlim(0, _lim)
ax_sc.set_ylim(0, _lim)
ax_sc.set_xlabel("width (px)")
ax_sc.set_ylabel("height (px)")
ax_sc.set_title("Image sizes")
ax_sc.legend(frameon=False, fontsize=9, markerscale=1.6)

_bins = np.linspace(image_stats["aspect"].min() * 0.98, image_stats["aspect"].max() * 1.02, 28)
ax_hist.hist(
    [image_stats.loc[image_stats["class"] == c, "aspect"] for c in CLASSES],
    bins=_bins,
    stacked=True,
    color=[CLASS_COLORS[c] for c in CLASSES],
    label=CLASSES,
)
for ratio, name in [(3 / 4, "3:4"), (1.0, "1:1"), (4 / 3, "4:3")]:
    ax_hist.axvline(ratio, color=INK, lw=0.8, ls=":")
    ax_hist.text(ratio, ax_hist.get_ylim()[1] * 0.97, name, ha="center", va="top", fontsize=8, color=INK)
ax_hist.set_xlabel("aspect ratio = width / height  (left of 1: portrait, right: landscape)")
ax_hist.set_ylabel("photos")
ax_hist.set_title("Aspect ratios")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The dots do not fill the plane: they sit on a few lines and clusters, because the photos share a common
longest side. The aspect histogram shows the same thing from the side: a landscape peak, a portrait peak and
a long tail of panoramas and tall crops. Nothing here is "wrong", but it means a model fed 224×224 squares
will see most of these photos squashed or cropped (the 🔀 Callout at the end of this section).

**Worked answer (Q4).** Not one size, and not a random cloud either: COCO downloaded its photos from Flickr
at one standard size, so the longest side is capped at the same value for almost every photo and the dots
line up along "width = cap" and "height = cap". A dataset from the park's own cameras would look different:
one dot per camera model (every frame of a camera has the same size), which is both tidier and a leakage
risk you will meet in section 6 (the size gives away the camera).

### Colour modes and colours

How a file stores colour is easy to forget and painful to discover during training: a single grayscale (`L`)
or `RGBA` file in a batch of `RGB` arrays crashes the data loader, or silently gets a channel duplicated or
dropped. The next cell counts the modes and draws the typical RGB histogram of each class.

In [ ]:
# Colour modes per class (always listing RGB / L / RGBA / CMYK), then the mean R, G, B histogram per class.
_modes = COLOUR_MODES + sorted(set(image_stats["mode"]) - set(COLOUR_MODES))
mode_by_class = pd.crosstab(image_stats["mode"], image_stats["class"])
mode_by_class = mode_by_class.reindex(index=_modes, columns=CLASSES, fill_value=0)
image_stats["looks_gray"] = (image_stats["mode"] == "L") | (image_stats["colourfulness"] < 2.0)
print("colour modes: " + " · ".join(f"{m} {int(mode_by_class.loc[m].sum())}" for m in _modes))
_n_gray = int(image_stats["looks_gray"].sum())
print(f"photos that look grayscale (mode L, or RGB with equal channels): {_n_gray}")

fig = plt.figure(figsize=(13, 4.6))
_gs = fig.add_gridspec(
    2, 4, width_ratios=[1.4, 1, 1, 1], hspace=0.65, wspace=0.3, left=0.05, right=0.99, top=0.86, bottom=0.12
)
ax_modes = fig.add_subplot(_gs[:, 0])
_bottom = np.zeros(len(_modes))
for c in CLASSES:
    ax_modes.bar(_modes, mode_by_class[c].to_numpy(), bottom=_bottom, color=CLASS_COLORS[c], label=c)
    _bottom += mode_by_class[c].to_numpy()
ax_modes.set_ylabel("photos")
ax_modes.set_title("Colour mode as stored in the file", fontsize=10)
ax_modes.legend(frameon=False, fontsize=8)

_edges = np.linspace(0, 256, HIST_BINS + 1)
_centres = (_edges[:-1] + _edges[1:]) / 2
for k, c in enumerate(CLASSES):
    ax = fig.add_subplot(_gs[k // 3, 1 + k % 3])
    _mean_hist = CHANNEL_HISTS[(image_stats["class"] == c).to_numpy()].mean(axis=0)
    for ch_idx, ch in enumerate("RGB"):
        ax.plot(_centres, _mean_hist[ch_idx], color=CHANNEL_COLORS[ch], lw=1.4, label=ch)
    ax.set_title(f"{c} ({int((image_stats['class'] == c).sum())} photos)", color=CLASS_COLORS[c], fontsize=10)
    ax.set_yticks([])
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(CLASS_COLORS[c])
        spine.set_linewidth(1.6)
    if k == 0:
        ax.legend(frameon=False, fontsize=7, loc="upper left")
fig.text(0.67, 0.95, "Mean R, G, B histogram per class (pixel value 0–255)", ha="center", color=INK)
display(fig)
plt.close(fig)

💡 Read the histograms as *typical colours of a class*: a class whose curves differ from the others' (a
strong blue channel for a pond, high values across all three for a sandy savanna) is a class a model can
recognise from colour alone. That is a shortcut you should know about: a dog on green grass in the paddock
may be classified as "sheep" because the model learned "green = paddock = sheep".

### Brightness and the extremes

Brightness is the mean grey level of a photo (0 black, 255 white). Its distribution tells you whether the
dataset has night shots at all, and its extremes are the fastest route to broken or odd files.

In [ ]:
# Brightness per class as box plots, in class colours. The dotted lines mark "very dark" and "very bright".
fig, ax = plt.subplots(figsize=(9, 3.8))
_data = [image_stats.loc[image_stats["class"] == c, "brightness"] for c in CLASSES]
_box = ax.boxplot(_data, tick_labels=CLASSES, patch_artist=True, widths=0.6, medianprops={"color": INK})
for patch, c in zip(_box["boxes"], CLASSES, strict=True):
    patch.set_facecolor(CLASS_COLORS[c])
    patch.set_alpha(0.85)
for level in (40, 215):
    ax.axhline(level, color=INK, lw=0.8, ls=":")
ax.set_ylim(0, 255)
ax.set_ylabel("brightness (mean grey level)")
ax.set_title("Brightness per class")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every class covers most of the range, so the dataset has dim and bright photos, but the medians sit
comfortably in the middle: these are daytime holiday photos. The park's cameras also run at dusk and at
night, which this data cannot teach.

Numbers point, eyes decide. `show_extremes(stat, n)` shows the `n` photos at both ends of any column of
`image_stats`, so you can see *what* a very dark, very wide or very grey photo actually is.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Look at the extremes</b><br>
Set <code>STAT_TO_SHOW</code> to one of <code>brightness, contrast, colourfulness, aspect, megapixels,
long_side</code> and <code>N_EXTREMES</code> to how many photos per end you want, then run the cell. Try
<code>"brightness"</code> first, then <code>"colourfulness"</code> (the low end is where grayscale pictures
hide) and <code>"aspect"</code>.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
STAT_TO_SHOW = "brightness"
N_EXTREMES = 4

if not isinstance(N_EXTREMES, int) or N_EXTREMES < 1:
    raise SystemExit(f"N_EXTREMES must be a positive integer, not {N_EXTREMES!r}")
n_extremes = min(N_EXTREMES, 3) if SMOKE else N_EXTREMES  # smoke mode caps the gallery, same code (N-014)
show_extremes(STAT_TO_SHOW, n_extremes)

💡 The darkest photos are usually real (an animal in shade, a night shot with flash) and worth keeping: they
are the kind of frame the park's cameras produce at dusk. The brightest are often snow, sand or an
over-exposed sky. When one end shows a uniform grey rectangle, a thumbnail-sized file or a photo of a photo,
you have found a file for section 5's cleaning list.

### The detective beat: a cluster is a clue

A statistic that most photos share, and a few do not, is more than a curiosity. Photos from one pipeline
(one camera, one download script, one phone model) come out with the same size and mode. A **cluster of odd
sizes or modes therefore points to a different source**: another camera, another download, a batch someone
converted by hand. On the park's cameras you would see it as two size clusters (two camera models) or a
group of grayscale frames (the infrared night mode). Here we look for the same thing in COCO.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Look back at the scatter from Q4 before you run the next cell.
</font></td></tr></table>

**Q5.** 🟡 Which longest-side value do most photos share, and how many photos do you think do *not* share
it? Write down a guess, then run the cell: it counts them, lists their sizes and shows them.

In [ ]:
# Q5  The most common longest side, the photos that break the pattern, and the photos that look grayscale.
COMMON_LONG_SIDE = int(image_stats["long_side"].mode().iloc[0])
odd_size = image_stats[image_stats["long_side"] != COMMON_LONG_SIDE].sort_values("long_side")
gray_like = image_stats[image_stats["looks_gray"]]

print(
    f"{len(image_stats) - len(odd_size)} of {len(image_stats)} photos have a longest side of "
    f"{COMMON_LONG_SIDE} px; {len(odd_size)} do not."
)
if len(odd_size):
    _sizes = odd_size.apply(lambda r: f"{r['width']}×{r['height']}", axis=1).value_counts()
    print("odd sizes: " + " · ".join(f"{s} ×{n}" for s, n in _sizes.items()))
else:
    print("every photo in this sample went through the same resize step.")
print(f"{len(gray_like)} photos look grayscale.")

N_ODD = 6 if SMOKE else 12  # how many of the odd ones to show (N-014)
show_rows(odd_size.head(N_ODD), f"photos whose longest side is not {COMMON_LONG_SIDE} px")
show_rows(gray_like.head(N_ODD), "photos that look grayscale")

💡 The odd sizes are small, not large: nobody uploaded a 4000-pixel photo here. The photos that fall below
the common cap were *already* smaller than it when COCO fetched them, which is a hint about their origin
(old uploads, low-resolution cameras, screenshots). A grayscale photo in a colour dataset is a similar hint,
and both go on the list of things to decide about in section 5.

**Worked answer (Q5).** On the full Play Dataset almost every photo has a longest side of 640 px, because
COCO downloaded Flickr's "640 px" rendition, and only a handful fall below it (none may appear in a small
sample). The number itself matters less than the habit: whenever a column of `image_stats` has one dominant
value, ask "what made the others different?", because the answer is usually a different source, and a
different source usually means different labels, lighting or quality. On your own dataset (section 11) this
is the first question to ask about size and mode.

### Across tasks: what resizing does to boxes and masks

Everything above was about the photo. The park's second and third jobs (count, outline) attach **boxes** and
**masks** to the photo, and they are affected by the first thing every training pipeline does: resize.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · Resizing</b><br>
<ul>
<li><b>Classification:</b> squashing a 4:3 photo into a 224×224 square distorts aspect ratios. Usually
harmless for "which animal", because every photo is distorted the same way. Cropping or padding keeps the
shape at the cost of pixels or borders.</li>
<li><b>Detection:</b> box coordinates live in pixels, so a box must be rescaled with the image: x and width
by <code>new_width / width</code>, y and height by <code>new_height / height</code>. A forgotten rescale is
the classic "the boxes are all in the top-left corner" bug.</li>
<li><b>Segmentation:</b> a mask holds class ids, not colours. Resizing it with the usual smooth (bilinear)
filter invents in-between values that are no class at all; masks need <b>nearest-neighbour</b>
resizing.</li>
</ul>
</font></td></tr></table>

**Q6.** 🟡 A box is stored as `(x, y, w, h)` in pixels on a `W × H` photo, and the photo is resized to
224×224. Write the formula for the new box. Then predict: how many distinct values does the mask hold after
a nearest-neighbour resize, and after a bilinear one? Run the cell to check both.

In [ ]:
# Q6  One photo, one box and one mask, squeezed to 224×224: the box is rescaled; the mask is resized two ways.
TARGET = 224
_polygon = annotations["segmentation"].apply(lambda s: isinstance(s, list))
_ann = annotations[(annotations["iscrowd"] == 0) & _polygon].iloc[0]
_photo = images.loc[images["id"] == _ann["image_id"]].iloc[0]
_color = CLASS_COLORS[_ann["class"]]

with Image.open(_photo["path"]) as _im:
    original = _im.convert("RGB")
W, H = original.size
sx, sy = TARGET / W, TARGET / H
x, y, w, h = _ann["bbox"]
new_box = (x * sx, y * sy, w * sx, h * sy)

mask = Image.new("L", (W, H), 0)
for _poly in _ann["segmentation"]:
    ImageDraw.Draw(mask).polygon(list(zip(_poly[::2], _poly[1::2], strict=True)), fill=1)
squashed = original.resize((TARGET, TARGET), Image.BILINEAR)
mask_nearest = np.asarray(mask.resize((TARGET, TARGET), Image.NEAREST))
mask_bilinear = np.asarray(mask.point(lambda v: v * 255).resize((TARGET, TARGET), Image.BILINEAR)) / 255.0

print(f"photo {W}×{H} → {TARGET}×{TARGET}: scale x by {sx:.3f}, y by {sy:.3f}")
_old = ", ".join(f"{v:.0f}" for v in (x, y, w, h))
_new = ", ".join(f"{v:.1f}" for v in new_box)
print(f"box  (x, y, w, h) = ({_old})  →  ({_new})")
print(
    f"mask values: original {np.unique(np.asarray(mask)).tolist()} · "
    f"nearest {np.unique(mask_nearest).tolist()} · bilinear {len(np.unique(mask_bilinear))} distinct values"
)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.9))
axes[0].imshow(original)
axes[0].add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=_color, lw=2))
axes[0].set_title(f"original {W}×{H} + box", fontsize=9)
axes[1].imshow(squashed)
axes[1].add_patch(Rectangle(new_box[:2], new_box[2], new_box[3], fill=False, edgecolor=_color, lw=2))
axes[1].set_title(f"resized {TARGET}×{TARGET} + rescaled box", fontsize=9)
axes[2].imshow(mask_nearest, cmap=ListedColormap(["white", _color]), vmin=0, vmax=1)
axes[2].set_title(f"mask, nearest: {len(np.unique(mask_nearest))} values", fontsize=9)
_soft_cmap = LinearSegmentedColormap.from_list("mask", ["white", _color])
axes[3].imshow(mask_bilinear, cmap=_soft_cmap, vmin=0, vmax=1)
axes[3].set_title(f"mask, bilinear: {len(np.unique(mask_bilinear))} values", fontsize=9)
for ax in axes:
    frame(ax, _color, width=2)
fig.suptitle(f"{_ann['class']}: what one resize does to the photo, its box and its mask", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The box follows the photo as long as you scale x and y separately. The mask does not survive a smooth
resize: the bilinear version has a soft grey rim of values between 0 and 1, which are not "background" and
not "the animal". With six classes those in-between values would become ids of *other* animals.

**Worked answer (Q6).** New box = `(x · 224/W, y · 224/H, w · 224/W, h · 224/H)`; x-like values use the
horizontal factor, y-like values the vertical one, and on a landscape photo the two factors differ, so the
box is squashed exactly like the photo. Nearest-neighbour resizing keeps the two original values (0 and 1).
Bilinear resizing produces many distinct values along every edge of the animal, and the number grows with
the length of the boundary, so thin parts (legs, tails, a bird's beak) suffer most. That is why segmentation
pipelines resize images with a smooth filter and masks with nearest-neighbour.

**Takeaway.** `image_stats` costs seconds and already answers four of the checklist's questions (size,
shape, colour mode, colour) for any folder of images, labelled or not. Section 4 adds the labels and asks
the question every count depends on: how many photos of each class do we have, and does it matter?

## 4. Labels & class distribution

**Objective:** count the photos of each class, see why a photo with several animals is a labelling problem,
and understand why the dog alert cares about imbalance more than about accuracy.

Sections 2 and 3 looked at the photos. This section looks at the **labels**: the one word per photo that a
classifier will learn to say. Two questions decide whether the labels are usable at all: how many photos of
each class do we have, and is "one word per photo" even the right shape for this data? Karpathy lists the
things he looks for at this point:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: imbalances and biases</b><br>
<i>"Luckily, your brain is pretty good at this. One time I discovered that the data contained duplicate
examples. Another time I found corrupted images / labels. I look for data imbalances and biases."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

Duplicates and corrupted files are section 5's job. Here we take the last sentence: imbalances and biases
in the labels.

### Images per class

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run. Section 1's checklist already printed the per-class counts once; this time you will
see them as a bar chart, next to each class's share of the dataset.
</font></td></tr></table>

**Q7.** 🟢 Which class do you expect to have the most photos, which the fewest, and roughly how many times
larger is the biggest class than the smallest? Write down your guess, then run the cell.

In [ ]:
# Q7  Photos per class as bars (CLASS_COLORS), with each class's share of the dataset written on the bar.
images_per_class = {c: int((images["class"] == c).sum()) for c in CLASSES}
RESULTS["images_per_class"] = images_per_class

_total = sum(images_per_class.values())
_largest = max(images_per_class, key=images_per_class.get)
_smallest = min(images_per_class, key=images_per_class.get)
_ratio = images_per_class[_largest] / max(1, images_per_class[_smallest])
print(f"{_total} photos · largest class {_largest} ({images_per_class[_largest]}) · smallest {_smallest} "
      f"({images_per_class[_smallest]}) · ratio {_ratio:.1f}×")

fig, ax = plt.subplots(figsize=(9, 3.8))
_order = sorted(CLASSES, key=images_per_class.get, reverse=True)
ax.bar(_order, [images_per_class[c] for c in _order], color=[CLASS_COLORS[c] for c in _order])
for i, c in enumerate(_order):
    ax.text(i, images_per_class[c], f"{100 * images_per_class[c] / _total:.0f} %", ha="center", va="bottom",
            fontsize=9)
ax.set_ylabel("photos")
ax.set_title("Images per class (one label per photo)")
ax.set_ylim(0, max(images_per_class.values()) * 1.15)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The bars are not flat: the pet classes come first because people photograph their pets, and sheep come
last because few people photograph sheep. Nobody designed this distribution; it is what Flickr uploads look
like, and it has nothing to do with what the park's cameras will see (sheep all day, a dog once a month).

**Worked answer (Q7).** On the full Play Dataset cat and dog are the largest classes (capped at about 130
photos each) and sheep is the smallest (about 50), so the biggest class is roughly 2–3× the smallest. In the
small smoke sample the bars are flat, because that sample takes the same number of photos per class.
Whatever the sample, the habit is the same: read the *ratio* between the largest and the smallest class, and
ask whether the smallest class is the one your use case cares about. Here it is not sheep that matters, it
is dog, and dog is well represented in the photos while being the *rare* event in the paddock.

### Several animals in one photo: which label then?

`images` has one `class` per photo, and section 0 already showed that `n_animals` is often above 1. That is
fine as long as all the animals in a photo are the same class, and it is a real problem the moment they are
not. The Play Dataset was built by keeping only COCO photos with **exactly one animal class**, which is a
decision someone took *for* you. Check what the decision hides.

**Q8.** 🟡 What share of the photos do you think hold more than one animal, and which class do you expect to
be the most "crowded"? Then run the cell: it counts animals per photo and shows the most crowded photos.

In [ ]:
# Q8  Animals per photo, stacked by class, then a gallery of the photos with the most animals.
_max_n = int(images["n_animals"].max())
_several = images[images["n_animals"] > 1]
print(f"{len(_several)} of {len(images)} photos ({100 * len(_several) / len(images):.0f} %) hold more than "
      f"one animal; the most crowded photo has {_max_n}.")
print("mean animals per photo: " + " · ".join(
    f"{c} {images.loc[images['class'] == c, 'n_animals'].mean():.1f}" for c in CLASSES))

fig, ax = plt.subplots(figsize=(9, 3.8))
_bins = np.arange(0.5, min(_max_n, 10) + 1.5, 1)
_per_class = [images.loc[images["class"] == c, "n_animals"].clip(upper=10) for c in CLASSES]
ax.hist(_per_class, bins=_bins, stacked=True, color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.set_xticks(range(1, min(_max_n, 10) + 1))
ax.set_xticklabels([str(i) if i < 10 else "10+" for i in range(1, min(_max_n, 10) + 1)])
ax.set_xlabel("animals in the photo")
ax.set_ylabel("photos")
ax.set_title("Animals per photo")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

N_CROWDED = 3 if SMOKE else 6  # how many crowded photos to show (N-014)
_crowded = images.sort_values(["n_animals", "id"], ascending=[False, True]).head(N_CROWDED)
show_rows(_crowded, "the most crowded photos", lambda r: f"{r['n_animals']} × {r['class']}", cols=N_CROWDED)

💡 "One label per photo" works here only because every animal in a photo is the same species, and even then
the label says nothing about *how many*: a flock of thirty sheep and a single lamb are both "sheep". Look
at the crowded photos: some also hold people, fences or a second species that COCO does not label as an
animal class we kept, and the label is silent about all of it.

**Worked answer (Q8).** On the full Play Dataset about a third of the photos hold more than one animal, and
sheep and birds are the crowded classes (flocks), while cats and dogs are mostly alone. "Which label then?"
has three honest answers, and the park has to pick one before labelling a single frame:
- **one label, the main animal**: cheap, but the labeller decides what "main" means, and the model learns
  that fuzzy rule;
- **one label per animal class present** (multi-label): "sheep and dog" is exactly the paddock alert, but
  it needs a different loss and metric than plain classification;
- **one box per animal** (detection, section 8): answers "which" and "how many" at once, at a higher
  annotation cost (the 🔀 Callout below).

COCO's builders took the first answer *for the photos they kept*, and the Play Dataset went further by
dropping every photo with two classes. For the paddock camera, where the one case that matters is a dog
**among** sheep, that filter would delete exactly the frames the alert is for.

### Why the dog alert cares about imbalance

Imbalance is harmless in a bar chart and dangerous in a metric. The park's paddock camera sees sheep almost
all the time and a dog very rarely, and the usual score, accuracy, is blind to that.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Accuracy hides a rare class</b><br>
Accuracy is the share of all photos labelled correctly, so it is dominated by the big classes. A model that
<b>never</b> predicts a rare class loses only that class's share of the points. If dogs are 2 % of the
paddock frames, "always say sheep" scores 98 % accuracy and misses every dog, which is the one event the
alert exists for. The number to watch for a rare class is its <b>recall</b>: of the real dogs, how many did
we catch? Section 7 turns this into the park's metric.
</font></td></tr></table>

No model is needed to feel this. The next cell builds the laziest classifier there is, "always predict the
biggest class", and scores it twice: on the Play Dataset and on a paddock day whose dog share you choose.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · A paddock day</b><br>
<code>DOG_SHARE</code> is the fraction of the paddock camera's frames that show a dog (the rest show sheep).
Try <code>0.02</code>, then <code>0.2</code>, then <code>0.5</code>, and watch what happens to the accuracy
of "always sheep" and to its recall on dog.
</font></td></tr></table>


In [ ]:
# ✏️ YOUR TURN
DOG_SHARE = 0.02

if not isinstance(DOG_SHARE, (int, float)) or not 0 <= DOG_SHARE <= 1:
    raise SystemExit(f"DOG_SHARE must be a number between 0 and 1, not {DOG_SHARE!r}")

**Q9.** 🟡 Before you run: what accuracy does "always predict the biggest class" get on the Play Dataset,
and what is its recall on dog? And on a paddock day with the `DOG_SHARE` you set? Write both down, then run.

In [ ]:
# Q9  "Always the biggest class" on the Play Dataset: every photo gets the same label.
_majority = max(images_per_class, key=images_per_class.get)
_pred = np.full(len(images), _majority)
_true = images["class"].to_numpy()
acc_play = float((_pred == _true).mean())
recall_dog_play = float((_pred[_true == "dog"] == "dog").mean()) if (_true == "dog").any() else float("nan")
print(f"Play Dataset · 'always {_majority}': accuracy {100 * acc_play:.0f} %, recall on dog "
      f"{100 * recall_dog_play:.0f} %")

# A paddock day: PADDOCK_FRAMES frames, a share DOG_SHARE of them show a dog, the rest sheep.
PADDOCK_FRAMES = 200 if SMOKE else 1000
_n_dog = int(round(DOG_SHARE * PADDOCK_FRAMES))
_day = np.array(["dog"] * _n_dog + ["sheep"] * (PADDOCK_FRAMES - _n_dog))
_always_sheep = np.full(PADDOCK_FRAMES, "sheep")
acc_day = float((_always_sheep == _day).mean())
recall_dog_day = float((_always_sheep[_day == "dog"] == "dog").mean()) if _n_dog else float("nan")
print(f"Paddock day   · {_n_dog} dog frames in {PADDOCK_FRAMES}: 'always sheep' accuracy "
      f"{100 * acc_day:.0f} %, recall on dog {100 * recall_dog_day:.0f} %")

fig, ax = plt.subplots(figsize=(9, 3.4))
_shares = np.linspace(0, 0.5, 51)
_acc_curve = (1 - _shares) * 100
ax.plot(_shares * 100, _acc_curve, color=CLASS_COLORS["sheep"], lw=2, label="accuracy of 'always sheep'")
ax.plot(_shares * 100, np.zeros_like(_shares), color=CLASS_COLORS["dog"], lw=2, label="its recall on dog")
ax.axvline(DOG_SHARE * 100, color=INK, lw=0.8, ls=":")
ax.text(DOG_SHARE * 100, 50, f" DOG_SHARE = {DOG_SHARE:g}", color=INK, fontsize=9)
ax.set_xlabel("share of paddock frames that show a dog (%)")
ax.set_ylabel("score (%)")
ax.set_ylim(-5, 105)
ax.set_title("The laziest classifier on a paddock day")
ax.legend(frameon=False, fontsize=9, loc="center right")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Accuracy of the lazy classifier *is* the majority share: the rarer the dog, the better "always sheep"
looks, and its recall on dog stays at zero the whole way. A metric that rewards ignoring the event you built
the system for is the wrong metric.

**Worked answer (Q9).** On the Play Dataset "always the biggest class" scores the biggest class's share
(about 20 % on the full set, a sixth in the balanced smoke sample) and a recall on dog of 0 % unless dog
*is* the biggest class, in which case it scores 100 % on dog and 0 % on everything else. On a paddock day
with `DOG_SHARE = 0.02` it reaches 98 % accuracy with 0 % recall on dog. Two lessons: a balanced-looking
Play Dataset says nothing about the imbalance of the *deployment* (the paddock), and whatever the imbalance,
the score has to be computed on the class that matters. Training-time fixes for imbalance (class weights,
oversampling, a focal loss) are next week's topic; choosing the metric is section 7's.

### Across tasks: what a label costs

All of the above assumed "one word per photo". The park's other two jobs need richer labels, and richer
labels cost more to make. The next cell counts what the Play Dataset's annotations contain, in the crudest
unit there is: how many clicks a labeller had to make.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · One label, one box, every pixel</b><br>
<ul>
<li><b>Classification:</b> one label per image. One click (or one keystroke) per photo; seconds each. The
cheapest label, and the one that cannot say "how many" or "where".</li>
<li><b>Detection:</b> one box per animal. Two clicks per box, and a decision for every animal (is that blur
a bird?); tens of seconds per photo, more when the photo is crowded. Imbalance is now counted in
<i>boxes</i>, and the ranking of the classes changes (section 8).</li>
<li><b>Segmentation:</b> every pixel gets a class. A polygon of many vertices per animal; a minute or more
per animal, and the hardest to get right at thin parts. Imbalance is counted in <i>pixels</i>, and birds end
up with a few per cent of them (section 9).</li>
</ul>
The same photo therefore costs roughly 1 : 10 : 100 to label for the three tasks, which is why real datasets
have many more image labels than boxes, and many more boxes than masks.
</font></td></tr></table>

In [ ]:
# Clicks per task on the Play Dataset: 1 per image label, 2 per box, one per polygon vertex for masks.
_polygons = annotations["segmentation"].apply(lambda s: s if isinstance(s, list) else [])
_vertices = _polygons.apply(lambda polys: sum(len(p) // 2 for p in polys))
_n_crowd = int((annotations["iscrowd"] == 1).sum())
annotation_cost = pd.DataFrame(
    {
        "unit": ["image labels", "boxes", "mask vertices"],
        "count": [len(images), len(annotations), int(_vertices.sum())],
        "clicks": [len(images), 2 * len(annotations), int(_vertices.sum())],
    }
).set_index("unit")
annotation_cost["clicks per photo"] = (annotation_cost["clicks"] / len(images)).round(1)
_label_clicks = annotation_cost.loc["image labels", "clicks"]
annotation_cost["× classification"] = (annotation_cost["clicks"] / _label_clicks).round(1)
if _n_crowd:
    print(f"({_n_crowd} crowd regions are stored as pixel masks, not polygons, and count 0 vertices here.)")
display(annotation_cost)

💡 The same photos, the same animals, and the click count grows by an order of magnitude per task. When the
park asks for "a quick model that also counts and outlines", this table is the answer to "how quick".

**Takeaway.** Count the labels before you trust them: the ratio between the biggest and the smallest class,
the share of photos with several animals, and the share of the class your use case depends on. Then
remember that the deployment has its own imbalance, and that accuracy will happily ignore the rare class the
park built the camera for. Section 5 goes after the labels and photos that are *wrong*, not just rare.

## 5. Common dataset errors & cleaning

**Objective:** find duplicates, flipped and rotated copies, broken files and suspicious labels in a dataset,
and produce a cleaned table you can split and train on.

Sections 2 to 4 asked whether the data *fits* the task. This section asks whether the data is *wrong*: the
same photo stored twice, a copy that someone flipped in a viewer, a file that no longer opens, a label that
a tired annotator got wrong. None of these show up in a bar chart, and all of them bite later: a duplicate
that lands in both train and test makes the score lie, and a wrong label teaches the model a wrong rule.

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: the two bugs Karpathy names</b><br>
<i>"One time I discovered that the data contained duplicate examples. Another time I found corrupted
images / labels."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

Two stories from Kaggle show how ordinary this is. In the <b>PetFinder.my Pawpularity</b> competition
(2021), teams found near-identical photos in the training set that carried *different* target scores, and
asked on the forum whether the same photos also sat in the hidden test set; the hosts answered that
duplicates between train and test had been removed, and left the pairs inside the training set as they
were, scores and all. In <b>Cassava Leaf Disease Classification</b> (2020), a competition known for its
noisy labels, teams reported a pair of identical leaf photos that carried two different disease labels.
Neither dataset was careless: both were built by people who cared, and both still shipped with duplicates.
Yours will too, unless you look.

### Exact duplicates: the same bytes twice

The cheapest check first. Two files that are byte-for-byte identical have the same **MD5 hash**, so one
pass over the files finds every exact copy, whatever its name.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
COCO val2017 is a curated benchmark that thousands of papers evaluate on. Does that mean it has no exact
duplicates? Decide before you run.
</font></td></tr></table>

**Q10.** 🟢 How many byte-identical pairs do you expect in the Play Dataset: none, a handful, or many? Write
your guess, then run the next two cells.

In [ ]:
# @title Hash helpers (MD5, perceptual hash, flipped and rotated variants) { display-mode: "form" }
import hashlib
import shutil
from io import BytesIO

import imagehash
from PIL import Image, ImageOps

THUMB_FOR_HASH = 256  # perceptual hashes are computed on a thumbnail: fast, and the same for every file


def file_md5(path) -> str:
    """MD5 of the file's bytes: identical files share it, a one-pixel change breaks it."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def phash(image: Image.Image) -> imagehash.ImageHash:
    """Perceptual hash of a PIL image (64 bits). `h1 - h2` is the number of bits that differ (0–64)."""
    return imagehash.phash(image)


def variants(image: Image.Image) -> dict[str, Image.Image]:
    """The image as stored plus its mirrored, flipped and rotated versions, keyed by what was done."""
    return {
        "original": image,
        "flip_lr": ImageOps.mirror(image),
        "flip_tb": ImageOps.flip(image),
        "rot90": image.transpose(Image.Transpose.ROTATE_90),
        "rot180": image.transpose(Image.Transpose.ROTATE_180),
        "rot270": image.transpose(Image.Transpose.ROTATE_270),
    }


def hash_bits(hashes: list[imagehash.ImageHash]) -> np.ndarray:
    """Stack ImageHash objects into an (n, 64) boolean matrix, so distances can be computed in one go."""
    return np.array([h.hash.flatten() for h in hashes], dtype=bool)


def hash_distances(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """(len(a), len(b)) matrix of Hamming distances between two (n, 64) hash matrices."""
    return (a[:, None, :] != b[None, :, :]).sum(axis=-1)


def exact_duplicate_groups(paths: list) -> list[list]:
    """Groups of paths whose bytes are identical (only groups with 2+ members), in first-seen order."""
    by_md5: dict[str, list] = {}
    for p in paths:
        by_md5.setdefault(file_md5(p), []).append(p)
    return [group for group in by_md5.values() if len(group) > 1]


def is_readable(path) -> bool:
    """True if PIL can open and fully decode the file."""
    try:
        with Image.open(path) as im:
            im.draft("RGB", (THUMB_FOR_HASH, THUMB_FOR_HASH))  # a reduced decode still reads every byte
            im.load()
        return True
    except Exception:
        return False

In [ ]:
# Q10  MD5 of every file in the Play Dataset; identical bytes share one hash.
exact_groups = exact_duplicate_groups(images["path"].tolist())
print(f"{len(images)} files · {len(exact_groups)} groups of byte-identical files")
for _group in exact_groups[:5]:
    print("  ", " = ".join(Path(p).name for p in _group))

💡 Whatever the count, this check cost one line and one pass over the files; run it on every folder you are
handed, because exact copies are the duplicates that slip through most often (a batch uploaded twice, a
"final" and a "final_v2" export of the same card).

**Worked answer (Q10).** COCO val2017 was deduplicated by its authors, so on the Play Dataset the expected
answer is none or close to it. That is the exception, not the rule: the park's cameras, like any camera
upload, will produce exact copies the first time someone re-exports a memory card. And MD5 is blind to
everything but identical bytes: re-save the same photo at a different JPEG quality and the hash changes
completely. For that we need a hash that looks at the picture, not the file.

### Near-duplicates: the same picture, different bytes

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · Perceptual hashing</b><br>
A <b>perceptual hash</b> (here pHash, from the <code>imagehash</code> package) shrinks the photo to a 32×32
grey thumbnail, takes its low-frequency content (a DCT, as in JPEG), and writes 64 bits saying whether each
coefficient is above or below the median. Two photos that <i>look</i> the same get almost the same bits
even when their files differ (resized, re-compressed, slightly cropped). The <b>Hamming distance</b>
between two hashes is the number of bits that differ: 0 means identical hashes, about 32 means unrelated.
A threshold on that distance decides what counts as "the same picture", and that threshold is a choice.
</font></td></tr></table>

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · The duplicate threshold</b><br>
<code>HASH_THRESHOLD</code> is the largest Hamming distance (0–64) at which two photos count as
near-duplicates. Start with the default, look at the closest pairs the next cell shows, then try a strict
<code>2</code> and a loose <code>14</code> and watch what enters the list.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
HASH_THRESHOLD = 8

if not isinstance(HASH_THRESHOLD, int) or not 0 <= HASH_THRESHOLD <= 64:
    raise SystemExit(f"HASH_THRESHOLD must be a whole number between 0 and 64, not {HASH_THRESHOLD!r}")

**Q11.** 🟡 With your threshold, how many pairs of the Play Dataset do you expect to be flagged, and will
the closest pairs be true copies or just two photos of the same kind of scene? Guess, then run.

In [ ]:
# Q11  pHash of every photo (and of its flipped/rotated variants, used below), then the closest pairs.
VARIANT_NAMES = list(variants(Image.new("RGB", (4, 4))))
_per_variant: dict[str, list] = {v: [] for v in VARIANT_NAMES}
for _p in images["path"]:
    _thumb = load_thumb(_p, THUMB_FOR_HASH)
    for _v, _im in variants(_thumb).items():
        _per_variant[_v].append(phash(_im))
VARIANT_HASHES = {v: hash_bits(hs) for v, hs in _per_variant.items()}  # each (n_images, 64)
HASHES = VARIANT_HASHES["original"]

_dist = hash_distances(HASHES, HASHES)
np.fill_diagonal(_dist, 99)
nearest_distance = _dist.min(axis=1)  # each photo's distance to its closest other photo
_i, _j = np.where(np.triu(_dist <= HASH_THRESHOLD, k=1))
near_pairs = pd.DataFrame({"a": images["id"].to_numpy()[_i], "b": images["id"].to_numpy()[_j],
                           "distance": _dist[_i, _j]}).sort_values("distance").reset_index(drop=True)
print(f"pairs within {HASH_THRESHOLD} bits: {len(near_pairs)} · median nearest-neighbour distance: "
      f"{np.median(nearest_distance):.0f} bits")

fig, ax = plt.subplots(figsize=(9, 3.4))
_bins = np.arange(-0.5, 41, 2)
ax.hist([nearest_distance[(images["class"] == c).to_numpy()] for c in CLASSES], bins=_bins, stacked=True,
        color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.axvline(HASH_THRESHOLD + 0.5, color=INK, ls=":", lw=1)
ax.text(HASH_THRESHOLD + 1, ax.get_ylim()[1] * 0.9, f"HASH_THRESHOLD = {HASH_THRESHOLD}", fontsize=9)
ax.set_xlabel("Hamming distance to the closest other photo (bits)")
ax.set_ylabel("photos")
ax.set_title("How close is each photo's nearest neighbour?")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

# The closest pairs, side by side, whether or not they pass the threshold: this is what "near" looks like.
N_PAIRS_TO_SHOW = 2 if SMOKE else 4
_upper = np.where(np.triu(np.ones_like(_dist, dtype=bool), k=1), _dist, 99)  # each pair once
_flat = np.argsort(_upper, axis=None)[:N_PAIRS_TO_SHOW]
_closest = list(zip(*np.unravel_index(_flat, _dist.shape), strict=True))
if len(near_pairs):
    display(near_pairs.head(10))
_rows = pd.concat([image_stats.iloc[[a, b]] for a, b in _closest])
_dist_of = {row_id: _dist[a, b] for a, b in _closest for row_id in images["id"].iloc[[a, b]]}
show_rows(_rows, "closest pairs by perceptual hash (each pair side by side)",
          lambda r: f"{r['class']} · {_dist_of[r['id']]} bits apart", cols=2)

💡 The histogram sits far to the right of the threshold for almost every photo: unrelated pictures are
about 30 bits apart, and the threshold only matters for the thin left tail. Look at the pairs: at a few bits
you see the same scene; at 10 or more you see two different photos that merely share a layout.

**Worked answer (Q11).** On COCO most of the closest pairs are different photos of a similar scene (two
zebras on grass, a cat on a sofa) at distances above the default threshold, so the flagged list is short.
Short, not empty: on the full Play Dataset the default threshold catches one true pair, two frames of the
same tabby cat in front of the same brick wall, looking away in one and at the camera in the other. A
curated benchmark, and still a burst that slipped through. (The small smoke sample may not contain it.)
With a loose threshold the list fills up with photos that are *not* copies. That is the trade-off: a strict
threshold misses re-compressed copies and bursts, a loose one deletes real data. Keep the default for now
and read the list by eye before deleting anything.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · A hash distance is not a similarity score</b><br>
Two burst frames of the same sheep can be 12 bits apart and two unrelated photos 10. The threshold only
separates "same picture, different file" from everything else; it does not rank how alike two photos are.
And a plain hash is <b>orientation-sensitive</b>: mirror a photo and its hash changes by dozens of bits,
so a flipped copy looks like a new photo. The next part fixes that.
</font></td></tr></table>

### A memory card from the park: the guided hunt

The park's intern hands you a memory card. Some files on it are the same photo twice: the camera software
saved a copy, someone rotated a photo in a viewer and saved it, a second version was exported at lower
quality for an e-mail. The next cell writes that card into `data/cache/hunt5/` (seeded, so everyone gets
the same one) with neutral file names. **The answer key is printed after the hunt, not before.**

In [ ]:
# @title Plant the memory card (seeded, at runtime; nothing is redistributed) { display-mode: "form" }
HUNT_DIR = DATA_DIR / "cache" / "hunt5"
N_CARD_ORIGINALS = 12 if SMOKE else 40  # distinct photos on the card (N-014: fewer in smoke mode)
PLANT_KINDS = ["exact", "exact", "flipped", "rotated", "recompressed"]  # ≥ 4 planted copies, always present


def plant_hunt5(dest: Path, n_originals: int, seed: int) -> tuple[dict[str, Path], list[dict]]:
    """Write a 'memory card': n_originals distinct photos plus one planted copy per entry of PLANT_KINDS.

    Returns (files, key): files maps the card's neutral file names to paths; key lists the planted copies as
    {"copy": name, "source": name, "kind": kind}. Deterministic for a given seed.
    """
    rng = random.Random(seed)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    chosen = images.sample(n=min(n_originals, len(images)), random_state=seed).sort_values("id")
    sources = [Path(p) for p in chosen["path"]]
    # (bytes, kind or None for an original, index of the source photo)
    entries: list[tuple[bytes, str | None, int]] = [(s.read_bytes(), None, i) for i, s in enumerate(sources)]
    planted: list[tuple[str, int]] = []
    source_idx = rng.sample(range(len(sources)), len(PLANT_KINDS))
    for kind, i in zip(PLANT_KINDS, source_idx, strict=True):
        with Image.open(sources[i]) as im:
            rgb = im.convert("RGB")
        buf = BytesIO()
        if kind == "exact":
            data = sources[i].read_bytes()
        elif kind == "flipped":
            ImageOps.mirror(rgb).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        elif kind == "rotated":
            rgb.transpose(Image.Transpose.ROTATE_90).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        else:  # recompressed: smaller and at a lower JPEG quality
            small = rgb.resize((max(8, int(rgb.width * 0.7)), max(8, int(rgb.height * 0.7))))
            small.save(buf, "JPEG", quality=55)
            data = buf.getvalue()
        entries.append((data, kind, i))
        planted.append((kind, i))
    order = list(range(len(entries)))
    rng.shuffle(order)
    names = {idx: f"card_{k:03d}.jpg" for k, idx in enumerate(order)}
    files, key = {}, []
    for idx, (data, kind, i) in enumerate(entries):
        path = dest / names[idx]
        path.write_bytes(data)
        files[names[idx]] = path
        if kind is not None:
            key.append({"copy": names[idx], "source": names[i], "kind": kind})
    return dict(sorted(files.items())), key

In [ ]:
hunt_files, HUNT5_KEY = plant_hunt5(HUNT_DIR, N_CARD_ORIGINALS, SEED)
hunt_images = {name: load_thumb(path, THUMB_FOR_HASH) for name, path in hunt_files.items()}
print(f"{len(hunt_files)} files on the card in {HUNT_DIR}/ · {len(HUNT5_KEY)} of them are copies of another "
      f"file on the card")

**Q12.** 🟡 The shoot-out. Two detectors run on the card: plain hashing (MD5 plus pHash with your threshold)
and *variant* hashing, which also hashes each photo's mirrored, flipped and rotated versions. Predict which
kinds of copy each one catches, then run.

In [ ]:
# @title Copy finder: plain hashing vs variant hashing { display-mode: "form" }
KIND_OF_VARIANT = {"original": "recompressed", "flip_lr": "flipped", "flip_tb": "flipped",
                   "rot90": "rotated", "rot180": "rotated", "rot270": "rotated"}


def find_copies(files: dict[str, Path], threshold: int, use_variants: bool) -> pd.DataFrame:
    """Pairs of files on a card that are copies of each other: columns a, b, kind, distance.

    kind is "exact" (same bytes), "recompressed" (hashes within `threshold`), or, with use_variants=True,
    "flipped" / "rotated" (a flipped or rotated variant of a hashes within `threshold` of b).
    """
    names = list(files)
    md5s = {n: file_md5(p) for n, p in files.items()}
    thumbs = {n: load_thumb(p, THUMB_FOR_HASH) for n, p in files.items()}
    plain = hash_bits([phash(thumbs[n]) for n in names])
    variant_names = VARIANT_NAMES if use_variants else ["original"]
    # distances[v][i, j]: hash of variant v of file i vs the plain hash of file j
    distances = {v: hash_distances(hash_bits([phash(variants(thumbs[n])[v]) for n in names]), plain)
                 for v in variant_names}
    rows = []
    for i, a in enumerate(names):
        for j in range(i + 1, len(names)):
            b = names[j]
            if md5s[a] == md5s[b]:
                rows.append({"a": a, "b": b, "kind": "exact", "distance": 0})
                continue
            best_v = min(variant_names, key=lambda v: distances[v][i, j])
            best_d = int(distances[best_v][i, j])
            if best_d <= threshold:
                rows.append({"a": a, "b": b, "kind": KIND_OF_VARIANT[best_v], "distance": best_d})
    return pd.DataFrame(rows, columns=["a", "b", "kind", "distance"])


def score_hunt(found: pd.DataFrame, key: list[dict]) -> int:
    """How many planted copies in `key` appear as a pair (either order) in `found`."""
    pairs = {frozenset((r["a"], r["b"])) for _, r in found.iterrows()}
    return sum(frozenset((k["copy"], k["source"])) in pairs for k in key)

In [ ]:
# Q12  Plain hashing first, then variant hashing, on the same card and the same threshold.
plain_found = find_copies(hunt_files, HASH_THRESHOLD, use_variants=False)
variant_found = find_copies(hunt_files, HASH_THRESHOLD, use_variants=True)
print(f"plain hashing   : {len(plain_found)} pairs → {dict(Counter(plain_found['kind']))}")
print(f"variant hashing : {len(variant_found)} pairs → {dict(Counter(variant_found['kind']))}")
display(variant_found)

💡 The two detectors agree on the exact and re-compressed copies and disagree on the rest: mirroring or
rotating a photo moves its hash by dozens of bits, so plain hashing files it as a new photo, while hashing
the variants brings it back within a few bits of its source.

**Worked answer (Q12).** Plain hashing catches the byte-identical copies (MD5) and the re-compressed one
(pHash is built to survive re-compression and resizing). It misses the flipped and the rotated copy, which
variant hashing catches because one of the six variants of the source now hashes like the copy. The price
is six hashes per photo instead of one, which is still seconds for thousands of photos.

### ✍️ Write the flip detector yourself

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>find_flipped_copies</code></b><br>
Fill in the function below: given one photo and a dictionary of candidate photos, return the names of the
candidates that are a flipped or rotated copy of it. You have <code>variants(image)</code>,
<code>phash(image)</code>, <code>h1 - h2</code> for the distance and <code>HASH_THRESHOLD</code>. The
<code>use()</code> line checks your answer against the reference on the card and continues with the
reference either way.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def find_flipped_copies(image, candidates):
    """Return the names of the candidates that are a flipped or rotated copy of `image`.

    image: a PIL image. candidates: dict {name: PIL image} (the image itself is not among them).
    Returns a sorted list of names whose pHash is within HASH_THRESHOLD bits of the pHash of one of
    image's flipped or rotated variants (every variant except "original").
    """
    ...


def _ref_find_flipped_copies(image, candidates):
    flipped_hashes = [phash(im) for name, im in variants(image).items() if name != "original"]
    found = []
    for name, cand in candidates.items():
        h = phash(cand)
        if min(h - fh for fh in flipped_hashes) <= HASH_THRESHOLD:
            found.append(name)
    return sorted(found)


_flip_source = next(k["source"] for k in HUNT5_KEY if k["kind"] == "flipped")
_others = {name: im for name, im in hunt_images.items() if name != _flip_source}
flipped_copies = use(find_flipped_copies, hunt_images[_flip_source], _others,
                     reference=_ref_find_flipped_copies)
print(f"flipped/rotated copies of {_flip_source}: {flipped_copies}")

### The hunt's score and the answer key

In [ ]:
# Score the reference detector against the planted key, then show every planted pair side by side.
hunt5_found = score_hunt(variant_found, HUNT5_KEY)
RESULTS["hunt5"] = {"planted": len(HUNT5_KEY), "found": int(hunt5_found),
                    "kinds": dict(Counter(k["kind"] for k in HUNT5_KEY))}
print(f"planted copies found by variant hashing: {hunt5_found} of {len(HUNT5_KEY)}")
print("answer key:")
for _k in HUNT5_KEY:
    print(f"  {_k['copy']}: {_k['kind']} copy of {_k['source']}")

fig, axes = plt.subplots(2, len(HUNT5_KEY), figsize=(2.4 * len(HUNT5_KEY), 5.2), squeeze=False)
for _col, _k in enumerate(HUNT5_KEY):
    for _row, _name in enumerate((_k["source"], _k["copy"])):
        ax = axes[_row, _col]
        ax.imshow(hunt_images[_name])
        frame(ax, INK if _row == 0 else CLASS_COLORS["dog"])
        ax.set_title(f"{_name}\n{'source' if _row == 0 else _k['kind'] + ' copy'}", fontsize=8)
fig.suptitle("the planted copies (top: source, bottom: copy)", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every planted copy is a pair the detector reported, and the pairs it reported are only the planted ones:
that is the result to insist on with your own data too, a detector that finds everything you planted and
nothing else. Before you trust it on a real folder, plant a few copies and check it the same way.

### Broken and odd files

A file can be in the table and still be useless: truncated during download, a grayscale or CMYK scan among
RGB photos, a PNG with an alpha channel, a thumbnail of 40 pixels or a 50-megapixel panorama. The reader
will either crash on it or silently feed the model something strange. Section 3 already measured the
modes and sizes; this cell adds "does it even open?" and lists everything odd in one table.

In [ ]:
# Flags per file: unreadable, non-RGB mode, extreme size. Reading every file fully takes a few seconds.
EXTREME_SIDE_PX = (64, 6000)  # longest side below / above this is "extreme"


def flag_odd_files(stats: pd.DataFrame) -> pd.DataFrame:
    """One row per flagged file of `stats` (rows of image_stats): id, class, flags. Empty if all is well."""
    rows = []
    for _, r in stats.iterrows():
        flags = []
        if not is_readable(r["path"]):
            flags.append("unreadable")
        if r["mode"] != "RGB":
            flags.append(f"mode {r['mode']}")
        if r["long_side"] < EXTREME_SIDE_PX[0] or r["long_side"] > EXTREME_SIDE_PX[1]:
            flags.append(f"extreme size {r['width']}×{r['height']}")
        if flags:
            rows.append({"id": r["id"], "class": r["class"], "flags": ", ".join(flags)})
    return pd.DataFrame(rows, columns=["id", "class", "flags"])


odd_files = flag_odd_files(image_stats)
print(f"{len(odd_files)} of {len(image_stats)} files carry a flag")
display(odd_files.head(10))

# A broken file for comparison: the first half of a JPEG. `is_readable` catches it; a plain `open` would not.
_half = Path(images["path"].iloc[0]).read_bytes()
_broken = HUNT_DIR / "broken_demo.jpg"
_broken.write_bytes(_half[: len(_half) // 2])
print(f"a truncated copy of {Path(images['path'].iloc[0]).name}: readable = {is_readable(_broken)}")

💡 Flags are not verdicts. A grayscale photo may be a perfectly good night-camera frame, and a large
panorama may be the park's best shot of the whole paddock; the table says "look at me", and a human decides
whether to convert, resize or drop. Only "unreadable" is dropped without discussion.

### Suspicious labels: each class's least typical photos

A wrong label cannot be found by hashing. What can be done without a model is to sort each class by how
*unlike the rest of its class* a photo is, using the colour histograms of section 3, and hand the top of
each list to a human. Most of them will be fine (an unusual but correct photo); the wrong ones, if any, are
far more likely to be here than in a random sample.

In [ ]:
# Each photo's distance to its own class's mean colour histogram; the farthest few per class go to a human.
N_ATYPICAL = 2 if SMOKE else 4
_features = CHANNEL_HISTS.reshape(len(image_stats), -1)
_class_mean = {c: _features[(image_stats["class"] == c).to_numpy()].mean(axis=0) for c in CLASSES}
image_stats["atypicality"] = [float(np.abs(f - _class_mean[c]).sum())
                              for f, c in zip(_features, image_stats["class"], strict=True)]

fig, ax = plt.subplots(figsize=(9, 3.4))
for k, c in enumerate(CLASSES):
    _vals = image_stats.loc[image_stats["class"] == c, "atypicality"].to_numpy()
    _jitter = np.random.RandomState(SEED).uniform(-0.25, 0.25, len(_vals))
    ax.scatter(k + _jitter, _vals, s=14, color=CLASS_COLORS[c], alpha=0.8)
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES)
ax.set_ylabel("distance to the class's mean histogram")
ax.set_title("How typical is each photo of its class? (high = look at me)")
fig.tight_layout()
display(fig)
plt.close(fig)

least_typical = (image_stats.sort_values("atypicality", ascending=False).groupby("class", sort=False)
                 .head(N_ATYPICAL).sort_values(["class", "atypicality"], ascending=[True, False]))
show_rows(least_typical, "least typical photos per class: does the label fit?",
          lambda r: f"{r['class']} · {r['atypicality']:.1f}", cols=N_ATYPICAL * 2)

**Q13.** 🟡 Look at the gallery. For each photo, is the label right? Is the animal even visible? Note the
ones you would send back to the annotator; the next section's worked answer says what you typically find.

💡 Colour cannot tell a cat from a dog, so this list finds photos that are *unusual*, not photos that are
*wrong*: a black cat on white snow, a horse photographed at night. That is still useful, because an
unusual photo is where a tired annotator slips, and because a model will struggle on it either way.

**Worked answer (Q13).** On the Play Dataset most of the least typical photos are correctly labelled but
odd: close-ups, night shots, an animal that fills a tenth of the frame, a person holding the animal. Real
label errors in COCO val are rare at the image level (they are more common in the masks, section 9), so an
honest answer is "nothing to send back, two to keep an eye on". On a crowd-sourced dataset, the same list is
where the wrong labels live. The 🔴 part below replaces colour with a learned representation, which does
tell a cat from a dog.

### 🔴 Optional: model-based search on precomputed embeddings

A hash sees pixels; an **embedding** (the feature vector a pretrained network computes for a photo) sees
content, so two photos of the same dog from different angles land close together, and a photo whose
embedding sits far from every other photo of its class is the best candidate for a wrong label. No model
runs here: the vectors are read from a precomputed file if it is available. When it is not, the cell says so
and runs the same search on the colour histograms of section 3 as a stand-in, so you can see the mechanics.

In [ ]:
# @title Embedding helpers (precomputed vectors, or colour histograms as a stand-in) { display-mode: "form" }
EMBEDDINGS_NAME = "coco_animals_embeddings.npz"  # npz keys: ids (n,), embeddings (n, d); matched by image id


def load_embeddings() -> tuple[np.ndarray, str]:
    """(vectors aligned with `images`, source). Source is "precomputed" or "colour histograms (stand-in)".

    Reads data/coco_animals_embeddings.npz, fetching it from DATA_BASE_URL outside smoke mode. Never runs a
    model. Falls back to the colour histograms when the file is missing or does not cover every image.
    """
    path = DATA_DIR / EMBEDDINGS_NAME
    if not path.exists() and not SMOKE:
        download_file(DATA_BASE_URL + EMBEDDINGS_NAME, path, headers=_github_headers())
    if path.exists():
        try:
            z = np.load(path)
            by_id = dict(zip(z["ids"].tolist(), z["embeddings"], strict=True))
            if all(i in by_id for i in images["id"]):
                return np.stack([by_id[i] for i in images["id"]]).astype(np.float32), "precomputed"
        except Exception:
            pass
    return CHANNEL_HISTS.reshape(len(images), -1).astype(np.float32), "colour histograms (stand-in)"


def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    """Cosine similarity between every pair of rows (after centring the columns)."""
    centred = vectors - vectors.mean(axis=0)
    norm = np.linalg.norm(centred, axis=1, keepdims=True) + 1e-9
    unit = centred / norm
    return unit @ unit.T

**Q14.** 🔴 Will the embedding search's closest pairs be the same pairs the hash found, or different ones?
And will its least typical photos per class overlap with the colour-based list? Guess, then run.

In [ ]:
# Q14  Nearest pairs and per-class outliers in embedding space (or in the stand-in space).
embeddings, embedding_source = load_embeddings()
print(f"vectors: {embeddings.shape[0]} × {embeddings.shape[1]} from {embedding_source}")
RESULTS["embedding_source"] = embedding_source

_sim = cosine_similarity_matrix(embeddings)
np.fill_diagonal(_sim, -1)
_i, _j = np.unravel_index(np.argsort(-np.triu(_sim, k=1), axis=None)[:N_PAIRS_TO_SHOW], _sim.shape)
_rows = pd.concat([image_stats.iloc[[a, b]] for a, b in zip(_i, _j, strict=True)])
_sim_of = {row_id: _sim[a, b] for a, b in zip(_i, _j, strict=True) for row_id in images["id"].iloc[[a, b]]}
show_rows(_rows, f"closest pairs by {embedding_source}",
          lambda r: f"{r['class']} · sim {_sim_of[r['id']]:.2f}", cols=2)

_class_arr = image_stats["class"].to_numpy()
image_stats["class_similarity"] = [float(_sim[k, (_class_arr == c) & (np.arange(len(_sim)) != k)].mean())
                                   for k, c in enumerate(_class_arr)]
embedding_outliers = (image_stats.sort_values("class_similarity").groupby("class", sort=False)
                      .head(N_ATYPICAL).sort_values(["class", "class_similarity"]))
show_rows(embedding_outliers, f"least typical per class by {embedding_source}",
          lambda r: f"{r['class']} · sim {r['class_similarity']:.2f}", cols=N_ATYPICAL * 2)
_overlap = len(set(embedding_outliers["id"]) & set(least_typical["id"]))
print(f"{_overlap} of {len(embedding_outliers)} embedding outliers are also in the colour-based list")

💡 With real embeddings the closest pairs are *semantic* near-duplicates (the same dog, another angle),
which a hash cannot find, and the outliers are photos whose content does not match the label. With the
stand-in they are colour matches, which is why the cell says which one you are looking at.

**Worked answer (Q14).** The hash finds copies of the same *picture*; the embedding finds photos of the same
*thing*. The two lists overlap on exact and near copies and differ everywhere else, and the overlap between
the two outlier lists is partial at best: colour and content are different notions of "typical". To get
real embeddings, compute them once on a GPU (any pretrained image model), save `ids` and `embeddings` to
`data/coco_animals_embeddings.npz`, and this cell picks them up.

### Clean the dataset

Time to act. The rule applied below is the one most teams use: keep the first photo of every duplicate
group (exact, near, flipped or rotated), drop every unreadable file, and keep everything else with its
flag so a human can still decide. The result is `clean_images`, the table the rest of the notebook uses.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Split after cleaning, not before</b><br>
If you split first and deduplicate each half on its own, a pair with one photo in train and one in test
survives, and the test score reports how well the model <i>memorised</i> that photo. Deduplicate the whole
table, then split (section 6).
</font></td></tr></table>

In [ ]:
# Duplicate groups on the Play Dataset with variant hashing, then the cleaned table and its summary line.
_ids = images["id"].to_numpy()
_drop: dict[int, str] = {}
for _group in exact_groups:  # exact: keep the first file of each group
    _group_ids = [int(images.loc[images["path"] == str(p), "id"].iloc[0]) for p in _group]
    for _gid in sorted(_group_ids)[1:]:
        _drop[_gid] = "exact duplicate"
_best = np.full(_dist.shape, 99)
for _v in VARIANT_NAMES:  # variant hashing across the whole Play Dataset: min distance over the six variants
    _best = np.minimum(_best, hash_distances(VARIANT_HASHES[_v], HASHES))
np.fill_diagonal(_best, 99)
_pi, _pj = np.where(np.triu(np.minimum(_best, _best.T) <= HASH_THRESHOLD, k=1))
for _a, _b in zip(_pi, _pj, strict=True):  # near/flipped/rotated: keep the lower id
    _drop.setdefault(int(max(_ids[_a], _ids[_b])), "near-duplicate (incl. flipped/rotated)")
for _, _r in odd_files.iterrows():
    if "unreadable" in _r["flags"]:
        _drop[int(_r["id"])] = "unreadable"

clean_images = images[~images["id"].isin(_drop)].copy()
clean_images["flags"] = clean_images["id"].map(odd_files.set_index("id")["flags"]).fillna("")
removed = pd.DataFrame({"id": list(_drop), "reason": list(_drop.values())})
RESULTS["n_clean"] = int(len(clean_images))
RESULTS["n_removed"] = int(len(removed))
_n_flagged = int((clean_images["flags"] != "").sum())
print(f"Play Dataset: removed {len(removed)}, kept {len(clean_images)} "
      f"(reasons: {dict(Counter(removed['reason']))}; {_n_flagged} kept with a flag)")

# The same rule on the memory card: one file per duplicate group.
_card_drop = set()
for _, _r in variant_found.sort_values(["a", "b"]).iterrows():
    if _r["a"] not in _card_drop:
        _card_drop.add(_r["b"])
print(f"memory card : removed {len(_card_drop)}, kept {len(hunt_files) - len(_card_drop)}")

💡 On a curated benchmark the "removed" number is small and the lesson is in the tooling; on the park's own
uploads the same six cells will be the difference between a test score you can trust and one you cannot.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · The same errors, three shapes</b><br>
<ul>
<li><b>Classification:</b> a duplicate pair across train and test leaks one label; a wrong label is one
word to fix.</li>
<li><b>Detection:</b> a duplicated photo duplicates <i>all</i> its boxes, and a flipped copy whose boxes
were not mirrored (<code>x' = W − x − w</code>) is worse than no copy at all: the boxes now point at
grass. Label errors become <i>missing</i> or <i>shifted</i> boxes (section 8).</li>
<li><b>Segmentation:</b> the same for masks, plus a new error class: masks that are the wrong shape
(swollen, shifted, crude) while the label is right (section 9).</li>
</ul>
Deduplicate once, before splitting, for all three; the label checks differ per task.
</font></td></tr></table>

**Takeaway.** Four cheap checks before any training: MD5 for exact copies, a perceptual hash with a
threshold you have looked at for near copies, the same hash on flipped and rotated variants, and a
"does it open, what mode, what size" pass for broken files. Then sort each class by how unusual its photos
are and let a human read the top of the list. The output is a table, `clean_images`, and one line you can
put in a report: removed N, kept M. Section 6 splits that table without leaking.

## 6. Splits & leakage

**Objective:** choose a split that does not leak, and understand what k-fold, stratified, leave-one-out
and leave-one-group-out cross-validation each buy you.

Section 5 ended with `clean_images`, a table without duplicates. The next decision is the one that decides
whether next week's test score means anything: **which photos does the model never see?** The held-out
part has to look like the photos the park's cameras will take *after* training, not like the ones they
already took. If a test photo has a near-twin in training, the score reports memory, not recognition.

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · Data leakage</b><br>
<b>Leakage</b> is any way information about the test photos reaches the model during training. The
obvious route is a duplicate in both halves (section 5). The quiet route is <b>a photo that is not a copy
but comes from the same moment</b>: two frames of one burst, two shots of the same animal in front of the
same bush, two photos from one camera that always sees the same tree. The model learns the bush, the
score goes up, and the first camera in a new place breaks it. The rule: <b>split by whatever will be new
at deployment</b> (a camera, a day, a patient, a farm), not by photo.
</font></td></tr></table>

### Random vs stratified split on the Play Dataset

The simplest split shuffles the photos and cuts them 80 / 20. A **stratified** split cuts *each class*
80 / 20, so both halves keep the class shares of the whole table. The code below does both on
`clean_images` and compares the class shares of the test halves.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Sheep is the smallest class, with a fifth of the photos of cat. Think of a random 20 % test half.
</font></td></tr></table>

**Q15.** 🟡 In a random split, which class share will move the most between train and test: the largest
class or the smallest? Does the stratified split fix that completely, or only mostly? Decide, then run.

In [ ]:
# Random split vs stratified split of clean_images (80 / 20), then the class share of each test half.
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    LeaveOneGroupOut,
    LeaveOneOut,
    StratifiedKFold,
    train_test_split,
)

TEST_SIZE = 0.2
_ids = clean_images["id"].to_numpy()
_y = clean_images["class"].to_numpy()
_tr_rand, _te_rand = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED)
_tr_strat, _te_strat = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED, stratify=_y)


def class_shares(ids) -> pd.Series:
    """Share of each class among the given image ids of clean_images (in CLASSES order, sums to 1)."""
    counts = clean_images.set_index("id").loc[list(ids), "class"].value_counts()
    return counts.reindex(CLASSES).fillna(0) / max(len(ids), 1)


shares = pd.DataFrame(
    {
        "whole table": class_shares(_ids),
        "random · test": class_shares(_te_rand),
        "stratified · test": class_shares(_te_strat),
    }
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col in zip(axes, ["random · test", "stratified · test"], strict=True):
    x = np.arange(len(CLASSES))
    ax.bar(x - 0.2, shares["whole table"], width=0.4, color="#DDE5E7", label="whole table")
    ax.bar(x + 0.2, shares[col], width=0.4, color=[CLASS_COLORS[c] for c in CLASSES], label=col)
    ax.set_xticks(x, CLASSES)
    ax.set_title(f"{col} ({len(_te_rand)} photos)")
    ax.set_ylabel("share of photos")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()
_drift = (shares.sub(shares["whole table"], axis=0).abs() * 100).round(1)
print("largest class-share drift from the whole table, in percentage points:")
print(f"  random split: {_drift['random · test'].max():.1f} ({_drift['random · test'].idxmax()})   "
      f"stratified split: {_drift['stratified · test'].max():.1f} ({_drift['stratified · test'].idxmax()})")
RESULTS["split_drift_random_pp"] = float(_drift["random · test"].max())
RESULTS["split_drift_stratified_pp"] = float(_drift["stratified · test"].max())

💡 The grey bars are the whole table and the coloured bars the test half: with a random cut the small
classes wobble the most in relative terms, and the stratified cut sits on top of the grey bars up to the
rounding that whole photos force.

**Worked answer (Q15).** In absolute percentage points the largest classes can move the most, but what
hurts is the *relative* move: a class of 50 photos that loses or gains five in the test half changes its
share by 10 %, and with a rare class like the paddock's dog the test half may end up with almost none of
the one class the alert is about. Stratification removes that wobble up to rounding (you cannot put 10.6
sheep in the test half), so the shares match to within one photo per class. It does nothing about the
content of the photos, which is the problem the rest of this section is about.

### Cross-validation: four ways to reuse one dataset

One split gives one score, and with a few hundred photos that score moves by several points when you
change the seed. **Cross-validation** rotates the test half around the table so every photo is tested
once, and reports the mean (and spread) over the folds. Four schemes, each with its reason:

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · k-fold, stratified k-fold, leave-one-out, leave-one-group-out</b><br>
<ul>
<li><b>k-fold:</b> cut the table into k pieces, test on each piece once, train on the other k−1.
Five or ten folds is the usual choice.</li>
<li><b>stratified k-fold:</b> the same, but every piece keeps the class shares. Use it whenever classes are
imbalanced, which is always.</li>
<li><b>leave-one-out (LOO):</b> k equals the number of photos. Every photo is tested once against a model
trained on all the others. Useful when the dataset is <b>very small</b> (a few dozen photos) and every
training example counts; otherwise it costs n trainings, and the n scores are each 0 or 1, so the mean is
noisy.</li>
<li><b>leave-one-group-out (LOGO):</b> the folds are <b>groups</b> that must stay together (one camera, one
day, one patient). Each fold holds out one whole group, which is the only scheme here that tests what
happens at a <i>new</i> camera.</li>
</ul>
</font></td></tr></table>

The cell below runs the first three on the Play Dataset and reports, for each, how many folds there are,
how big a test fold is, and the smallest class share any test fold has (0 means some fold tests a class
it never sees). LOO runs on a small slice of the table, because that is where it belongs.

**Q16.** 🟡 Before running: how many folds does leave-one-out make on the Play Dataset, and what is the
smallest class share a LOO test fold can have?

In [ ]:
# k-fold, stratified k-fold and leave-one-out on the Play Dataset: folds, test size, smallest class share.
K_FOLDS = 5
LOO_N = 12 if SMOKE else 40  # leave-one-out on a slice: a "very small dataset" (N-014: same code, fewer rows)


def summarise_folds(name: str, splitter, X, y, groups=None) -> dict:
    """One row per CV scheme: folds, test-fold size and the smallest class share across the test folds."""
    sizes, min_share = [], 1.0
    for _, test_idx in splitter.split(X, y, groups):
        sizes.append(len(test_idx))
        counts = pd.Series(y[test_idx]).value_counts(normalize=True).reindex(sorted(set(y))).fillna(0)
        min_share = min(min_share, float(counts.min()))
    return {"scheme": name, "folds": len(sizes), "test fold size": f"{min(sizes)}–{max(sizes)}",
            "smallest class share in a test fold": round(min_share, 3)}


_loo_ids = _ids[:LOO_N]
cv_rows = [
    summarise_folds("kfold", KFold(K_FOLDS, shuffle=True, random_state=SEED), _ids, _y),
    summarise_folds("stratified_kfold", StratifiedKFold(K_FOLDS, shuffle=True, random_state=SEED), _ids, _y),
    summarise_folds("loo", LeaveOneOut(), _loo_ids, _y[:LOO_N]),
]
cv_schemes = [r["scheme"] for r in cv_rows]
display(pd.DataFrame(cv_rows).set_index("scheme"))
print(f"(loo ran on the first {LOO_N} photos: {LOO_N} folds, {LOO_N} trainings for one score)")

💡 Plain k-fold already drops the smallest class's share in some fold below its overall share; stratified
k-fold holds it steady; LOO's test folds are one photo each, so the share column reads 0 whatever you do.

**Worked answer (Q16).** LOO makes one fold per photo, so as many folds as rows, and it would cost that many
trainings next week. Each test fold is a single photo, so its smallest class share is always 0: five of
six classes are absent from every fold. That is harmless for the mean score but shows why LOO tells you
nothing per class. The fourth scheme, leave-one-group-out, needs groups, and the Play Dataset has none:
COCO photos come from thousands of unrelated Flickr accounts. The park's next cameras do have groups.

### The woodland trail cameras

The park has just hung **trail cameras in its woodland reserve**. A trail camera fires on motion and takes
a **burst** of three frames a second apart, every night, from the same tree, for years. We use a slice of
the Caltech Camera Traps dataset (Beery et al., 2018) as a stand-in: eight camera locations, five
animals (opossum, raccoon, coyote, bobcat, rabbit), whole bursts kept together. `data/cct20_slice.csv`
lists the photos; the next cell downloads them from LILA's public bucket and shrinks them to 512 px, so
**the first run takes a minute** and afterwards the cell is instant. The five woodland classes borrow the
park's six colours so every plot keeps one colour per class.

In [ ]:
# @title Trail-camera helpers (manifest, download + resize to 512 px, burst gallery) { display-mode: "form" }
CCT_DIR = DATA_DIR / "cache" / "cct20"
CCT_SIDE = 512
CCT_CLASSES = ["opossum", "raccoon", "coyote", "bobcat", "rabbit"]
CCT_COLORS = dict(zip(CCT_CLASSES, [CLASS_COLORS[c] for c in CLASSES], strict=False))
SMOKE_CCT_LOCATIONS = 3
SMOKE_CCT_IMAGES = 30


def select_cct_rows(table: pd.DataFrame) -> pd.DataFrame:
    """The slice to use: everything, or in smoke mode whole bursts from 3 locations up to 30 photos."""
    if not SMOKE:
        return table
    keep = []
    locations = sorted(table["location"].unique())[:SMOKE_CCT_LOCATIONS]
    per_location = SMOKE_CCT_IMAGES // SMOKE_CCT_LOCATIONS
    for loc in locations:
        n = 0
        at_loc = table[table["location"] == loc].sort_values(["seq_id", "frame_num"])
        for _, burst in at_loc.groupby("seq_id", sort=True):
            if n + len(burst) > per_location:
                continue
            keep.append(burst)
            n += len(burst)
    return pd.concat(keep).reset_index(drop=True)


def fetch_cct_image(row: dict) -> tuple[str, Path | None]:
    """Download one trail-camera photo and store a 512 px copy; returns (file_name, path or None)."""
    dest = CCT_DIR / row["file_name"]
    if dest.exists() and dest.stat().st_size > 0:
        return row["file_name"], dest
    raw = download_file(row["url"], CCT_DIR / "raw" / row["file_name"])
    if raw is None:
        return row["file_name"], None
    with Image.open(raw) as im:
        im.draft("RGB", (CCT_SIDE, CCT_SIDE))
        small = im.convert("RGB")
        small.thumbnail((CCT_SIDE, CCT_SIDE))
        small.save(dest, "JPEG", quality=90)
    raw.unlink(missing_ok=True)
    return row["file_name"], dest


def download_cct(table: pd.DataFrame, workers: int = 16) -> pd.DataFrame:
    """Fetch every photo of `table` (parallel, cached), return the rows whose photo is on disk with a path."""
    CCT_DIR.mkdir(parents=True, exist_ok=True)
    paths, done = {}, 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for name, path in pool.map(fetch_cct_image, table.to_dict("records")):
            done += 1
            if path is not None:
                paths[name] = str(path)
            print(f"\r  trail-camera photos ready: {done}/{len(table)}", end="", flush=True)
    print()
    out = table[table["file_name"].isin(paths)].copy()
    out["path"] = out["file_name"].map(paths)
    return out.sort_values(["location", "seq_id", "frame_num"]).reset_index(drop=True)


def show_bursts(table: pd.DataFrame, n_bursts: int, size: int = 256) -> None:
    """One row per burst: its frames side by side, framed in the class colour, titled with the location."""
    bursts = [b for _, b in table.groupby("seq_id", sort=True) if len(b) > 1][:n_bursts]
    cols = max(len(b) for b in bursts)
    fig, axes = plt.subplots(len(bursts), cols, figsize=(2.6 * cols, 2.1 * len(bursts)), squeeze=False)
    for r, burst in enumerate(bursts):
        for c in range(cols):
            ax = axes[r, c]
            if c < len(burst):
                row = burst.iloc[c]
                ax.imshow(load_thumb(row["path"], size))
                frame(ax, CCT_COLORS[row["label"]])
                ax.set_title(f"camera {row['location']} · {row['label']} · frame {row['frame_num']}",
                             fontsize=8)
            else:
                ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Load the slice, download and shrink the photos (cached), then look at a few bursts.
cct_all = pd.read_csv(fetch_manifest("cct20_slice.csv"))
cct = download_cct(select_cct_rows(cct_all))
if len(cct) < len(select_cct_rows(cct_all)):
    print(f"  ⚠️ {len(select_cct_rows(cct_all)) - len(cct)} photos could not be downloaded; run again.")
CCT_LOCATIONS = sorted(cct["location"].unique().tolist())
print(f"{len(cct)} photos · {cct['seq_id'].nunique()} bursts · cameras {CCT_LOCATIONS}")
print("photos per class: " + " · ".join(f"{c} {int((cct['label'] == c).sum())}" for c in CCT_CLASSES))
show_bursts(cct, n_bursts=3)

💡 The frames of a burst differ by one second and a few steps of the animal; everything else, the tree,
the ground and the exposure, is identical. If one frame is in test and its siblings in train, the model is
asked to recognise a photo it has almost seen.

### Leakage, measured without a model

No training is needed to see the leak. For any split, count **the share of test photos that have a burst
sibling (same `seq_id`) in training**. A random split scatters the frames of each burst over both halves;
a split **by camera** keeps every burst, and every tree, on one side.

**Q17.** 🟡 Predict the sibling share of a random 80 / 20 split of the trail-camera photos: about 0 %,
about 20 %, or most of the test photos? And of a split by camera?

In [ ]:
# Share of test photos with a burst sibling in training: random split vs split by camera.
def sibling_rate(train: pd.DataFrame, test: pd.DataFrame) -> float:
    """Share of test photos whose seq_id (burst) also appears in the training table (0 = no leak)."""
    if len(test) == 0:
        return 0.0
    return float(test["seq_id"].isin(set(train["seq_id"])).mean())


_tr, _te = train_test_split(cct.index.to_numpy(), test_size=TEST_SIZE, random_state=SEED)
sibling_random = sibling_rate(cct.loc[_tr], cct.loc[_te])

_gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
_tr_g, _te_g = next(_gss.split(cct, groups=cct["location"]))
sibling_camera = sibling_rate(cct.iloc[_tr_g], cct.iloc[_te_g])
_held_out = sorted(cct.iloc[_te_g]["location"].unique().tolist())

RESULTS["sibling_rate_random"] = sibling_random
RESULTS["sibling_rate_by_camera"] = sibling_camera
print(f"random split    : {sibling_random:5.1%} of the {len(_te)} test photos have a burst sibling in train")
print(f"split by camera : {sibling_camera:5.1%} of the {len(_te_g)} test photos (held out: {_held_out})")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(["random split", "split by camera"], [sibling_random, sibling_camera],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["zebra"]])
ax.set_ylabel("test photos with a burst sibling in train")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

💡 With bursts of three frames, a random split gives most test photos a sibling in training; splitting by
camera brings the share to zero by construction, because a burst never spans two cameras.

**Worked answer (Q17).** A random split leaks for most test photos: a frame lands in test, and with high
probability at least one of its two siblings landed in train. The split by camera leaks for none. The same
effect shows in the paper the data comes from: Beery et al. (2018) report a classification **error of
20.8 % at cameras seen in training and 41.1 % at cameras never seen**, doubling at the first new tree. A
random split would have reported the 20.8 % and the park would have discovered the 41.1 % in the field.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Leave-one-out leaks like a random split</b><br>
LOO sounds strict (train on everything but one photo), but on burst data it is the <i>worst</i> case: the
held-out frame's siblings are always in training. The fix is <b>leave-one-<i>group</i>-out</b> with the
camera as the group: every fold holds out one whole camera, so the sibling share is zero in every fold
and the score is the one a new camera will see. The same holds for any "one item at a time" scheme: the
unit you leave out must be the unit that will be new.
</font></td></tr></table>

The cell below checks that claim fold by fold: leave-one-out and leave-one-group-out on the trail-camera
photos, each reporting the mean sibling share over its folds.

In [ ]:
# Leave-one-out vs leave-one-group-out (camera) on the trail cameras: sibling share per fold.
def mean_sibling_rate(splitter, table: pd.DataFrame, groups=None) -> float:
    """Mean over the folds of the share of test photos with a burst sibling in the fold's training part."""
    rates = [sibling_rate(table.iloc[tr], table.iloc[te]) for tr, te in splitter.split(table, groups=groups)]
    return float(np.mean(rates))


loo_sibling = mean_sibling_rate(LeaveOneOut(), cct)
logo_sibling = mean_sibling_rate(LeaveOneGroupOut(), cct, groups=cct["location"])
cv_schemes += ["logo"]
RESULTS["cv_schemes"] = cv_schemes
RESULTS["sibling_rate_loo"] = loo_sibling
RESULTS["sibling_rate_logo"] = logo_sibling
print(f"leave-one-out        : {len(cct)} folds, sibling share {loo_sibling:.1%}")
print(f"leave-one-group-out  : {cct['location'].nunique()} folds (one per camera), sibling share "
      f"{logo_sibling:.1%}")
print(f"cross-validation schemes run in this notebook: {cv_schemes}")

💡 LOO's sibling share is the share of photos that belong to a burst of more than one frame, which is
nearly all of them; LOGO by camera is zero in every fold.

### ✍️ Choose and run a split

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>split_by_camera</code></b><br>
Fill in the function: given the trail-camera table and a list of camera locations to hold out, return a
dictionary <code>{"train": [...], "test": [...]}</code> of <code>file_name</code> values, every photo of a
held-out camera in test and every other photo in train, each list sorted. Then choose
<code>TEST_LOCATIONS</code> (any cameras from <code>CCT_LOCATIONS</code>). The <code>use()</code> line
checks your function against the reference and continues with the reference either way.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def split_by_camera(table, test_locations):
    """Split the trail-camera table by camera.

    table: a DataFrame with columns file_name and location (cct). test_locations: list of location ids.
    Returns {"train": sorted file_names of every other camera, "test": sorted file_names of test_locations}.
    """
    ...


def _ref_split_by_camera(table, test_locations):
    in_test = table["location"].isin(test_locations)
    return {"train": sorted(table.loc[~in_test, "file_name"]),
            "test": sorted(table.loc[in_test, "file_name"])}


TEST_LOCATIONS = [CCT_LOCATIONS[0], CCT_LOCATIONS[-1]]  # choose: any cameras from CCT_LOCATIONS
_bad = [loc for loc in TEST_LOCATIONS if loc not in CCT_LOCATIONS]
if _bad or not TEST_LOCATIONS or len(set(TEST_LOCATIONS)) >= len(CCT_LOCATIONS):
    raise SystemExit(f"TEST_LOCATIONS must be a non-empty subset of {CCT_LOCATIONS} that leaves cameras for "
                     f"training; unknown: {_bad}")

my_split = use(split_by_camera, cct, TEST_LOCATIONS, reference=_ref_split_by_camera)
_train_t = cct[cct["file_name"].isin(my_split["train"])]
_test_t = cct[cct["file_name"].isin(my_split["test"])]
print(f"train: {len(_train_t)} photos from cameras {sorted(_train_t['location'].unique().tolist())}")
print(f"test : {len(_test_t)} photos from cameras {sorted(_test_t['location'].unique().tolist())}")
print(f"sibling share of this split: {sibling_rate(_train_t, _test_t):.1%}")
_split_shares = pd.DataFrame({
    "train": _train_t["label"].value_counts(normalize=True).reindex(CCT_CLASSES).fillna(0),
    "test": _test_t["label"].value_counts(normalize=True).reindex(CCT_CLASSES).fillna(0),
}).round(2)
display(_split_shares.T)
_absent = [c for c in CCT_CLASSES if _split_shares.loc[c, "test"] == 0]
if _absent:
    print(f"⚠️ no {', '.join(_absent)} photos in the test cameras: those classes get no score. "
          "Try other cameras.")

💡 A split by camera has no siblings across the cut, but it can leave a class without test photos
(not every animal walks past every camera): you trade a clean score for a less balanced one, and you choose
the held-out cameras with both in mind.

### 🔴 Optional: a ruler to see the gap

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · The nearest-neighbour ruler (not training)</b><br>
No network is trained in this notebook. To <i>see</i> the gap the two splits create, we use a
<b>ruler</b>: a 1-nearest-neighbour classifier on each photo's colour histogram (the 96 numbers section 3
computed). It has no parameters to learn, runs in seconds on a CPU, and is nowhere near a real model. Its
only job is to measure the same thing twice: once with k-fold (random, leaks across bursts) and once with
leave-one-camera-out.
</font></td></tr></table>

**Q18.** 🔴 Will the ruler score higher with k-fold or with leave-one-camera-out, and roughly by how much?
Predict, then run.

In [ ]:
# The colour-histogram ruler: 1-NN accuracy under k-fold vs leave-one-camera-out on the trail cameras.
from sklearn.neighbors import KNeighborsClassifier

RULER_THUMB = 64
_features = np.stack([measure_image(p, RULER_THUMB)[1].ravel() for p in cct["path"]])
_labels = cct["label"].to_numpy()


def ruler_accuracy(splitter, groups=None) -> float:
    """Mean 1-NN accuracy over the folds of `splitter` on the colour-histogram features."""
    scores = []
    for tr, te in splitter.split(_features, _labels, groups):
        knn = KNeighborsClassifier(n_neighbors=1).fit(_features[tr], _labels[tr])
        scores.append(float((knn.predict(_features[te]) == _labels[te]).mean()))
    return float(np.mean(scores))


_n_cameras = cct["location"].nunique()
ruler_kfold = ruler_accuracy(KFold(min(K_FOLDS, _n_cameras), shuffle=True, random_state=SEED))
ruler_by_camera = ruler_accuracy(GroupKFold(_n_cameras), groups=cct["location"])
RESULTS["ruler_kfold"] = ruler_kfold
RESULTS["ruler_by_camera"] = ruler_by_camera
print(f"ruler accuracy, k-fold (random)      : {ruler_kfold:.1%}")
print(f"ruler accuracy, leave-one-camera-out : {ruler_by_camera:.1%}")
print(f"chance level with {len(CCT_CLASSES)} classes: {1 / len(CCT_CLASSES):.0%}")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(["k-fold (random)", "leave-one-camera-out"], [ruler_kfold, ruler_by_camera],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["zebra"]])
ax.axhline(1 / len(CCT_CLASSES), color=INK, linestyle="--", linewidth=1, label="chance")
ax.set_ylabel("1-NN ruler accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

💡 The ruler scores by finding the most similar photo in training: under k-fold that photo is usually a
burst sibling with the same animal in front of the same tree, so it looks good; under leave-one-camera-out
it has to find a similar photo from another camera and falls towards chance.

**Worked answer (Q18).** The random scheme scores higher, and the gap is large: on the full slice the ruler
roughly doubles its accuracy when its neighbours are allowed to be siblings. The exact numbers depend on
the slice and the seed, so read the direction and the size of the gap, not the digits. A real model would
score higher on both, and the gap would still be there: that is the 20.8 % → 41.1 % of the paper.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · Splitting for boxes and masks</b><br>
<ul>
<li><b>Classification:</b> stratify by the image label; group by camera, day or source.</li>
<li><b>Detection:</b> the unit is still the <i>image</i> (never split boxes of one photo across halves),
but one photo carries several labels, so plain stratification does not apply. <i>Iterative
stratification</i> balances the box counts per class across folds (section 8).</li>
<li><b>Segmentation:</b> the same, with pixel shares instead of box counts; a rare class with a few
large masks can end up entirely in one fold if you do not check (section 9).</li>
</ul>
In all three, deduplicate first (section 5), then split, then never look at the test half again.
</font></td></tr></table>

**Takeaway.** A split decides what the score means. Stratify so every class is scored; group by whatever
will be new when the model is deployed, so the score is the one you will see there; and measure leakage
before any model with one number, the share of test photos with a sibling in training. Leave-one-out is
for tiny datasets and is no protection against siblings; leave-one-group-out is. Section 7 picks the
metric to compute on that test half.

## 7. Metrics for the park

**Objective:** pick the metric that matches the use case, and see why accuracy is the wrong one for the
dog alert.

Section 4 ended with the laziest classifier: "always sheep" scores 98 % on a paddock day and never sees
a dog. That was a toy. This section does the same comparison on something closer to real life: a table of
**prepared predictions**, one per photo of `clean_images`, that looks like the output of a decent
classifier. Nothing is trained here (that is next week); the predictions are generated with a seed so that
everyone looks at the same table, and they are built to behave like many real models on imbalanced data:
right most of the time, and wrong mostly on the rare class.

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · Accuracy, recall, precision and the confusion matrix</b><br>
A <b>confusion matrix</b> counts photos by (true class, predicted class): rows are what the photo shows,
columns are what the classifier said, and the diagonal is the correct ones. Three numbers come out of it:
<ul>
<li><b>Accuracy</b> = diagonal / everything. One number for the whole table, dominated by the big
classes.</li>
<li><b>Recall</b> of a class = correct / <i>row</i> total: "of all the real dogs, how many did we catch?"
A missed dog (a dog called sheep) lowers it.</li>
<li><b>Precision</b> of a class = correct / <i>column</i> total: "of all the photos we called dog, how
many were dogs?" A false alarm (a sheep called dog) lowers it.</li>
</ul>
No metric is right in itself. The use case says which mistake is expensive, and that picks the metric.
</font></td></tr></table>

### Prepared predictions

The collapsed cell writes `data/cache/park_predictions.csv`, one row per photo with the true class and a
predicted class, and reads it back into `predictions`. Open the file if you like: it is the kind of table
you get from any classifier's test run.

In [ ]:
# @title Prepared predictions { display-mode: "form" }
# Fake classifier output for clean_images, fixed by SEED (no model): each class gets a share of wrong
# predictions, dog a large one, and each wrong prediction goes to a plausible look-alike class.
PRED_PATH = DATA_DIR / "cache" / "park_predictions.csv"
ERROR_SHARE = {"cat": 0.10, "dog": 0.50, "bird": 0.10, "horse": 0.10, "sheep": 0.10, "zebra": 0.10}
CONFUSED_WITH = {  # when the prediction is wrong, which class it tends to say
    "cat": ["dog"],
    "dog": ["sheep", "sheep", "sheep", "cat"],
    "bird": ["cat"],
    "horse": ["sheep"],
    "sheep": ["horse"],
    "zebra": ["horse"],
}


def generate_predictions(table: pd.DataFrame, seed: int) -> pd.DataFrame:
    """A DataFrame (id, file, true, pred) with ERROR_SHARE of each class predicted as a look-alike class."""
    rng = np.random.default_rng(seed)
    true = table["class"].to_numpy()
    pred = true.copy()
    for c in CLASSES:
        idx = np.flatnonzero(true == c)
        n_wrong = int(round(len(idx) * ERROR_SHARE[c]))
        wrong = rng.choice(idx, n_wrong, replace=False)
        pred[wrong] = rng.choice(CONFUSED_WITH[c], n_wrong)
    return pd.DataFrame({"id": table["id"].to_numpy(), "file": table["file"].to_numpy(),
                         "true": true, "pred": pred})


PRED_PATH.parent.mkdir(parents=True, exist_ok=True)
generate_predictions(clean_images, SEED).to_csv(PRED_PATH, index=False)
predictions = pd.read_csv(PRED_PATH)
print(f"{len(predictions)} predictions written to {PRED_PATH} and read back")
predictions.head()

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Read the first rows above: some predictions are wrong. Before computing anything, think about which
number the park's manager would quote and which one the sheep farmer would.
</font></td></tr></table>

**Q19.** 🟡 Guess the accuracy of `predictions` (over all photos) and the recall on dog. Will the two be
close, or far apart? Write both guesses down, then run.

In [ ]:
# Q19  Confusion matrix of the prepared predictions, then accuracy, recall and precision per class.
_true = predictions["true"].to_numpy()
_pred = predictions["pred"].to_numpy()
confusion = pd.DataFrame(0, index=CLASSES, columns=CLASSES, dtype=int)
for t, p in zip(_true, _pred, strict=True):
    confusion.loc[t, p] += 1
confusion.index.name, confusion.columns.name = "true", "predicted"

_diag = np.diag(confusion.to_numpy())
accuracy = float(_diag.sum() / len(predictions))
recall = pd.Series(_diag / confusion.sum(axis=1).replace(0, np.nan).to_numpy(), index=CLASSES)
precision = pd.Series(_diag / confusion.sum(axis=0).replace(0, np.nan).to_numpy(), index=CLASSES)
metrics = pd.DataFrame({"photos": confusion.sum(axis=1), "recall": recall.round(2),
                        "precision": precision.round(2)})
RESULTS["accuracy"] = accuracy
RESULTS["recall_dog"] = float(recall["dog"])
print(f"accuracy: {accuracy:.0%}   ·   recall on dog: {recall['dog']:.0%}   ·   precision on dog: "
      f"{precision['dog']:.0%}")
display(metrics.T)

# The confusion matrix, each row tinted in the true class's colour (darker = a larger share of that row).
fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axes[0]
_rgba = np.zeros((len(CLASSES), len(CLASSES), 4))
_row_share = confusion.to_numpy() / confusion.sum(axis=1).replace(0, 1).to_numpy()[:, None]
for i, c in enumerate(CLASSES):
    _rgba[i, :, :3] = mpl.colors.to_rgb(CLASS_COLORS[c])
    _rgba[i, :, 3] = 0.1 + 0.9 * _row_share[i]
ax.imshow(_rgba)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, str(confusion.iat[i, j]), ha="center", va="center", fontsize=9,
                color="white" if _row_share[i, j] > 0.5 else INK)
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES)), CLASSES)
for lab in ax.get_yticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("Confusion matrix (rows = true class)")
ax.grid(False)

ax = axes[1]
ax.bar(CLASSES, recall.to_numpy(), color=[CLASS_COLORS[c] for c in CLASSES])
ax.axhline(accuracy, color=INK, linestyle="--", linewidth=1, label=f"accuracy {accuracy:.0%}")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_ylabel("recall per class")
ax.set_title("Recall per class vs overall accuracy")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 One dashed line, six bars: accuracy sits near the tall bars because it is a photo-weighted average,
and the dog bar sits far below it. The dog row of the matrix says where the dogs went: most of the missed
ones were called *sheep*, which on a paddock camera is exactly the frame that should have raised the alert.

**Worked answer (Q19).** Accuracy is high (around 80 %) and recall on dog is around 50 %: the two are
far apart. Accuracy is the same number the park's
manager would be happy with, and it is reached while every second dog walks through the paddock unnoticed.
The exact values depend on the table (dedup removed some photos in section 5), so compare the gap, not
the digits.

Instead of staring at the table, look at the mistakes that cost the most. These are the dogs the alert
would have missed.

In [ ]:
# The photos the alert missed: true dog, predicted something else (framed in the dog colour).
N_MISSED = 6
_missed = predictions[(predictions["true"] == "dog") & (predictions["pred"] != "dog")].head(N_MISSED)
_missed_rows = clean_images.set_index("id").loc[_missed["id"]].reset_index()
_missed_rows["pred"] = _missed["pred"].to_numpy()
show_rows(_missed_rows, f"{len(_missed)} of the {int((_true == 'dog').sum())} dogs: missed",
          caption=lambda row: f"true dog · said {row['pred']}")

💡 Nothing about these photos is special: they are ordinary dogs that a classifier biased towards the big
classes labelled as something else. Accuracy never looked at this row; recall on dog is nothing but this
row.

### The same classes, a different shop

Now move the same six classes to a **pet-food shop**. Its webcam looks at the customer's pet and the
screen recommends food for it. Horses and zebras never walk in, so their rows of the matrix are empty and
their recall does not matter. A false "cat" is cheap: the customer smiles, taps "no, a dog", and the screen
corrects itself. What costs the shop is *insisting*: calling a dog a cat and recommending cat food with
confidence, so the number to watch is **precision on cat and on dog**, "when we say cat, is it a cat?".
Same classes, same predictions, same confusion matrix, different column: the paddock reads the dog *row*
(recall), the shop reads the cat and dog *columns* (precision). The metric is a property of the use case,
not of the data.

In [ ]:
# The shop's view of the same predictions: precision on cat and dog (the columns), horse and zebra ignored.
_shop = metrics.loc[["cat", "dog"], ["precision", "recall"]]
print("Pet-food shop, same predictions:")
display(_shop.T)
print(f"The paddock quotes recall on dog ({recall['dog']:.0%}); the shop quotes precision on cat "
      f"({precision['cat']:.0%}) and dog ({precision['dog']:.0%}).")

💡 The dog column is precise (few non-dogs are called dog) even though the dog row is poor: this classifier
would be fine for the shop and useless for the paddock, on the very same test set.

### ✍️ Pick the alert metric

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>ALERT_METRIC</code></b><br>
The paddock alert fires when a dog is in front of the sheep camera. Missing a dog is worse than a false
alarm (the park story, section 0). Replace the <code>...</code> with one of <code>"accuracy"</code>,
<code>"precision"</code> or <code>"recall"</code>: the metric, computed on the dog class, that the alert
should be judged on. The check cell tells you whether it agrees and continues with the reference choice
either way, like the <code>use()</code> cells.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
ALERT_METRIC = ...  # one of "accuracy", "precision", "recall": the number the paddock alert is judged on

In [ ]:
# Check ALERT_METRIC against the reference choice; the notebook continues with the reference either way.
_REFERENCE_ALERT_METRIC = "recall"
_WHY = {
    "accuracy": "accuracy is dominated by the big classes: 'always sheep' scores 98 % on a paddock day and "
                "catches no dog (section 4).",
    "precision": "precision on dog counts false alarms. A false alarm costs the farmer a look at the screen; "
                 "a missed dog costs sheep.",
}
_dog_scores = {"accuracy": accuracy, "precision": float(precision["dog"]), "recall": float(recall["dog"])}
if ALERT_METRIC is ...:
    print("✍️  ALERT_METRIC: not chosen yet, using the reference choice.")
elif ALERT_METRIC not in _dog_scores:
    raise SystemExit(f"ALERT_METRIC must be one of {sorted(_dog_scores)}, not {ALERT_METRIC!r}")
elif ALERT_METRIC == _REFERENCE_ALERT_METRIC:
    print(f"✅ ALERT_METRIC = {ALERT_METRIC!r} matches the reference choice.")
else:
    print(f"❌ ALERT_METRIC = {ALERT_METRIC!r}: {_WHY[ALERT_METRIC]} Using the reference.")
alert_metric = _REFERENCE_ALERT_METRIC
RESULTS["alert_metric"] = alert_metric
print(f"The paddock alert is judged on {alert_metric} on dog: {_dog_scores[alert_metric]:.0%} on the "
      f"prepared predictions (accuracy would have said {accuracy:.0%}).")

**Worked answer.** `"recall"`. The alert exists to catch the rare event, and recall on dog is the share
of real dogs it catches. Precision on dog would be the shop's choice, and accuracy nobody's: it is the
number that rewards ignoring the dog. Next week, when a real model is trained, this is the number to put
on the validation curve, not accuracy.

### Quick check

**Q20.** 🟢 The shop's webcam now also greets *rabbits* (a new, rare class). A customer with a rabbit is
shown dog food once a week; a customer with a dog is never shown rabbit food. Which metric did the shop
just lose, and which is still fine?

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>QUIZ_ANSWER</code></b><br>
Set <code>QUIZ_ANSWER</code> to one letter:<br>
<b>A</b> recall on rabbit dropped, precision on dog dropped<br>
<b>B</b> recall on rabbit dropped, precision on dog is fine<br>
<b>C</b> precision on rabbit dropped, recall on dog dropped<br>
<b>D</b> accuracy dropped, nothing else changed
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
QUIZ_ANSWER = None  # "A", "B", "C" or "D"

_QUIZ_KEY = "A"
_QUIZ_WHY = {
    "A": "a rabbit called dog is a miss in the rabbit row (recall on rabbit) and a wrong entry in the dog "
         "column (precision on dog). One mistake, two metrics.",
    "B": "a rabbit called dog also lands in the dog column: everything called dog that is not a dog lowers "
         "precision on dog.",
    "C": "no rabbit is predicted for a dog, so the rabbit column (precision on rabbit) and the dog row "
         "(recall on dog) are untouched.",
    "D": "accuracy barely moves (rabbits are rare), and two per-class metrics did change.",
}
if QUIZ_ANSWER is None:
    print("✏️  QUIZ_ANSWER not set yet; the answer is shown below.")
elif str(QUIZ_ANSWER).upper() not in _QUIZ_WHY:
    raise SystemExit(f"QUIZ_ANSWER must be one of A, B, C, D, not {QUIZ_ANSWER!r}")
elif str(QUIZ_ANSWER).upper() == _QUIZ_KEY:
    print(f"✅ {_QUIZ_KEY}: {_QUIZ_WHY[_QUIZ_KEY]}")
else:
    print(f"❌ {str(QUIZ_ANSWER).upper()}: {_QUIZ_WHY[str(QUIZ_ANSWER).upper()]}")

**Worked answer (Q20).** A. One mistake, "rabbit called dog", sits in the rabbit *row* and the dog
*column* at once, so it lowers recall on rabbit and precision on dog. Precision on rabbit and recall on
dog only move when a dog is called rabbit, which never happens. Accuracy hardly notices a weekly rabbit.

### Across tasks: counting and outlining

The park's two other jobs have their own numbers, and the same lesson applies: the use case names the
expensive mistake, and the metric follows.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · Counting error, mAP, IoU and Dice</b><br>
<ul>
<li><b>Detection (the daily headcount):</b> the park wants <b>counting error</b>, |predicted − true|
animals per photo or per day, summed over the day. It is the number the farmer reads, and it hides
compensating mistakes (one missed sheep plus one bush called sheep is a perfect count). The research
metric, <b>mAP</b> (mean average precision), scores every box: a box counts as correct when it overlaps a
true box enough (IoU ≥ 0.5), precision and recall are traced as the confidence threshold moves, the area
under that curve is averaged over classes. Tiny far-away birds (section 8) hurt mAP most.</li>
<li><b>Segmentation (how much of the enclosure each animal uses):</b> <b>IoU</b> (intersection over
union) of the predicted and the true mask, per class, then the mean: the park's "share of the enclosure"
is a pixel count, so a mask metric is the honest one. <b>Dice</b> = 2·|A∩B| / (|A|+|B|) is the same
overlap counted more kindly (Dice ≥ IoU always). Birds are ~6 % of the pixels (section 9), so the mean
over classes, not over pixels, is the one that notices them.</li>
</ul>
In all three tasks the overall number (accuracy, total count, pixel accuracy) is the one that hides the
rare class, and the per-class number (recall on dog, per-class count error, per-class IoU) is the one
that shows it.
</font></td></tr></table>

**Q21.** 🟡 A detector looks at a paddock photo with 5 sheep. It finds 4 of them and also calls a bush a
sheep. What is the counting error for that photo, and how many mistakes would mAP see? Decide, then run.

In [ ]:
# Q21  Counting error vs box-level mistakes on one photo, and IoU vs Dice of a shifted box.
true_count, found, false_boxes = 5, 4, 1
counting_error = abs((found + false_boxes) - true_count)
_box_mistakes = true_count - found + false_boxes
print(f"headcount: {found + false_boxes} predicted vs {true_count} true → counting error {counting_error}; "
      f"box-level mistakes: {true_count - found} missed + {false_boxes} false = {_box_mistakes}")


def box_iou_dice(a: tuple, b: tuple) -> tuple[float, float]:
    """IoU and Dice of two boxes given as (x, y, w, h)."""
    ix = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = ix * iy
    area_a, area_b = a[2] * a[3], b[2] * b[3]
    return inter / (area_a + area_b - inter), 2 * inter / (area_a + area_b)


_true_box = (20, 20, 60, 40)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, shift in zip(axes, [0, 15, 30], strict=True):
    _pred_box = (_true_box[0] + shift, _true_box[1] + shift // 2, _true_box[2], _true_box[3])
    iou, dice = box_iou_dice(_true_box, _pred_box)
    ax.add_patch(mpl.patches.Rectangle(_true_box[:2], *_true_box[2:], fill=True, alpha=0.35,
                                       color=CLASS_COLORS["sheep"], label="true"))
    ax.add_patch(mpl.patches.Rectangle(_pred_box[:2], *_pred_box[2:], fill=False, lw=2,
                                       edgecolor=CLASS_COLORS["dog"], label="predicted"))
    ax.set_xlim(0, 120)
    ax.set_ylim(90, 0)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(f"shift {shift} px · IoU {iou:.2f} · Dice {dice:.2f}", fontsize=9)
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
fig.suptitle("The same overlap, two scores: Dice is always the kinder one", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The headcount is off by zero while the detector made two mistakes: a counting error is what the farmer
wants to read, and mAP is what tells you the detector is not yet trustworthy. On the boxes, IoU and Dice
rank the same predictions in the same order; only their scale differs, so compare numbers across papers
with care.

**Worked answer (Q21).** Counting error 0 (5 predicted, 5 true), box-level mistakes 2 (one missed sheep,
one false box). The first number is the park's, the second is the one you fix the model with; a headcount
that is right for the wrong reasons will be wrong on the next photo. For the sheep-paddock alert the same
split holds: one summary number for the manager, the per-class number for the engineer.

## 8. Same data, boxes

**Objective:** see what imbalance, object size, position and label quality mean once the park wants a
*headcount*, and find one box that an annotator forgot.

Sections 4 to 7 treated each photo as one label. The park's second job, **count** the sheep, zebras and
birds every day, needs a **box per animal** instead, and the same photos now tell a different story.
`annotations` already holds one row per animal with its box as `[x, y, w, h]` in pixels (COCO format: the
top-left corner and the size). This section keeps the photos of `clean_images` and looks at their boxes.

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · What changes when the unit is a box</b><br>
A classifier sees one example per photo. A detector sees one example per <b>box</b>, so the photo with
twelve sheep weighs twelve times the photo with one cat. Four things follow:
<ul>
<li><b>Imbalance is counted in boxes</b>, not photos. The class with the most photos is not the class with
the most boxes.</li>
<li><b>Size matters.</b> COCO calls a box <i>small</i> under 32×32 px, <i>medium</i> up to 96×96 px, and
<i>large</i> above. Detectors score far worse on small boxes, and the park's far-away birds are small.</li>
<li><b>Position matters.</b> If every animal sits in the middle of the frame, the detector learns
"look in the middle", and the fence-edge sheep is missed.</li>
<li><b>Crowding matters.</b> Overlapping boxes of the same class are the hard case for both the detector
and the annotator, who may simply forget one.</li>
</ul>
</font></td></tr></table>

The collapsed cell defines `show_boxes`, a gallery that draws each photo's boxes in its class colour.

In [ ]:
# @title Box helpers (draw boxes on thumbnails) { display-mode: "form" }
def box_area_bucket(w: float, h: float) -> str:
    """COCO's size buckets from a box's width and height in pixels: small < 32², medium < 96², else large."""
    area = w * h
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"


def show_boxes(rows_df: pd.DataFrame, boxes_df: pd.DataFrame, title: str, caption=None, cols: int = 4,
               size: int = 384) -> None:
    """Gallery of the photos in `rows_df` with their boxes from `boxes_df` drawn in the class colour.

    `caption(row) -> str` writes the title above each photo; by default the class and the number of boxes.
    """
    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(3.4 * cols, 3.2 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        thumb = load_thumb(row["path"], size)
        scale = thumb.width / row["width"]
        ax.axis("on")
        ax.imshow(thumb)
        own = boxes_df[boxes_df["image_id"] == row["id"]]
        for _, b in own.iterrows():
            x, y, w, h = b["bbox"]
            ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False,
                                               lw=2, edgecolor=CLASS_COLORS[b["class"]]))
        frame(ax, CLASS_COLORS[row["class"]])
        text = caption(row) if caption else f"{row['class']} · {len(own)} boxes"
        ax.set_title(text, fontsize=9)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

### Boxes per class vs photos per class

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Section 4 ranked the classes by photos: cat and dog on top, sheep at the bottom. Now the same photos are
counted by animal.
</font></td></tr></table>

**Q22.** 🟡 Will the ranking by *boxes* be the same as the ranking by *photos*? Name the class you expect to
climb most, then run the cell. It needs your first function of the section.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>count_boxes_per_class</code></b><br>
Write the function in the next cell: given an annotations table (one row per animal, with a
<code>class</code> column), return a dict <code>{class: number of boxes}</code> with <b>every</b> class in
<code>CLASSES</code> as a key, counts as plain <code>int</code>s. The <code>use()</code> line checks it
against the reference and continues with the reference either way.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def count_boxes_per_class(annotations):
    """Number of boxes (rows) per class.

    Inputs: `annotations`, a DataFrame with one row per animal and a `class` column.
    Output: a dict {class: int} with every class of CLASSES as a key (0 when a class has no box).
    """
    ...


def _ref_count_boxes_per_class(annotations):
    counts = annotations["class"].value_counts()
    return {c: int(counts.get(c, 0)) for c in CLASSES}


boxes = annotations[annotations["image_id"].isin(clean_images["id"])].reset_index(drop=True)
boxes_per_class = use(count_boxes_per_class, boxes, reference=_ref_count_boxes_per_class)

In [ ]:
# Q22  Boxes per class next to photos per class, both ranked, on the photos of clean_images.
photos_per_class = {c: int((clean_images["class"] == c).sum()) for c in CLASSES}
RESULTS["boxes_per_class"] = boxes_per_class
RESULTS["n_boxes"] = int(len(boxes))

_by_photos = sorted(CLASSES, key=lambda c: -photos_per_class[c])
_by_boxes = sorted(CLASSES, key=lambda c: -boxes_per_class[c])
print(f"{len(clean_images)} photos · {len(boxes)} boxes")
print("ranked by photos: " + " > ".join(f"{c} {photos_per_class[c]}" for c in _by_photos))
print("ranked by boxes:  " + " > ".join(f"{c} {boxes_per_class[c]}" for c in _by_boxes))
for c in CLASSES:
    move = _by_photos.index(c) - _by_boxes.index(c)
    if move:
        print(f"  {c}: rank {_by_photos.index(c) + 1} by photos → rank {_by_boxes.index(c) + 1} by boxes "
              f"({'↑' if move > 0 else '↓'}{abs(move)})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=False)
for ax, counts, order, what in zip(axes, [photos_per_class, boxes_per_class], [_by_photos, _by_boxes],
                                   ["photos", "boxes"], strict=True):
    ax.bar(order, [counts[c] for c in order], color=[CLASS_COLORS[c] for c in order])
    ax.set_title(f"{what} per class (ranked)")
    ax.set_ylabel(what)
    for lab in ax.get_xticklabels():
        lab.set_color(CLASS_COLORS[lab.get_text()])
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Same photos, two rankings: the flock classes (bird, sheep, zebra) climb because one photo of them holds
many animals, and cat and dog sink because they are photographed alone. A detector trained on this set is
*not* short of sheep examples the way a classifier is.

**Worked answer (Q22).** No. On the full set bird climbs from 4th by photos to 1st by boxes, and sheep from
last to 2nd, while cat and dog drop to the bottom. The exact ranks depend on the photos you have (the smoke
subset is smaller), so compare the *shapes* of the two charts, not the digits. Say which unit you counted
every time you say "imbalance": the headcount model and the alert model see different ones.

### Box sizes: small, medium, large

**Q23.** 🟢 Which class do you expect to have the smallest boxes? Pick one, then run the cell: it buckets
every box with COCO's thresholds and shows the smallest and largest boxes in the set.

In [ ]:
# Q23  Box area per class on a log scale, COCO's small/medium/large buckets, and the extreme boxes.
boxes["w"] = boxes["bbox"].map(lambda b: b[2])
boxes["h"] = boxes["bbox"].map(lambda b: b[3])
boxes["box_area"] = boxes["w"] * boxes["h"]
boxes["side"] = np.sqrt(boxes["box_area"])  # the side of a square of the same area, in pixels
boxes["bucket"] = [box_area_bucket(w, h) for w, h in zip(boxes["w"], boxes["h"], strict=True)]
bucket_share = (boxes.groupby("class")["bucket"].value_counts(normalize=True).unstack(fill_value=0)
                .reindex(index=CLASSES, columns=["small", "medium", "large"], fill_value=0))
RESULTS["small_box_share"] = {c: float(bucket_share.loc[c, "small"]) for c in CLASSES}
print("share of each class's boxes that is small / medium / large:")
display((bucket_share * 100).round(0).astype(int).T)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axes[0]
_sides = [boxes.loc[boxes["class"] == c, "side"].clip(lower=4).to_numpy() for c in CLASSES]
_parts = ax.boxplot(_sides, tick_labels=CLASSES, patch_artist=True, showfliers=False, widths=0.6)
for patch, c in zip(_parts["boxes"], CLASSES, strict=True):
    patch.set_facecolor(CLASS_COLORS[c])
    patch.set_alpha(0.75)
for med in _parts["medians"]:
    med.set_color(INK)
ax.set_yscale("log")
for thr, name in [(32, "small ↓"), (96, "large ↑")]:
    ax.axhline(thr, color=INK, linestyle="--", linewidth=1)
    ax.text(0.5, thr, f" {thr} px · {name}", va="bottom", fontsize=8, color=INK)
ax.set_ylabel("box side √(w·h), px (log)")
ax.set_title("Box size per class")
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])

ax = axes[1]
_bottom = np.zeros(len(CLASSES))
for bucket, alpha in [("small", 1.0), ("medium", 0.55), ("large", 0.25)]:
    vals = bucket_share[bucket].to_numpy()
    ax.bar(CLASSES, vals, bottom=_bottom, color=[CLASS_COLORS[c] for c in CLASSES], alpha=alpha,
           label=bucket, edgecolor="white")
    _bottom += vals
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("Share of small (solid) / medium / large (pale) boxes")
ax.legend(frameon=False, fontsize=8, loc="lower right")
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
fig.tight_layout()
display(fig)
plt.close(fig)

N_EXTREME_BOXES = 2 if SMOKE else 4  # photos with the smallest / largest boxes to show (N-014)
_smallest = boxes.sort_values(["box_area", "id"]).drop_duplicates("image_id").head(N_EXTREME_BOXES)
_largest = boxes.sort_values(["box_area", "id"], ascending=[False, True]).drop_duplicates("image_id").head(
    N_EXTREME_BOXES)
_extreme_ids = list(_smallest["image_id"]) + list(_largest["image_id"])
_rows = clean_images.set_index("id").loc[_extreme_ids].reset_index()
_side_of = dict(zip(_extreme_ids, list(_smallest["side"]) + list(_largest["side"]), strict=True))
show_boxes(_rows, boxes, f"the {N_EXTREME_BOXES} smallest and the {N_EXTREME_BOXES} largest boxes",
           caption=lambda r: f"{r['class']} · box side {_side_of[r['id']]:.0f} px", cols=2 * N_EXTREME_BOXES)

💡 The box plot is on a log scale because the spread is enormous: a bird can be a few pixels on a pond
and a horse can fill the frame. The small share is the number to watch: those boxes are the ones every
detector misses first, and in the park they are exactly the far-away birds the headcount needs.

**Worked answer (Q23).** Bird, by a wide margin: on the full set a large share of its boxes is *small* in
COCO's sense, while cat, dog and horse are mostly *large*. Whatever subset you run on, look at the
smallest-box gallery: a box of 10 px is a speck, and resizing the photo to a network's input makes it
smaller still (section 3).

### Where the animals are: a box-centre heatmap

Each box has a centre. Dividing it by the photo's width and height puts every centre on the same
0–1 square, whatever the photo's size, and a 2-D histogram of those points shows where the annotator
(and so the detector) expects an animal. The next cell draws one small heatmap per class.

In [ ]:
# Box centres on the unit square, one heatmap per class in its own colour (white = no boxes there).
_size = clean_images.set_index("id")[["width", "height"]]
_img_w = boxes["image_id"].map(_size["width"])
_img_h = boxes["image_id"].map(_size["height"])
boxes["cx"] = (boxes["bbox"].map(lambda b: b[0] + b[2] / 2) / _img_w).clip(0, 1)
boxes["cy"] = (boxes["bbox"].map(lambda b: b[1] + b[3] / 2) / _img_h).clip(0, 1)
HEAT_BINS = 6 if SMOKE else 12  # coarser cells with few boxes (N-014)
_edge_share = float(((boxes["cx"] < 0.2) | (boxes["cx"] > 0.8) | (boxes["cy"] < 0.2) | (boxes["cy"] > 0.8))
                    .mean())
RESULTS["edge_box_share"] = _edge_share

fig, axes = plt.subplots(1, len(CLASSES), figsize=(2.3 * len(CLASSES), 2.8))
for ax, c in zip(axes, CLASSES, strict=True):
    own = boxes[boxes["class"] == c]
    cmap = mpl.colors.LinearSegmentedColormap.from_list(c, ["white", CLASS_COLORS[c]])
    heat, _, _ = np.histogram2d(own["cy"], own["cx"], bins=HEAT_BINS, range=[[0, 1], [0, 1]])
    ax.imshow(heat, cmap=cmap, extent=[0, 1, 1, 0], vmin=0)
    ax.set_title(f"{c} · {len(own)} boxes", fontsize=9, color=CLASS_COLORS[c])
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.grid(False)
axes[0].set_ylabel("box centre y (0 = top)")
fig.suptitle(f"Where the box centres are · {_edge_share:.0%} of all boxes are in the outer 20 % of the frame",
             color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The centre of the frame is darker in every panel: photographers point the camera at the animal. A
park camera does not, so a detector that learned "animals live in the middle" will miss the sheep at the
fence. The flock classes (sheep, bird, zebra) spread wider because a flock fills the frame, and that is the
one bias the Play Dataset shares with the paddock.

### Animals per photo, and crowding

Section 4 counted animals per photo to ask "which label then?". For the headcount the question is
different: how often do boxes of the same class **overlap**? Two sheep standing behind each other give
two boxes that share pixels, which is hard for the detector (which one is which?) and easy for the
annotator to get wrong (one box for two sheep, or no box for the hidden one).

**Q24.** 🟡 Which class will have the most overlapping pairs of boxes per photo? Decide, then run the
cell: for every photo it computes the overlap (IoU, section 7) of every pair of its boxes.

In [ ]:
# Q24  Boxes per photo per class, and crowding: pairs of boxes in one photo with IoU above a threshold.
OVERLAP_IOU = 0.1  # two boxes "overlap" when their IoU is above this


def overlap_pairs(boxes_df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """One row per photo: n_boxes, n_pairs (all pairs) and n_overlapping (pairs with IoU > threshold)."""
    rows = []
    for image_id, own in boxes_df.groupby("image_id"):
        bb = [tuple(b) for b in own["bbox"]]
        pairs = [(i, j) for i in range(len(bb)) for j in range(i + 1, len(bb))]
        n_over = sum(box_iou_dice(bb[i], bb[j])[0] > threshold for i, j in pairs)
        rows.append({"image_id": image_id, "class": own["class"].iloc[0], "n_boxes": len(bb),
                     "n_pairs": len(bb) * (len(bb) - 1) // 2, "n_overlapping": int(n_over)})
    return pd.DataFrame(rows)


crowding = overlap_pairs(boxes, OVERLAP_IOU)
_per_class = crowding.groupby("class").agg(photos=("image_id", "size"), boxes_per_photo=("n_boxes", "mean"),
                                           crowded_photos=("n_overlapping", lambda s: float((s > 0).mean())),
                                           overlapping_pairs=("n_overlapping", "sum")).reindex(CLASSES)
_per_class["boxes_per_photo"] = _per_class["boxes_per_photo"].round(1)
_per_class["crowded_photos"] = (_per_class["crowded_photos"] * 100).round(0).astype(int).astype(str) + " %"
RESULTS["crowded_photo_share"] = float((crowding["n_overlapping"] > 0).mean())
_n_crowded = int((crowding["n_overlapping"] > 0).sum())
print(f"{_n_crowded} of {len(crowding)} photos have at least one pair of boxes with IoU > {OVERLAP_IOU} "
      f"({RESULTS['crowded_photo_share']:.0%}).")
display(_per_class.T)

N_CROWDED_BOXES = 2 if SMOKE else 4  # most crowded photos to show with their boxes (N-014)
_most = crowding.sort_values(["n_overlapping", "n_boxes", "image_id"], ascending=[False, False, True]).head(
    N_CROWDED_BOXES)
_rows = clean_images.set_index("id").loc[_most["image_id"]].reset_index()
_over_of = dict(zip(_most["image_id"], _most["n_overlapping"], strict=True))
show_boxes(_rows, boxes, "the most crowded photos (every box drawn)",
           caption=lambda r: f"{r['n_animals']} × {r['class']} · {_over_of[r['id']]} overlapping pairs",
           cols=N_CROWDED_BOXES)

💡 Look at the crowded gallery and try to count the animals *without* the boxes: that is the annotator's
job, and the detector's. Where boxes pile up, a missing or doubled box is a matter of attention, which is
why the next part plants one.

**Worked answer (Q24).** The flock classes: sheep and zebra on the full set (herds stand close), with
bird next (flocks on water). Cats and dogs are mostly alone, so they have no pairs at all. On a small
subset the leader can change, so read the `crowded_photos` row: the share of a class's photos with at
least one overlapping pair.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · What handles each of these later (next week and beyond)</b><br>
Each finding above has a standard answer on the model side. Know the name, so that the data report can
say "we saw this, the training recipe handles it with …":
<ul>
<li><b>Imbalance in boxes</b> (bird 1st, cat last) → a <b>weighted</b> or <b>focal loss</b>, which
down-weights the easy, frequent boxes and lets the rare class count.</li>
<li><b>Small boxes</b> (far-away birds) → <b>tiling</b>: cut the photo into overlapping crops and detect in
each, so a 10 px bird becomes a 40 px bird.</li>
<li><b>A wide range of sizes</b> (a bird of 10 px and a horse of 400 px in one set) → <b>multi-scale</b>
training and feature pyramids, so the detector looks at several resolutions at once.</li>
<li><b>Splitting photos that hold several boxes</b> (a photo is "sheep" <i>and</i> has 12 boxes) →
<b>iterative stratification</b>, the multi-label version of the stratified split of section 6, which
balances the box counts per class across the splits, not only the photo counts.</li>
</ul>
None of these fixes a photo whose annotation is wrong. That is the data side, and it is yours.
</font></td></tr></table>

### Guided hunt: the box that is not there

The park sent its first batch of paddock photos to an annotation team and got the boxes back. The keeper
also wrote down a headcount per photo on the day. In the Play Dataset the keeper's headcount is the
`n_animals` column of `clean_images` (counted from every instance in section 0), and the team's boxes are
`park_boxes`, built by the collapsed cell: it is `boxes` with **one box removed** from a photo of a small
flock. Which photo, and which animal, is decided at runtime from `SEED`.

In [ ]:
# @title Plant the hunt (one box removed, seeded) { display-mode: "form" }
def plant_hunt8(boxes_df: pd.DataFrame, table: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, dict]:
    """Return (park_boxes, key): `boxes_df` minus one box of a photo with ≥ 3 animals of one class.

    Sheep photos are preferred (the paddock story); otherwise the class with the most such photos. With no
    photo of 3 or more animals, the most crowded photo is used, so one box is always planted.
    """
    rng = random.Random(seed)
    counts = boxes_df.groupby("image_id").size()
    class_of = table.set_index("id")["class"]
    candidates = [int(i) for i in counts[counts >= 3].index]
    sheep = [i for i in candidates if class_of.get(i) == "sheep"]
    if sheep:
        candidates = sheep
    if not candidates:
        candidates = [int(counts.idxmax())]
    image_id = rng.choice(sorted(candidates))
    own = boxes_df[boxes_df["image_id"] == image_id]
    ann_id = int(rng.choice(sorted(own["id"])))
    park = boxes_df[boxes_df["id"] != ann_id].reset_index(drop=True)
    return park, {"image_id": image_id, "ann_id": ann_id, "class": str(class_of.get(image_id)),
                  "n_animals": int(counts[image_id])}


park_boxes, HUNT8_KEY = plant_hunt8(boxes, clean_images, SEED)
print(f"park_boxes: {len(park_boxes)} boxes for {park_boxes['image_id'].nunique()} photos "
      f"(boxes had {len(boxes)}). One box is missing. Find the photo.")

**Q25.** 🟡 You cannot look at hundreds of photos. Which two columns, compared per photo, point straight at
the one with a missing box? Say it in one sentence, then run the cell.

In [ ]:
# Q25  The keeper's headcount vs the team's box count, per photo: any photo where they differ is suspect.
box_counts = park_boxes.groupby("image_id").size()
headcount = clean_images.set_index("id")["n_animals"]
missing_boxes = (headcount - box_counts.reindex(headcount.index, fill_value=0))
suspects = missing_boxes[missing_boxes != 0].sort_index()
hunt8_found = int(HUNT8_KEY["image_id"] in suspects.index)
RESULTS["hunt8"] = {"planted": 1, "found": hunt8_found}
print(f"{len(suspects)} photo(s) where the headcount and the box count differ:")
for image_id, diff in suspects.items():
    print(f"  photo {image_id}: headcount {headcount[image_id]}, boxes {int(box_counts.get(image_id, 0))} "
          f"({'missing' if diff > 0 else 'extra'} {abs(int(diff))})")
print(f"planted: photo {HUNT8_KEY['image_id']} ({HUNT8_KEY['n_animals']} × {HUNT8_KEY['class']}, one box "
      f"removed) · found: {'✅' if hunt8_found else '❌'}")

_rows = clean_images.set_index("id").loc[list(suspects.index)].reset_index()
show_boxes(_rows, park_boxes, "the suspect photo(s): one animal has no box",
           caption=lambda r: f"{r['class']} · headcount {r['n_animals']} · "
                             f"{int(box_counts.get(r['id'], 0))} boxes")

💡 Two counts of the same thing from two sources, and a photo where they disagree: that is the whole
trick, and it costs nothing. Without a second count (the keeper's), the only way is a visual check list,
photo by photo, which is why annotation teams ask for a per-image count in the brief.

**Worked answer (Q25).** Compare the keeper's headcount (`n_animals`) with the number of boxes per photo
in `park_boxes`. One photo has one box fewer than animals; the gallery shows the animal without a frame.
In section 9 the same comparison works between masks and boxes, and the Challenge hides another missing
box among other faults.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Counting different things across tasks</b><br>
"Sheep is 9 % of the dataset" is true by photos and false by boxes (22 %). "Bird is rare" is true by
photos and false by boxes, and true again by pixels (section 9). Whenever you report imbalance, size or a
split, say what the unit is: photo, box or pixel. The park's three jobs count three different things from
the same photos.
</font></td></tr></table>

## 9. Same data, masks

**Objective:** see what imbalance, object size, position and thin parts mean once the park wants an
*outline* of every animal, and find five ugly masks hidden among the real ones.

The park's third job, **outline** each animal to see how much of the enclosure it uses, needs a **mask**:
for every animal, the exact pixels it covers. `annotations` already holds one in its `segmentation` column,
either as a COCO *polygon* (the x, y points an annotator clicked around the animal) or, for crowds, as
*RLE* (run-length encoding: "300 background pixels, then 25 animal pixels, then …"). `pycocotools`, the
COCO team's own library, turns both into a 0/1 array the size of the photo. This section keeps the photos
of `clean_images` and counts what is inside those arrays.

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · What changes when the unit is a pixel</b><br>
A segmentation model makes one decision per <b>pixel</b>: which class is this pixel, or is it background?
Four things follow:
<ul>
<li><b>Imbalance is counted in pixels, and background is a class too.</b> Most of every photo is grass,
sky and water, and a big horse outweighs a whole flock of far-away birds.</li>
<li><b>Size is everything.</b> A bird of 10×10 px is 100 decisions; a horse filling the frame is
100,000. The loss hears the horse a thousand times louder.</li>
<li><b>Thin parts</b> (legs, tails, beaks, a bird far away) are few pixels, and they are the first to
vanish when the photo is shrunk to the network's input size.</li>
<li><b>Mask quality is hard to see.</b> A box is four numbers; a mask is dozens of clicks, so masks are
sloppier, and a bad one looks like any other row in a table.</li>
</ul>
</font></td></tr></table>

The collapsed cell defines the mask tools: decode and encode RLE, measure a mask's shape, and `show_masks`,
a gallery that paints each mask in its class colour over the photo, with its annotated box dashed.

In [ ]:
# @title Mask helpers (decode, encode, measure, draw) { display-mode: "form" }
import cv2
from pycocotools import mask as mask_utils

MIN_JUDGED_AREA = 32**2  # masks under COCO's "small" limit (32×32 px) are too few pixels to judge their shape
THIN_PX = 7  # parts narrower than this many photo pixels count as thin (about 2 px at 224×224)
BLOB_SHARE = 0.005  # pieces and holes under 0.5 % of the mask (or 16 px) are rasterisation noise, not shape


def to_rle(segmentation, height: int, width: int) -> dict:
    """A COCO segmentation (polygon list, uncompressed or compressed RLE) as one compressed RLE."""
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation["counts"], list):
        return mask_utils.frPyObjects(segmentation, height, width)
    return segmentation


def decode(rle: dict) -> np.ndarray:
    """RLE → 0/1 uint8 array of the photo's size (C-ordered, so OpenCV accepts it)."""
    return np.ascontiguousarray(mask_utils.decode(rle))


def encode(mask: np.ndarray) -> dict:
    """0/1 array → compressed RLE, the way COCO stores masks."""
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))


def thin_part(mask: np.ndarray) -> np.ndarray:
    """The pixels of `mask` in parts narrower than THIN_PX: what an opening with a THIN_PX disc removes."""
    disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (THIN_PX, THIN_PX))
    return mask & (1 - cv2.morphologyEx(mask, cv2.MORPH_OPEN, disc))


def mask_shape(mask: np.ndarray, bbox) -> dict:
    """Shape numbers of one mask (0/1 array) against its annotated box (x, y, w, h).

    box_iou: IoU of the mask's own tight box with the annotated box · fill: mask area / box area ·
    pieces, holes: separate parts and enclosed gaps · solidity: mask area / area of its convex hull (a rubber
    band around it; 1.0 = no dents at all) · thin: share of the mask's pixels in thin parts.
    """
    area = int(mask.sum())
    if area == 0:
        return {"box_iou": 0.0, "fill": 0.0, "pieces": 0, "holes": 0, "solidity": 0.0, "thin": 0.0}
    mx, my, mw, mh = cv2.boundingRect(mask)
    crop = np.pad(mask[my:my + mh, mx:mx + mw], 1)
    min_blob = max(16, BLOB_SHARE * area)
    _, _, parts, _ = cv2.connectedComponentsWithStats(crop, connectivity=8)
    pieces = int((parts[1:, cv2.CC_STAT_AREA] >= min_blob).sum())
    _, labels, gaps, _ = cv2.connectedComponentsWithStats(1 - crop, connectivity=4)
    outside = labels[0, 0]  # the background that touches the border is not a hole
    holes = sum(int(gaps[i, cv2.CC_STAT_AREA] >= min_blob) for i in range(1, len(gaps)) if i != outside)
    contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = np.zeros_like(crop)
    cv2.fillPoly(hull, [cv2.convexHull(np.vstack(contours))], 1)
    w, h = bbox[2], bbox[3]
    return {"box_iou": box_iou_dice((mx, my, mw, mh), tuple(bbox))[0],
            "fill": area / max(w * h, 1),
            "pieces": pieces,
            "holes": holes,
            "solidity": area / max(int(hull.sum()), 1),
            "thin": int(thin_part(crop).sum()) / area}


def show_masks(rows_df: pd.DataFrame, masks_df: pd.DataFrame, title: str, caption=None, cols: int = 4,
               size: int = 384, mark_thin: bool = False) -> None:
    """Gallery of the photos in `rows_df` with every mask of `masks_df` (column `rle`) painted in its class
    colour and its annotated box dashed. `mark_thin=True` paints the thin parts dark.

    `caption(row) -> str` writes the title above each photo; by default the class and the number of masks.
    """
    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(3.4 * cols, 3.2 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        thumb = load_thumb(row["path"], size)
        scale = thumb.width / row["width"]
        overlay = np.zeros((thumb.height, thumb.width, 4))
        own = masks_df[masks_df["image_id"] == row["id"]]
        for _, m in own.iterrows():
            full = decode(m["rle"])
            small = cv2.resize(full, thumb.size, interpolation=cv2.INTER_NEAREST) > 0  # nearest: section 3
            overlay[small] = (*mpl.colors.to_rgb(CLASS_COLORS[m["class"]]), 0.6)
            if mark_thin:
                thin = cv2.resize(thin_part(full), thumb.size, interpolation=cv2.INTER_NEAREST) > 0
                overlay[thin] = (*mpl.colors.to_rgb(INK), 0.95)
        ax.axis("on")
        ax.imshow(thumb)
        ax.imshow(overlay)
        for _, m in own.iterrows():
            x, y, w, h = m["bbox"]
            ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False,
                                               lw=1.5, ls="--", edgecolor=CLASS_COLORS[m["class"]]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row) if caption else f"{row['class']} · {len(own)} masks", fontsize=9)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

The next cell turns every mask of `clean_images` into one compressed RLE (the `rle` column of a new table,
`masks`) and counts its pixels. RLE is small and fast: the union and the area of masks are computed
without ever building the full 0/1 array.

In [ ]:
# One row per mask of the photos in clean_images (crowd regions included), as RLE, with its pixel count.
_size = clean_images.set_index("id")[["width", "height"]]
masks = boxes[["id", "image_id", "class", "bbox", "iscrowd", "segmentation"]].copy()
masks["rle"] = [to_rle(seg, int(_size.at[i, "height"]), int(_size.at[i, "width"]))
                for seg, i in zip(masks["segmentation"], masks["image_id"], strict=True)]
masks["mask_area"] = [int(mask_utils.area(r)) for r in masks["rle"]]
masks = masks.drop(columns="segmentation")
print(f"{len(masks)} masks on {masks['image_id'].nunique()} photos "
      f"({int(masks['iscrowd'].sum())} of them crowd regions: one mask for a whole group)")
masks.head(3)

### Pixels per class, background included

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Section 4 counted photos and section 8 counted boxes. A segmentation model counts pixels, and every pixel
that is not an animal is <i>background</i>, which is a class like any other for the model.
</font></td></tr></table>

**Q26.** 🟢 What share of all pixels in the Play Dataset do you expect to be background: about a quarter,
a half, or most of them? And will bird be frequent or rare by pixels? Write down both guesses, then run the
cell: it merges each photo's masks into one (a pixel belongs to at most one animal) and counts.

In [ ]:
# Q26  Pixels per class including background (shares of all pixels), next to the photo and box shares.
photo_rle = {}  # one merged mask per photo, reused by the heatmaps below
_animal_px = dict.fromkeys(CLASSES, 0)
for image_id, own in masks.groupby("image_id"):
    photo_rle[image_id] = mask_utils.merge(list(own["rle"]), intersect=False)
    _animal_px[own["class"].iloc[0]] += int(mask_utils.area(photo_rle[image_id]))
_total_px = int((clean_images["width"] * clean_images["height"]).sum())
pixels_per_class = {c: _animal_px[c] / _total_px for c in CLASSES}
pixels_per_class["background"] = (_total_px - sum(_animal_px.values())) / _total_px
RESULTS["pixels_per_class"] = pixels_per_class

BACKGROUND_COLOR = "#D9DEDF"  # neutral grey: background is not an animal, so it gets no class colour
_units = {
    "photos": {c: photos_per_class[c] / sum(photos_per_class.values()) for c in CLASSES},
    "boxes": {c: boxes_per_class[c] / sum(boxes_per_class.values()) for c in CLASSES},
    "pixels": {c: _animal_px[c] / max(sum(_animal_px.values()), 1) for c in CLASSES},
}
print(f"{_total_px / 1e6:.0f} million pixels · background {pixels_per_class['background']:.1%} · "
      f"all animals together {1 - pixels_per_class['background']:.1%}")
print("share of ALL pixels:  " + " · ".join(f"{c} {pixels_per_class[c]:.1%}" for c in CLASSES))
print("among animals only, by photos / boxes / pixels:")
for c in CLASSES:
    print(f"  {c:6s} {_units['photos'][c]:5.0%}  {_units['boxes'][c]:5.0%}  {_units['pixels'][c]:5.0%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
_x = np.arange(len(CLASSES))
for k, (unit, alpha) in enumerate([("photos", 0.3), ("boxes", 0.6), ("pixels", 1.0)]):
    ax.bar(_x + (k - 1) * 0.27, [_units[unit][c] for c in CLASSES], width=0.27, alpha=alpha,
           color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="white", label=unit)
ax.set_xticks(_x, CLASSES)
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("Share among the animals: photos (pale) · boxes · pixels (solid)")
_legend = [mpl.patches.Patch(color=INK, alpha=a, label=u) for u, a in [("photos", 0.3), ("boxes", 0.6),
                                                                         ("pixels", 1.0)]]
ax.legend(handles=_legend, frameon=False, fontsize=8)

ax = axes[1]
_left = 0.0
for name in ["background", *CLASSES]:
    share = pixels_per_class[name]
    ax.barh(0, share, left=_left, color=BACKGROUND_COLOR if name == "background" else CLASS_COLORS[name],
            edgecolor="white")
    if share > 0.06:
        ax.text(_left + share / 2, 0, f"{name}\n{share:.0%}", ha="center", va="center", fontsize=8, color=INK)
    _left += share
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("All pixels, background included")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Background dwarfs everything: a model that answers "background" for every pixel is right most of the
time and outlines nothing, the pixel version of section 4's "always sheep". Among the animals, the solid
bars reorder the classes once more: cat and dog come back, because a pet photo is a close-up.

**Worked answer (Q26).** Most of them: on the full set about 85 % of all pixels are background (the smoke
subset gives about 80 %). Bird is rare again: first by boxes in section 8, but about 1 % of all pixels
(6–7 % of the animal pixels), because most birds are far away. Photos, boxes, pixels: the
same dataset gives three rankings, which is why "bird is rare" needs its unit every time.

### How big is a mask, and how much of its box does it fill?

A mask always sits inside its box, so a second number comes for free: the **fill ratio**, mask area
divided by box area. A box around a curled-up cat is mostly cat; a box around a horse is mostly the air
between its legs. The next cell measures every mask's shape (it decodes each one, so it takes a few
seconds) and plots mask size and fill ratio per class. Crowd regions are left out: they outline a group,
not an animal.

**Q27.** 🟡 What fraction of its box does a typical animal fill: about 25 %, 50 % or 80 %? Which class do
you expect to fill its box least? Guess, then run.

In [ ]:
# Q27  Shape numbers of every (non-crowd) mask, then mask area and fill ratio per class.
animals = masks[masks["iscrowd"] == 0].reset_index(drop=True)
_shape = pd.DataFrame([mask_shape(decode(r), b)
                       for r, b in zip(animals["rle"], animals["bbox"], strict=True)])
animals = pd.concat([animals, _shape], axis=1)
_photo_px = animals["image_id"].map(_size["width"] * _size["height"])
animals["photo_share"] = animals["mask_area"] / _photo_px
_fill = animals.groupby("class")["fill"].median().reindex(CLASSES)
RESULTS["median_fill"] = {c: float(_fill[c]) for c in CLASSES}
print(f"{len(animals)} animal masks measured · median fill ratio " + " · ".join(
    f"{c} {_fill[c]:.0%}" for c in CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, label, log in [(axes[0], "photo_share", "mask area, share of the photo (log)", True),
                            (axes[1], "fill", "fill ratio = mask area / box area", False)]:
    data = [animals.loc[animals["class"] == c, col].clip(lower=1e-5).to_numpy() for c in CLASSES]
    parts = ax.boxplot(data, tick_labels=CLASSES, patch_artist=True, showfliers=not log, widths=0.6,
                       flierprops={"markersize": 3, "markeredgecolor": INK})
    for patch, c in zip(parts["boxes"], CLASSES, strict=True):
        patch.set_facecolor(CLASS_COLORS[c])
        patch.set_alpha(0.75)
    for med in parts["medians"]:
        med.set_color(INK)
    if log:
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0, decimals=2))
    else:
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_ylabel(label)
    for lab in ax.get_xticklabels():
        lab.set_color(CLASS_COLORS[lab.get_text()])
axes[0].set_title("How big is a mask?")
axes[1].set_title("How much of its box does it fill?")
fig.tight_layout()
display(fig)
plt.close(fig)

N_EXTREME_MASKS = 2 if SMOKE else 3  # photos with the lowest / highest fill ratio to show (N-014)
_judged = animals[animals["mask_area"] >= MIN_JUDGED_AREA].sort_values(["fill", "id"])
_ends = pd.concat([_judged.drop_duplicates("image_id").head(N_EXTREME_MASKS),
                   _judged.drop_duplicates("image_id", keep="last").tail(N_EXTREME_MASKS)])
_fill_of = dict(zip(_ends["image_id"], _ends["fill"], strict=True))
show_masks(clean_images.set_index("id").loc[list(_ends["image_id"])].reset_index(), masks,
           f"the {N_EXTREME_MASKS} lowest and the {N_EXTREME_MASKS} highest fill ratios",
           caption=lambda r: f"{r['class']} · fill {_fill_of[r['id']]:.0%}", cols=2 * N_EXTREME_MASKS)

💡 Fill ratios sit in a narrow band, roughly 40–60 % for every class: animals are not rectangles. That
band is what makes the fill ratio a quality check. A mask that fills 98 % of its box is a rectangle, not an
animal, and one that fills 5 % has lost most of the animal.

**Worked answer (Q27).** About half: on the full set the median fill is near 45 % for bird and horse
(wings, legs, necks) and near 55 % for cat and sheep (compact shapes). The low end of the gallery shows
what drives it down (spread wings, a diagonal pose); the high end shows close-ups cut by the photo's edge.
The size plot repeats section 8 in pixels: birds are a sliver of the photo, cats and dogs a large part.

### Where the animal pixels are: one heatmap per class

Section 8 drew where the box *centres* are. A mask says more: which pixels of the frame are animal. The
next cell shrinks every photo's merged mask to the same small square and averages them per class, so a
dark spot means "in many of this class's photos, this spot is animal".

In [ ]:
# Mean mask per class on a common square: the share of the class's photos in which each spot is animal.
HEAT_RES = 24 if SMOKE else 48  # coarser grid with few photos (N-014)
_class_of = clean_images.set_index("id")["class"]
heat = {c: np.zeros((HEAT_RES, HEAT_RES)) for c in CLASSES}
for image_id, rle in photo_rle.items():
    small = cv2.resize(decode(rle).astype(np.float32), (HEAT_RES, HEAT_RES), interpolation=cv2.INTER_AREA)
    heat[_class_of[image_id]] += small

fig, axes = plt.subplots(1, len(CLASSES), figsize=(2.3 * len(CLASSES), 2.8))
for ax, c in zip(axes, CLASSES, strict=True):
    n_photos = max(int((_class_of == c).sum()), 1)
    cmap = mpl.colors.LinearSegmentedColormap.from_list(c, ["white", CLASS_COLORS[c]])
    ax.imshow(heat[c] / n_photos, cmap=cmap, vmin=0, vmax=1, extent=[0, 1, 1, 0])
    ax.set_title(f"{c} · {heat[c].sum() / n_photos / HEAT_RES**2:.0%} of a photo", fontsize=9,
                 color=CLASS_COLORS[c])
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.grid(False)
fig.suptitle("Where the animal pixels are (same scale for every class: white = never, full colour = always)",
             color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every class is darkest in the centre, but by very different amounts: cat and dog spread a large blob
over the middle and lower half (close-ups), while bird stays pale everywhere because its pixels are few and
scattered. A model trained on this learns that the edges of the frame are background, which the paddock
camera will contradict every day.

### Thin parts

A leg, a tail, a beak or a far-away bird is only a few pixels wide. When the photo is shrunk to a network's
input (say 224×224, about a third of a COCO photo), a part of 7 px becomes 2 px, and a mask resized with
nearest neighbour (section 3) can lose it altogether. The next cell marks, in every mask, the pixels in
parts narrower than `THIN_PX` = 7 px.

**Q28.** 🟡 Which class has the most masks with thin parts, and is it for the same reason as the
runner-up? Pick one, then run.

In [ ]:
# Q28  Share of each class's masks with at least 5 % of their pixels in thin parts, and the thinnest masks.
THIN_SHARE = 0.05  # a mask "has thin parts" when at least this share of its pixels is thin
_big = animals["mask_area"] >= MIN_JUDGED_AREA
_has_thin = animals["thin"] >= THIN_SHARE
thin_by_class = pd.DataFrame({
    "all masks": _has_thin.groupby(animals["class"]).mean(),
    f"masks ≥ {int(MIN_JUDGED_AREA**0.5)}² px": _has_thin[_big].groupby(animals.loc[_big, "class"]).mean(),
}).reindex(CLASSES).fillna(0)
RESULTS["thin_mask_share"] = {c: float(thin_by_class.loc[c, "all masks"]) for c in CLASSES}
display((thin_by_class * 100).round(0).astype(int).astype(str).add(" %").T)

fig, ax = plt.subplots(figsize=(8, 3.2))
_x = np.arange(len(CLASSES))
for k, (col, alpha) in enumerate([(thin_by_class.columns[0], 1.0), (thin_by_class.columns[1], 0.45)]):
    ax.bar(_x + (k - 0.5) * 0.38, thin_by_class[col], width=0.38, alpha=alpha,
           color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="white", label=col)
ax.set_xticks(_x, CLASSES)
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title(f"Masks with ≥ {THIN_SHARE:.0%} of their pixels thinner than {THIN_PX} px: all (solid), "
             "large enough to judge (pale)", fontsize=10)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

N_THIN_SHOW = 2 if SMOKE else 4  # photos with the most thin pixels to show (N-014)
animals["thin_px"] = animals["thin"] * animals["mask_area"]
_most_thin = animals.sort_values(["thin_px", "id"], ascending=[False, True]).drop_duplicates("image_id").head(
    N_THIN_SHOW)
show_masks(clean_images.set_index("id").loc[list(_most_thin["image_id"])].reset_index(), masks,
           f"the photos with the most thin pixels (dark = narrower than {THIN_PX} px)", mark_thin=True,
           cols=N_THIN_SHOW)

💡 Thin pixels are a tiny share of all pixels, so a pixel-counting loss barely notices when it gets them
wrong, and IoU barely moves. Yet they are what makes an outline an outline: the legs tell a horse from a
rock. Look at the gallery: the dark pixels are legs, tails and ears.

**Worked answer (Q28).** Bird, by far, on all masks (about two thirds on the full set), but not because
birds have long legs: most birds are so small that the *whole bird* is a thin part. Among the masks large
enough to judge, horse leads (legs), with bird, sheep and zebra next, and cat almost never has one. Two
different problems hide behind one number: tiny objects and thin parts of big ones.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · What handles each of these later (next week and beyond)</b><br>
The same vocabulary as section 8, now for pixels:
<ul>
<li><b>Pixel imbalance</b> (background ~85 %, bird ~1 %) → a <b>class-weighted</b> cross-entropy or a
<b>Dice loss</b>, which scores each class's overlap on its own scale, and reporting <b>mean IoU over
classes</b> (section 7), so that birds count as much as background.</li>
<li><b>Tiny animals and thin parts</b> → a higher input resolution, or <b>crops / tiles</b> of the photo,
so that a 7 px leg is still 7 px for the network; boundary-aware losses weigh the outline more.</li>
<li><b>Everything in the centre</b> → random crops and shifts when training, so the edge of the frame is
not always background.</li>
</ul>
None of these fixes a mask that is wrong. That is the data side, and the next part is about it.
</font></td></tr></table>

### Guided hunt: five ugly masks

The park paid an annotation team per mask, and the masks came back. In the Play Dataset they are `masks`;
the batch the team sent back is `park_masks`, built by the collapsed cell: `masks` with **five masks
spoiled at runtime**, one of each kind an annotation team really delivers:
<ul>
<li><b>swollen</b>: the outline drawn generously around the animal (the mask grown by a few pixels);</li>
<li><b>shifted</b>: the right outline, about 15 px off to the side;</li>
<li><b>a chunk missing</b>: a piece of the body cut out;</li>
<li><b>another animal's mask</b>: the outline of a different animal pasted onto this one;</li>
<li><b>a crude polygon</b>: an annotator in a hurry clicked five or six points around the animal.</li>
</ul>
Which masks, of which photos, is decided at runtime from `SEED`. They are hidden among hundreds of real
masks, and you will not look at hundreds of photos.

In [ ]:
# @title Plant the hunt (five ugly masks, seeded) { display-mode: "form" }
HUNT9_KINDS = ["another animal", "chunk missing", "shifted", "swollen", "crude polygon"]
SHIFT_PX = 15


def _paste(mask: np.ndarray, height: int, width: int) -> np.ndarray:
    """`mask` placed at the same coordinates on an empty canvas of another photo's size."""
    out = np.zeros((height, width), np.uint8)
    h, w = min(height, mask.shape[0]), min(width, mask.shape[1])
    out[:h, :w] = mask[:h, :w]
    return out


def _crude_polygon(mask: np.ndarray, max_vertices: int = 6) -> list[float]:
    """A polygon of at most `max_vertices` points around the mask (its convex hull, simplified)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(np.vstack(contours))
    eps = 0.02 * cv2.arcLength(hull, True)
    poly = cv2.approxPolyDP(hull, eps, True)
    while len(poly) > max_vertices:
        eps *= 1.5
        poly = cv2.approxPolyDP(hull, eps, True)
    return [float(v) for v in poly.reshape(-1)]


def plant_hunt9(animals_df: pd.DataFrame, table: pd.DataFrame,
                seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (park_masks, key): the masks of `animals_df` with one ugly mask of each of HUNT9_KINDS.

    Each kind spoils a different photo's mask, picked in a seeded order among masks large enough to judge,
    and the mask is changed through pycocotools (decode → OpenCV → encode). With no mask fitting a kind's
    condition (a small subset), the first free one is used, so five masks are always planted.
    """
    rng = random.Random(seed)
    size = table.set_index("id")[["width", "height"]]
    pool = animals_df[animals_df["mask_area"] >= MIN_JUDGED_AREA].set_index("id")
    order = sorted(pool.index)
    rng.shuffle(order)
    used, new_rle, key = set(), {}, []

    def fits(kind: str, ann_id: int):
        """The spoiled mask as a 0/1 array (or an RLE), plus a note, or None when this mask does not suit."""
        a = pool.loc[ann_id]
        x, y, w, h = a["bbox"]
        W, H = int(size.at[a["image_id"], "width"]), int(size.at[a["image_id"], "height"])
        if kind == "another animal":  # prefer a neighbour in the same photo: a duplicated mask, as in COCO
            others = pool[pool["image_id"] != a["image_id"]].index
            same = pool[(pool["image_id"] == a["image_id"]) & (pool.index != ann_id)].index
            for b in [*sorted(same), *[i for i in order if i in others]]:
                donor = _paste(decode(pool.at[b, "rle"]), H, W)
                if donor.any() and box_iou_dice(cv2.boundingRect(donor), tuple(a["bbox"]))[0] < 0.3:
                    where = "next to it" if b in same else "from another photo"
                    return donor, f"the mask of a {pool.at[b, 'class']} {where}"
            return None
        mask = decode(a["rle"])
        if kind == "chunk missing":
            dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
            r = float(dist.max())
            if r < max(15, 0.15 * a["mask_area"] ** 0.5):
                return None
            cy, cx = np.unravel_index(int(dist.argmax()), dist.shape)
            out = mask.copy()
            cv2.circle(out, (int(cx), int(cy)), int(0.8 * r), 0, -1)
            return out, f"a disc of radius {int(0.8 * r)} px cut out of the body"
        if kind == "shifted":
            if x + w + SHIFT_PX > W or y + h + SHIFT_PX > H or max(w, h) > 300:
                return None
            move = np.float32([[1, 0, SHIFT_PX], [0, 1, SHIFT_PX]])
            moved = cv2.warpAffine(mask, move, (W, H), flags=cv2.INTER_NEAREST)
            return moved, f"moved {SHIFT_PX} px right and down"
        if kind == "swollen":
            d = max(6, round(0.08 * (w * h) ** 0.5))
            if x < d or y < d or x + w + d > W or y + h + d > H:
                return None
            disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * d + 1, 2 * d + 1))
            return cv2.dilate(mask, disc), f"grown by {d} px all round"
        if kind == "crude polygon":
            if a["solidity"] > 0.85:  # pick an animal with clear dents, so the shortcut shows
                return None
            poly = _crude_polygon(mask)
            return to_rle([poly], H, W), f"{len(poly) // 2} clicked points"
        raise ValueError(kind)

    for kind in HUNT9_KINDS:
        free = [i for i in order if pool.at[i, "image_id"] not in used and i not in new_rle]
        spoiled = next(((i, out) for i in free if (out := fits(kind, i)) is not None), None)
        if spoiled is None:  # nothing fits the condition: give the first free mask another photo's mask
            i, j = free[0], free[1]
            height, width = decode(pool.at[i, "rle"]).shape
            pasted = _paste(decode(pool.at[j, "rle"]), height, width)
            spoiled = (i, (pasted, "another photo's mask (fallback)"))
        ann_id, (out, note) = spoiled
        new_rle[ann_id] = out if isinstance(out, dict) else encode(out)
        used.add(pool.at[ann_id, "image_id"])
        key.append({"kind": kind, "ann_id": int(ann_id), "image_id": int(pool.at[ann_id, "image_id"]),
                    "class": pool.at[ann_id, "class"], "how": note})
    park = animals_df[["id", "image_id", "class", "bbox", "iscrowd"]].copy()
    park["rle"] = [new_rle.get(i, r) for i, r in zip(animals_df["id"], animals_df["rle"], strict=True)]
    park["mask_area"] = [int(mask_utils.area(r)) for r in park["rle"]]
    return park, pd.DataFrame(key)


park_masks, HUNT9_KEY = plant_hunt9(animals, clean_images, SEED)
print(f"park_masks: {len(park_masks)} animal masks on {park_masks['image_id'].nunique()} photos. "
      "Five of them are ugly. Find them.")

You have four checks, all computed from the mask and its annotated box, no photo needed:
<ul>
<li><b>mask–box agreement</b>: the IoU (section 7) between the box drawn around the mask and the box the
annotator drew. On real COCO masks it is almost always above 0.95, because the box was made from the
mask;</li>
<li><b>fill ratio</b>: mask area / box area, from Q27;</li>
<li><b>pieces and holes</b>: an animal is usually one piece (two or three when something stands in front
of it) and almost never has a hole;</li>
<li><b>solidity</b>: mask area / area of a rubber band stretched around it. Animals have dents (between
the legs, under the neck), so it is rarely close to 1.</li>
</ul>
Only masks of at least 32×32 px are judged: below that, a few pixels of rounding change every number.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Set the thresholds</b><br>
Each threshold decides when a mask becomes a suspect. Run the hunt with the defaults first. Then make one
check stricter (for example <code>IOU_MIN = 0.97</code> or <code>SOLIDITY_MAX = 0.9</code>) and one looser,
rerun this cell and the next, and watch how many <i>real</i> masks join the suspects.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
IOU_MIN = 0.9  # suspect when the mask's own box overlaps the annotated box less than this (IoU, 0–1)
FILL_RANGE = (0.15, 0.95)  # suspect when mask area / box area is outside this range
MAX_PIECES = 3  # suspect with more separate pieces than this
MAX_HOLES = 0  # suspect with more holes than this
SOLIDITY_MAX = 0.97  # suspect when the mask fills more than this share of its rubber band (no dents at all)

_ok = (isinstance(IOU_MIN, (int, float)) and 0 <= IOU_MIN <= 1
       and isinstance(FILL_RANGE, tuple) and len(FILL_RANGE) == 2 and 0 <= FILL_RANGE[0] < FILL_RANGE[1]
       and isinstance(MAX_PIECES, int) and MAX_PIECES >= 1 and isinstance(MAX_HOLES, int) and MAX_HOLES >= 0
       and isinstance(SOLIDITY_MAX, (int, float)) and 0 < SOLIDITY_MAX <= 1)
if not _ok:
    raise SystemExit("Thresholds: IOU_MIN and SOLIDITY_MAX are numbers between 0 and 1, FILL_RANGE is a pair "
                     "(low, high) with low < high, MAX_PIECES is a whole number ≥ 1 and MAX_HOLES one ≥ 0.")

**Q29.** 🟡 Before you run: which check do you expect to catch each of the five kinds? Some kinds may trip
more than one check, and one check may catch none of the planted masks. Write your guess as five pairs
(kind → check), then run the cell and compare.

In [ ]:
# Q29  Measure every judged mask of park_masks, flag the ones outside the thresholds, and score the hunt.
_judge = park_masks[park_masks["mask_area"] >= MIN_JUDGED_AREA].reset_index(drop=True)
park_shape = pd.concat([_judge, pd.DataFrame([mask_shape(decode(r), b)
                                              for r, b in zip(_judge["rle"], _judge["bbox"], strict=True)])],
                       axis=1)
CHECKS = {
    "box IoU": park_shape["box_iou"] < IOU_MIN,
    "fill": (park_shape["fill"] < FILL_RANGE[0]) | (park_shape["fill"] > FILL_RANGE[1]),
    "pieces": park_shape["pieces"] > MAX_PIECES,
    "holes": park_shape["holes"] > MAX_HOLES,
    "solidity": park_shape["solidity"] > SOLIDITY_MAX,
}
park_shape["checks"] = [", ".join(name for name, hit in CHECKS.items() if hit.iloc[i])
                        for i in range(len(park_shape))]
mask_suspects = park_shape[park_shape["checks"] != ""].copy()
mask_suspects["n_checks"] = mask_suspects["checks"].str.count(",") + 1
mask_suspects = mask_suspects.sort_values(["n_checks", "box_iou", "id"], ascending=[False, True, True])

print(f"{len(_judge)} masks judged · {len(mask_suspects)} suspects "
      f"({len(mask_suspects) / max(len(_judge), 1):.1%}). Suspects per check:")
for name, hit in CHECKS.items():
    print(f"  {name:9s} {int(hit.sum())}")
_cols = ["id", "image_id", "class", "checks", "box_iou", "fill", "pieces", "holes", "solidity"]
display(mask_suspects[_cols].head(12).round(2).set_index("id"))

N_SUSPECTS_SHOW = 4 if SMOKE else 8  # suspect photos to show (N-014)
_show = mask_suspects.drop_duplicates("image_id").head(N_SUSPECTS_SHOW)
_checks_of = dict(zip(_show["image_id"], _show["checks"], strict=True))
show_masks(clean_images.set_index("id").loc[list(_show["image_id"])].reset_index(), park_masks,
           f"the {len(_show)} strongest suspects (as the team sent them)",
           caption=lambda r: f"{r['class']} · {_checks_of[r['id']]}")

Now the answer key: the five planted masks, what the cell above said about each, and each one next to the
real COCO mask it replaced.

In [ ]:
# The key: each planted mask, the checks that fired on it, and the original mask (top) vs the planted one.
_checks_by_id = park_shape.set_index("id")["checks"]
HUNT9_KEY["checks"] = HUNT9_KEY["ann_id"].map(_checks_by_id).fillna("")
HUNT9_KEY["found"] = HUNT9_KEY["checks"] != ""
hunt9_found = int(HUNT9_KEY["found"].sum())
RESULTS["hunt9"] = {"planted": len(HUNT9_KEY), "found": hunt9_found}
RESULTS["mask_suspects_real"] = int((~mask_suspects["id"].isin(HUNT9_KEY["ann_id"])).sum())
for _, k in HUNT9_KEY.iterrows():
    print(f"{'✅' if k['found'] else '❌'} {k['kind']:15s} photo {k['image_id']} ({k['class']}, {k['how']}) "
          f"→ {k['checks'] or 'no check fired'}")
print(f"found {hunt9_found} of {len(HUNT9_KEY)} planted · plus {RESULTS['mask_suspects_real']} real COCO "
      "masks among the suspects")

_by_image = clean_images.set_index("id")
fig, axes = plt.subplots(2, len(HUNT9_KEY), figsize=(3.2 * len(HUNT9_KEY), 6.2), squeeze=False)
for col, (_, k) in enumerate(HUNT9_KEY.iterrows()):
    photo = _by_image.loc[k["image_id"]]
    thumb = load_thumb(photo["path"], 320)
    scale = thumb.width / photo["width"]
    color = mpl.colors.to_rgb(CLASS_COLORS[k["class"]])
    for row, source in enumerate([masks, park_masks]):
        ax = axes[row, col]
        full = decode(source.loc[source["id"] == k["ann_id"], "rle"].iloc[0])
        small = cv2.resize(full, thumb.size, interpolation=cv2.INTER_NEAREST) > 0
        overlay = np.zeros((thumb.height, thumb.width, 4))
        overlay[small] = (*color, 0.6)
        ax.imshow(thumb)
        ax.imshow(overlay)
        x, y, w, h = source.loc[source["id"] == k["ann_id"], "bbox"].iloc[0]
        ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False, lw=1.5,
                                           ls="--", edgecolor=CLASS_COLORS[k["class"]]))
        frame(ax, CLASS_COLORS[k["class"]] if row == 0 else INK)
        ax.set_title(f"{k['class']} · original" if row == 0 else f"{k['kind']}\n{k['checks'] or '—'}",
                     fontsize=9)
fig.suptitle("the five planted masks (top: the real COCO mask, bottom: what the team sent)", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 No check catches everything, and together they catch all five, with real masks on top (on the full
set a few dozen, about 5 % of the masks judged: a list a person can look through in ten minutes).
The box check does most of the work because COCO built its boxes *from* the masks: any mask that moved,
grew or belongs elsewhere stops matching its box. The two others need a shape check: a chunk cut out of
the middle leaves the box intact but makes a hole, and a crude polygon fills its rubber band completely.

**Worked answer (Q29).** Expect: *another animal* → box IoU (the pasted mask sits around a different
animal); *shifted* → box IoU (15 px of a 100–300 px box is a large share); *swollen* → box IoU, often fill
too (a swollen mask can fill more than its box); *chunk missing* → holes; *crude polygon* → solidity (a
polygon of a few points has no dents). Pieces catches none of the planted five, but look at the real masks
it flags: animals broken up by a fence or a branch, sometimes correctly. Read the `checks` column of your
own run against this list: on a different seed, a kind may trip one check more or less.

The real suspects are worth a look too. COCO is not clean: a 2024 re-annotation study found that about
**2.4 % of COCO's val masks are poor**, with duplicated mask pairs among them (Singh et al., *COCO-ReM*,
ECCV 2024), and the masks with the highest solidity in the Play Dataset are polygons of 5 to 14 clicked
points, real crude polygons. Some of your real suspects are genuine annotation errors, not false alarms.

### 🔴 Optional: does the outline follow the photo's edges?

Every check so far ignored the photo. A good outline runs along the animal's edge, where the colour
changes; a sloppy one cuts across fur and grass. The next cell finds the photo's edges (Canny, the classic
edge detector) and measures, for each mask, the share of its outline that lies within 2 px of an edge. It
does this for the five planted masks and a sample of real ones.

**Q30.** 🔴 Which of the five kinds would this check catch, and which one is it blind to? Think about what
the outline of each one runs along, then run.

In [ ]:
# Q30  Edge alignment: share of each mask's outline within 2 px of a Canny edge in the photo.
N_EDGE_REAL = 10 if SMOKE else 150  # real masks to compare with (N-014)


def edge_alignment(path, mask: np.ndarray, band: int = 2) -> float:
    """Share of the mask's outline pixels that lie within `band` px of an edge of the photo."""
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None or gray.shape != mask.shape:
        return float("nan")
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    near = cv2.dilate(edges, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * band + 1, 2 * band + 1))) > 0
    outline = (mask - cv2.erode(mask, np.ones((3, 3), np.uint8))) > 0
    return float(near[outline].mean()) if outline.any() else float("nan")


_real_pool = park_shape[~park_shape["id"].isin(HUNT9_KEY["ann_id"])]
_real = _real_pool.sample(min(N_EDGE_REAL, len(_real_pool)), random_state=SEED)
_planted = park_shape[park_shape["id"].isin(HUNT9_KEY["ann_id"])]
_edge_rows = pd.concat([_real, _planted])
_edge_rows["alignment"] = [edge_alignment(_by_image.at[i, "path"], decode(r))
                           for i, r in zip(_edge_rows["image_id"], _edge_rows["rle"], strict=True)]
_real_align = _edge_rows.loc[_real.index, "alignment"]
_kind_of = dict(zip(HUNT9_KEY["ann_id"], HUNT9_KEY["kind"], strict=True))
_planted_align = _edge_rows[_edge_rows["id"].isin(_kind_of)]
RESULTS["edge_alignment"] = {_kind_of[i]: float(a) for i, a in zip(_planted_align["id"],
                                                                  _planted_align["alignment"], strict=True)}
print(f"real masks: median alignment {_real_align.median():.0%} (middle half "
      f"{_real_align.quantile(0.25):.0%}–{_real_align.quantile(0.75):.0%}, n = {_real_align.notna().sum()})")
for kind, a in RESULTS["edge_alignment"].items():
    print(f"  {kind:15s} {a:.0%}  (lower than {(_real_align > a).mean():.0%} of the real masks)")

fig, ax = plt.subplots(figsize=(10, 3.4))
_y = {c: i for i, c in enumerate(CLASSES)}
_jitter = np.random.default_rng(SEED).uniform(-0.25, 0.25, len(_real))
ax.scatter(_edge_rows.loc[_real.index, "alignment"], _real["class"].map(_y) + _jitter, s=14, alpha=0.6,
           c=[CLASS_COLORS[c] for c in _real["class"]], label="real masks")
for _, p in _planted_align.iterrows():
    ax.scatter(p["alignment"], _y[p["class"]], s=160, marker="*", color=CLASS_COLORS[p["class"]],
               edgecolor=INK, linewidth=1.2, zorder=3)
    ax.annotate(_kind_of[p["id"]], (p["alignment"], _y[p["class"]]), textcoords="offset points",
                xytext=(6, 6), fontsize=8, color=INK)
ax.set_yticks(range(len(CLASSES)), CLASSES)
for lab in ax.get_yticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_xlabel("share of the outline within 2 px of a photo edge")
ax.set_title("Does the outline follow the photo's edges? dots: real masks · stars: the planted five")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Real outlines are spread widely on this scale and rarely reach 100 %: a smooth animal on a smooth
background has stretches without a strong edge, and a busy one has edges everywhere.

**Worked answer (Q30).** Think of where each outline runs. *Shifted*, *swollen* and *crude polygon* run a
few pixels off the animal's edge, through grass or fur, so they tend to sit low. *Chunk missing* keeps its
real outline but adds the rim of the hole, which runs through the middle of the body, so it drops by as
much as the hole is large. *Another animal* is the blind spot: when the pasted mask is a neighbour's from
the same photo, it is a real, careful outline that follows real edges, just around the wrong animal, and
it sits among the real masks, and only the box check sees it. When it comes from another photo, its
outline lands on whatever happens to be there, and it sits low. If a planted mask lands among the real
ones on your run, look at its photo: a busy background (grass, water, a fence) has edges everywhere, so
any line lies close to one. That is why this check ranks masks for a human to look at rather than
judging them.

## 10. Data-centric AI in practice: the Challenge

**Objective:** find the problems planted in a new batch of photos from the park, using only the tools of
sections 3 to 9, and see how many you catch.

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · Data-centric AI</b><br>
There are two ways to make a model better. <b>Model-centric</b> work keeps the data fixed and changes the
model: a bigger network, another loss, more epochs. <b>Data-centric</b> work keeps the model fixed and
changes the data: remove the copies, fix the wrong labels, drop the broken files, make the boxes and masks
consistent. Andrew Ng gave the second approach its name in 2021, and the reason it matters is simple: the
data is where the cheap gains are. Northcutt, Athalye and Mueller (2021) found label errors in the
<i>test sets</i> of ten widely used benchmarks, on average at least 3.3 % of the labels (at least 6 % of
ImageNet's validation set), enough to change which model looks best.<br>
Everything you did in sections 5 to 9 was data-centric work. This section puts it together on one batch,
without telling you where the problems are.
</font></td></tr></table>

### A new batch from the park

The park's cameras have delivered a new batch: about 150 photos, a label per photo typed in by the camera
operator, a train/test split made by the intern, the keeper's headcount per photo, and boxes and masks
from the annotation team. Somebody has to check it before it goes into next week's training. The next cell
builds that batch into `data/cache/challenge/` with neutral file names. (The park has no real cameras: the
photos come from the Play Dataset, picked among those that passed every check of sections 5 to 9, so
**every problem in the batch was planted at runtime**, from `SEED`.)

The planted problems are of ten kinds:
- **exact duplicate**: the same file twice;
- **flipped copy** and **rotated copy**: a photo mirrored or turned by 90° and saved again;
- **recompressed copy**: a photo resized and saved at a lower JPEG quality;
- **train/test overlap**: a copy whose two files sit in *different* splits (whatever kind of copy it is);
- **unreadable**: a file that does not open;
- **grayscale**: a photo saved without colour;
- **wrong label**: the operator typed the wrong class;
- **missing box**: an animal the keeper counted but the annotation team did not annotate;
- **ugly mask**: a mask that does not match its animal (section 9).

In [ ]:
# @title Plant the new batch (seeded, at runtime; nothing is redistributed) { display-mode: "form" }
CHALLENGE_DIR = DATA_DIR / "cache" / "challenge"
N_BATCH_ORIGINALS = 24 if SMOKE else 140  # distinct photos in the batch (N-014: fewer in smoke mode)
TEST_SHARE = 0.2
RECOMPRESS_SCALE = 0.7
CHALLENGE_TYPES = ["exact duplicate", "flipped copy", "rotated copy", "recompressed copy",
                   "train/test overlap", "unreadable", "grayscale", "wrong label", "missing box", "ugly mask"]
COPY_KIND = {"exact duplicate": "exact", "flipped copy": "flipped", "rotated copy": "rotated",
             "recompressed copy": "recompressed", "train/test overlap": "recompressed"}
N_COPIES = {"exact duplicate": 1 if SMOKE else 2, "flipped copy": 1, "rotated copy": 1,
            "recompressed copy": 1, "train/test overlap": 1}
N_WRONG_LABELS = 2 if SMOKE else 3
UGLY_MASK_KINDS = ["shifted", "swollen"]  # two of section 9's five kinds


def challenge_pool() -> pd.DataFrame:
    """Rows of clean_images that pass every check of sections 3–9 at the default settings: no flag, colourful,
    no crowd region, headcount = boxes, and every judged mask agrees with its box and has a plain shape."""
    judged = animals[animals["mask_area"] >= MIN_JUDGED_AREA]
    bad_mask = judged[(judged["box_iou"] < 0.93) | (judged["fill"] < 0.15) | (judged["fill"] > 0.95)
                      | (judged["pieces"] > 3) | (judged["holes"] > 0) | (judged["solidity"] > 0.97)]
    crowd = masks.loc[masks["iscrowd"] == 1, "image_id"]
    gray = image_stats.loc[image_stats["looks_gray"], "id"]
    n_boxes = boxes.groupby("image_id").size()
    excluded = set(bad_mask["image_id"]) | set(crowd) | set(gray)
    ok = (clean_images["flags"] == "") & ~clean_images["id"].isin(excluded)
    ok &= clean_images["n_animals"] == clean_images["id"].map(n_boxes)
    return clean_images[ok]


def _jpeg(image: Image.Image, quality: int) -> bytes:
    buf = BytesIO()
    image.save(buf, "JPEG", quality=quality)
    return buf.getvalue()


def _orientation_gap(path, transpose) -> int:
    """Bits between the pHash of a photo and of its flipped or rotated version (small if symmetric)."""
    thumb = load_thumb(path, THUMB_FOR_HASH)
    return int(phash(thumb) - phash(thumb.transpose(transpose)))


def _move_annotation(rle: dict, bbox, kind: str, width: int, height: int,
                     new_size: tuple) -> tuple[dict, list]:
    """An annotation (RLE mask + box) carried over to an exact, flipped, rotated or resized copy."""
    x, y, w, h = bbox
    if kind == "exact":
        return rle, [x, y, w, h]
    mask = decode(rle)
    if kind == "flipped":
        return encode(mask[:, ::-1]), [width - x - w, y, w, h]
    if kind == "rotated":  # 90° counter-clockwise, like PIL's ROTATE_90
        return encode(np.rot90(mask)), [y, width - x - w, h, w]
    sx, sy = new_size[0] / width, new_size[1] / height  # resized: x and y scale separately (section 3)
    resized = cv2.resize(mask, new_size, interpolation=cv2.INTER_NEAREST)
    return encode(resized), [x * sx, y * sy, w * sx, h * sy]


def plant_challenge(dest: Path, n_originals: int, seed: int):
    """Write the batch: n_originals photos of challenge_pool() plus planted copies and problems.

    Returns (files, batch, batch_animals, key). files: {name: path}. batch: one row per file (file, class,
    split, n_animals, width, height, path). batch_animals: one row per annotated animal (id, file, class,
    bbox, rle, mask_area). key: one dict per planted problem (type, files, class, detail); for a copy,
    `files` holds the copy and its source. Every problem gets its own photo. Deterministic for a given seed.
    """
    rng = random.Random(seed)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    pool = challenge_pool()
    chosen = pool.sample(n=min(n_originals, len(pool)), random_state=seed).set_index("id").sort_index()
    ids = list(chosen.index)
    shuffled = ids[:]
    rng.shuffle(shuffled)
    n_test = max(2, round(TEST_SHARE * len(ids)))
    split = {i: "test" if k < n_test else "train" for k, i in enumerate(shuffled)}
    label = dict(chosen["class"])
    anns = animals[animals["image_id"].isin(ids)].set_index("id")
    order = ids[:]
    rng.shuffle(order)
    used: set = set()

    def take(ok=lambda i: True) -> int:
        """The next unused photo in the seeded order that satisfies ok (the next unused one if none does)."""
        free = [i for i in order if i not in used]
        i = next((i for i in free if ok(i)), free[0])
        used.add(i)
        return i

    def resize_safe(i) -> bool:  # masks that stay judged after resizing keep agreeing with their box
        a = anns[anns["image_id"] == i]
        return bool((a.loc[a["mask_area"] * RECOMPRESS_SCALE**2 >= MIN_JUDGED_AREA, "box_iou"] >= 0.95).all())

    in_place = []  # (type, photo id, detail)
    # A missing box: one animal of a photo with several loses its annotation.
    i = take(lambda i: chosen.at[i, "n_animals"] >= 2)
    dropped = rng.choice(sorted(anns.index[anns["image_id"] == i]))
    in_place.append(("missing box", i, f"the keeper counted {chosen.at[i, 'n_animals']}, "
                                       f"the team annotated {chosen.at[i, 'n_animals'] - 1}"))
    # Ugly masks: section 9's planting on the photos still free, two of its five kinds kept.
    free_anns = anns[~anns["image_id"].isin(used)].reset_index()
    spoiled, spoil_key = plant_hunt9(free_anns, chosen.reset_index(), seed)
    spoiled_rle = spoiled.set_index("id")["rle"]
    new_rle = {}
    for _, k in spoil_key[spoil_key["kind"].isin(UGLY_MASK_KINDS)].iterrows():
        new_rle[k["ann_id"]] = spoiled_rle[k["ann_id"]]
        used.add(k["image_id"])
        in_place.append(("ugly mask", k["image_id"], f"{k['kind']} mask of a {k['class']} ({k['how']})"))
    for _ in range(N_WRONG_LABELS):
        i = take()
        label[i] = rng.choice([c for c in CLASSES if c != chosen.at[i, "class"]])
        in_place.append(("wrong label", i, f"labelled {label[i]}, the photo shows a {chosen.at[i, 'class']}"))
    gray_id, unreadable_id = take(), take()
    in_place += [("grayscale", gray_id, "saved without colour (mode L)"),
                 ("unreadable", unreadable_id, "the second half of the file is missing")]
    # Flipped and rotated copies come from photos that do not look the same after the turn (no symmetry).
    flip, turn = Image.Transpose.FLIP_LEFT_RIGHT, Image.Transpose.ROTATE_90
    condition = {"flipped copy": lambda i: _orientation_gap(chosen.at[i, "path"], flip) >= 20,
                 "rotated copy": lambda i: _orientation_gap(chosen.at[i, "path"], turn) >= 20,
                 "recompressed copy": resize_safe,
                 "train/test overlap": lambda i: split[i] == "train" and resize_safe(i)}
    copies = [(t, take(condition.get(t, lambda i: True))) for t, n in N_COPIES.items() for _ in range(n)]

    # Every file of the batch: the originals (two of them spoiled in place), then the copies.
    entries = []
    for i in ids:
        path, size = Path(chosen.at[i, "path"]), (int(chosen.at[i, "width"]), int(chosen.at[i, "height"]))
        data = path.read_bytes()
        if i == gray_id:
            with Image.open(path) as im:
                data = _jpeg(im.convert("L"), 92)
        elif i == unreadable_id:
            data = data[: len(data) // 2]
        entries.append({"data": data, "source": i, "kind": None, "split": split[i], "size": size})
    for t, i in copies:
        path, (w, h) = Path(chosen.at[i, "path"]), (int(chosen.at[i, "width"]), int(chosen.at[i, "height"]))
        kind = COPY_KIND[t]
        with Image.open(path) as im:
            rgb = im.convert("RGB")
        if kind == "exact":
            data, size = path.read_bytes(), (w, h)
        elif kind == "flipped":
            data, size = _jpeg(ImageOps.mirror(rgb), 92), (w, h)
        elif kind == "rotated":
            data, size = _jpeg(rgb.transpose(Image.Transpose.ROTATE_90), 92), (h, w)
        else:
            size = (max(8, round(w * RECOMPRESS_SCALE)), max(8, round(h * RECOMPRESS_SCALE)))
            data = _jpeg(rgb.resize(size), 55)
        other = "test" if split[i] == "train" else "train"
        entries.append({"data": data, "source": i, "kind": kind, "type": t, "size": size,
                        "split": other if t == "train/test overlap" else split[i]})
    slots = list(range(len(entries)))
    rng.shuffle(slots)
    rows, ann_rows = [], []
    for slot, e in zip(slots, entries, strict=True):
        e["name"] = f"park_{slot:03d}.jpg"
        (dest / e["name"]).write_bytes(e["data"])
        i = e["source"]
        rows.append({"file": e["name"], "class": label[i], "split": e["split"],
                     "n_animals": int(chosen.at[i, "n_animals"]), "width": e["size"][0],
                     "height": e["size"][1], "path": str(dest / e["name"])})
        for ann_id, a in anns[anns["image_id"] == i].iterrows():
            if ann_id == dropped:
                continue
            rle, bbox = _move_annotation(new_rle.get(ann_id, a["rle"]), a["bbox"], e["kind"] or "exact",
                                         int(chosen.at[i, "width"]), int(chosen.at[i, "height"]), e["size"])
            ann_rows.append({"file": e["name"], "class": a["class"],
                             "bbox": [round(float(v), 2) for v in bbox],
                             "rle": rle, "mask_area": int(mask_utils.area(rle))})

    name_of = {(e["source"], e["kind"]): e["name"] for e in entries}
    key = [{"type": t, "files": [name_of[(i, None)]], "class": chosen.at[i, "class"], "detail": d}
           for t, i, d in in_place]
    for t, i in copies:
        copy, source = name_of[(i, COPY_KIND[t])], name_of[(i, None)]
        where = next(e["split"] for e in entries if e["name"] == copy)
        key.append({"type": t, "files": [copy, source], "class": chosen.at[i, "class"],
                    "detail": f"{COPY_KIND[t]} copy of {source} ({split[i]}), the copy is in {where}"})
    key.sort(key=lambda k: (CHALLENGE_TYPES.index(k["type"]), k["files"][0]))
    table = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
    annotated = pd.DataFrame(ann_rows).sort_values("file").reset_index(drop=True)
    annotated.insert(0, "id", range(1, len(annotated) + 1))
    files = {r["file"]: Path(r["path"]) for _, r in table.iterrows()}
    return files, table, annotated, key


def show_batch(file_names, title: str, caption=None, cols: int = 5, annotations: bool = False) -> None:
    """Gallery of batch files, each framed in the colour of its label (the `class` column of `batch`).

    annotations=True paints the annotation team's masks in their class colour and draws their boxes dashed.
    A file that does not open shows as an empty frame. `caption(name) -> str` writes the title of each photo.
    """
    names = list(file_names)
    if not names:
        print(f"{title}: no files to show.")
        return
    by_file = batch.set_index("file")
    cols = min(cols, len(names))
    n_rows = -(-len(names) // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.9 * cols, 3.0 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, name in zip(axes.ravel(), names, strict=False):
        row = by_file.loc[name]
        ax.axis("on")
        if is_readable(row["path"]):
            thumb = load_thumb(row["path"], 256)
            ax.imshow(thumb)
            if annotations:
                scale = thumb.width / row["width"]
                overlay = np.zeros((thumb.height, thumb.width, 4))
                own = batch_animals[batch_animals["file"] == name]
                for _, a in own.iterrows():
                    small = cv2.resize(decode(a["rle"]), thumb.size, interpolation=cv2.INTER_NEAREST) > 0
                    overlay[small] = (*mpl.colors.to_rgb(CLASS_COLORS[a["class"]]), 0.55)
                ax.imshow(overlay)
                for _, a in own.iterrows():
                    x, y, w, h = a["bbox"]
                    ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale,
                                                       fill=False, lw=1.5, ls="--",
                                                       edgecolor=CLASS_COLORS[a["class"]]))
        else:
            ax.text(0.5, 0.5, "cannot be opened", ha="center", va="center", transform=ax.transAxes, color=INK)
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(name) if caption else f"{name}\n{row['class']} · {row['split']}", fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

The next cell writes the batch (a few seconds) and loads it into three objects: `batch_files` (file name →
path), `batch` (one row per file: the label as typed, the split, the keeper's headcount `n_animals`, the
size) and `batch_animals` (one row per annotated animal: its class, box and mask as RLE, as in section 9).

In [ ]:
# Write the batch, then show its tables and a first look at a dozen of its photos.
batch_files, batch, batch_animals, _CHALLENGE_KEY = plant_challenge(CHALLENGE_DIR, N_BATCH_ORIGINALS, SEED)
RESULTS["challenge_types"] = len({k["type"] for k in _CHALLENGE_KEY})
print(f"{len(batch_files)} files in {CHALLENGE_DIR}/ · {(batch['split'] == 'test').sum()} in test · "
      f"{len(batch_animals)} annotated animals")
print(f"planted: {len(_CHALLENGE_KEY)} problems of {RESULTS['challenge_types']} kinds")
display(batch.drop(columns="path").head())
show_batch(batch["file"].head(12), "the first dozen files of the batch (frame = the label as typed)", cols=6)

Nothing in the tables says "problem" yet: the files look like any other photos, and the labels, the splits
and the annotations look like any other rows. That is what a real delivery looks like.

### Your tools

Everything you need was built in the earlier sections:
- `exact_duplicate_groups(paths)` and `find_copies(files, threshold, use_variants=True)` (section 5): pairs
  of files that are copies, and what kind of copy;
- `is_readable(path)` (section 5): does the file open at all?
- `measure_image(path, thumb)[0]` (section 3): `mode` and `colourfulness` of a photo (0 for grayscale);
- `batch["n_animals"]` against the number of rows per file in `batch_animals` (section 8);
- `mask_shape(decode(rle), bbox)` (section 9): `box_iou`, `fill`, `pieces`, `holes`, `solidity`;
- `show_batch(names, title, annotations=True)`: look at any files, with their boxes and masks.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Plan before you hunt: a minute with the list above saves ten minutes of looking at photos one by one.
</font></td></tr></table>

**Q31.** 🟡 For each of the ten kinds, which tool would you reach for, and what does it read: the files,
the `batch` table or the `batch_animals` table? Which kinds cannot be seen in the pixels at all? Write
your plan as ten short lines.

**Q32.** 🟡 The hunt. In the next cell, write one check per kind (the first one is done for you, and the
comments name a tool for the others). Collect what you find in `findings` in the cell after it and run the
score. Rerun both as often as you like.

In [ ]:
# Q32  Your hunt: one check per kind. The first is written; the comments name a tool for each of the others.
cant_open = sorted(name for name, path in batch_files.items() if not is_readable(path))
print("unreadable:", cant_open)

# Copies (exact, flipped, rotated, recompressed) on the files that open:
#   opens = {n: p for n, p in batch_files.items() if n not in cant_open}
#   find_copies(opens, HASH_THRESHOLD, use_variants=True), then compare batch["split"] of each pair's files.
# Grayscale: measure_image(path, 64)[0]["mode"] and ["colourfulness"] for each file that opens.
# Wrong labels: batch["class"] vs the class of the file's animals in batch_animals.
# Missing boxes: batch["n_animals"] vs batch_animals.groupby("file").size().
# Ugly masks: mask_shape(decode(a["rle"]), a["bbox"])["box_iou"] for each row a of batch_animals.

In [ ]:
# @title Score helpers: check_findings, reveal_answers { display-mode: "form" }
def check_findings(findings, verbose: bool = True) -> dict:
    """Score a list of (file, kind) pairs against the planted problems: found k of n per kind.

    A planted problem is found when one pair names its kind and one of its files (for a copy, the copy or its
    source). Returns {"found", "planted", "per_type": {kind: [found, planted]}, "unmatched"}.
    """
    if not isinstance(findings, (list, tuple)):
        raise SystemExit("findings must be a list of (file, kind) pairs, "
                         "e.g. [('park_999.jpg', 'grayscale')]")
    pairs, skipped = set(), []
    for item in findings:
        if (isinstance(item, (list, tuple)) and len(item) == 2 and all(isinstance(v, str) for v in item)
                and item[1].strip().lower() in CHALLENGE_TYPES):
            pairs.add((Path(item[0].strip()).name, item[1].strip().lower()))
        else:
            skipped.append(item)
    hit = [any(kind == k["type"] and name in k["files"] for name, kind in pairs) for k in _CHALLENGE_KEY]
    matched = {(name, kind) for name, kind in pairs
               if any(kind == k["type"] and name in k["files"] for k in _CHALLENGE_KEY)}
    per_type = {t: [sum(h for h, k in zip(hit, _CHALLENGE_KEY, strict=True) if k["type"] == t),
                    sum(k["type"] == t for k in _CHALLENGE_KEY)] for t in CHALLENGE_TYPES}
    score = {"found": int(sum(hit)), "planted": len(_CHALLENGE_KEY), "per_type": per_type,
             "unmatched": len(pairs) - len(matched)}
    if verbose:
        for t, (k, n) in per_type.items():
            print(f"  {'✅' if k == n else '⬜'} {t:19s} found {k} of {n}")
        print(f"found {score['found']} of {score['planted']} planted problems · {score['unmatched']} of your "
              "findings match no planted problem")
        if skipped:
            print(f"⚠️ skipped {len(skipped)} entries that are not (file, kind) pairs with a kind from "
                  f"CHALLENGE_TYPES, e.g. {skipped[0]!r}")
    return score


def reveal_answers() -> None:
    """Print the answer key, one line per planted problem, and show the files with each problem."""
    for k in _CHALLENGE_KEY:
        print(f"{k['type']:19s} {' & '.join(k['files'])}: {k['detail']}")
    type_of = {k["files"][0]: k["type"] for k in _CHALLENGE_KEY}
    in_annotations = {"wrong label", "missing box", "ugly mask"}
    file_problems = [k["files"][0] for k in _CHALLENGE_KEY if k["type"] not in in_annotations]
    show_batch(file_problems, "problems in the files (copies, broken, grayscale)",
               caption=lambda n: f"{type_of[n]}\n{n}", cols=6)
    annotation_problems = [k["files"][0] for k in _CHALLENGE_KEY if k["type"] in in_annotations]
    show_batch(annotation_problems, "problems in the labels and annotations (frame = the label as typed)",
               caption=lambda n: f"{type_of[n]}\n{n} · labelled {batch.set_index('file').at[n, 'class']}",
               cols=6, annotations=True)

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Your findings</b><br>
Write one <code>(file, kind)</code> pair per problem into <code>findings</code>. The kind is one of the ten
names above, spelled as in <code>CHALLENGE_TYPES</code>. For a copy, name either of its two files. You may
build the list from your variables, e.g. <code>[(f, "unreadable") for f in cant_open] + [...]</code>.
<code>check_findings</code> says how many of each kind you found, and how many of your findings match no
planted problem; it never says which ones.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
# One (file, kind) pair per problem; kind is one of CHALLENGE_TYPES. For a copy, either of its files counts.
findings = [
    # ("park_999.jpg", "grayscale"),  (the format; there is no park_999.jpg)
]

challenge_score = check_findings(findings)

💡 Read the ⬜ lines as a to-do list: each one is a kind you have not caught yet, and the cell above names a
tool for it. A finding that matches nothing is either the wrong kind for the right file (a copy across the
splits is a *train/test overlap*, not a recompressed copy) or a false alarm.

Can every problem be found with this notebook's tools alone? The collapsed cell below runs a reference
solution, one check per kind with the tools listed above at their default settings, and scores it the same
way. It prints only the score; open it after your own hunt if you want to read the code.

In [ ]:
# @title Reference solution (run it now, read it after your hunt) { display-mode: "form" }
_REF_HASH_THRESHOLD = 8  # section 5's default
_REF_IOU_MIN = 0.9  # section 9's default
_TYPE_OF_COPY = {"exact": "exact duplicate", "flipped": "flipped copy", "rotated": "rotated copy",
                 "recompressed": "recompressed copy"}


def _reference_findings() -> list[tuple[str, str]]:
    """One check per kind, with the tools of sections 3–9; returns (file, kind) pairs."""
    found = []
    readable = {n: p for n, p in batch_files.items() if is_readable(p)}
    found += [(n, "unreadable") for n in batch_files if n not in readable]
    split_of = batch.set_index("file")["split"]
    for _, r in find_copies(readable, _REF_HASH_THRESHOLD, use_variants=True).iterrows():
        across = split_of[r["a"]] != split_of[r["b"]]
        found.append((r["b"], "train/test overlap" if across else _TYPE_OF_COPY[r["kind"]]))
    for n, p in readable.items():
        stats = measure_image(p, 64)[0]
        if stats["mode"] != "RGB" or stats["colourfulness"] < 2.0:
            found.append((n, "grayscale"))
    per_file = batch_animals.groupby("file")
    annotated_class = per_file["class"].agg(lambda s: s.mode().iloc[0])
    n_annotated = per_file.size()
    for _, r in batch.iterrows():
        if r["file"] in annotated_class.index and annotated_class[r["file"]] != r["class"]:
            found.append((r["file"], "wrong label"))
        if int(n_annotated.get(r["file"], 0)) != r["n_animals"]:
            found.append((r["file"], "missing box"))
    for _, a in batch_animals[batch_animals["mask_area"] >= MIN_JUDGED_AREA].iterrows():
        if mask_shape(decode(a["rle"]), a["bbox"])["box_iou"] < _REF_IOU_MIN:
            found.append((a["file"], "ugly mask"))
    return sorted(set(found))


_reference = _reference_findings()
_reference_score = check_findings(_reference, verbose=False)
RESULTS["challenge_reference"] = {"found": _reference_score["found"], "planted": _reference_score["planted"],
                                  "unmatched": _reference_score["unmatched"]}
print(f"reference solution: found {_reference_score['found']} of {_reference_score['planted']} planted "
      f"problems with {len(_reference)} findings ({_reference_score['unmatched']} false alarms)")

### The answer key

When you are done (or stuck), open the worked answer below, then run `reveal_answers()` for the files of
your batch.

<details><summary><b>Worked answer (Q31 and Q32): which check finds which kind</b></summary>

- **Unreadable**: `is_readable(path)` on every file. Run it first, because every other check that opens the
  photo stumbles on this file.
- **Exact, flipped, rotated and recompressed copies**: `find_copies(..., use_variants=True)` on the files
  that open. MD5 finds the exact ones; the perceptual hash of each photo's six variants finds the
  recompressed (variant "original"), flipped and rotated ones. Plain hashing would miss two of the four.
- **Train/test overlap**: the same pairs, then `batch["split"]` of both files. A pair across the splits is
  leakage whatever kind of copy it is: the test half then scores memory (section 6). In the planted batch
  it is a recompressed copy, so it hides among the near-duplicates.
- **Grayscale**: `measure_image` gives `mode` "L" and a `colourfulness` of 0 (section 3).
- **Wrong label**: no tool looks at the pixels and says "this is a dog". The batch has a second source,
  though: the annotation team's animals carry their own class, so a file whose label disagrees with the
  class of its animals is a wrong label (or a wrong annotation; a human decides).
- **Missing box**: the keeper's headcount `n_animals` vs the number of animals annotated per file, as in
  section 8.
- **Ugly masks**: `mask_shape(...)["box_iou"]` below 0.9 (section 9). The two planted kinds are a shifted
  and a swollen mask, and both stop matching their box.

The four kinds that cannot be seen in the pixels are train/test overlap, wrong label, missing box and
ugly mask: all four are disagreements between two sources (two splits, a label and its annotation, a
headcount and the boxes, a mask and its box). That is the data-centric habit: when you can, get a second
source and compare.
</details>

In [ ]:
# The key: every planted problem with its file(s), then the files themselves.
reveal_answers()

💡 Look at which problems you found by looking and which only by comparing: the copies and the grayscale
photo show in a gallery if you happen to see both files, but the wrong labels, the missing box and the
ugly masks need two sources side by side, and nobody spots them by scrolling through 150 photos.

### 🔴 Optional: the ruler on the dirty and the cleaned batch

Section 6's nearest-neighbour **ruler** comes back for one measurement. It is not training: a
1-nearest-neighbour classifier on colour histograms, no parameters, seconds on a CPU. It labels each test
photo of the batch with the label of its most similar training photo, three times: on the batch as
delivered, without the files in your `findings`, and without the files the reference solution found.

**Q33.** 🔴 Will the ruler score higher on the dirty batch or on the cleaned one? Think about what the
train/test overlap and the wrong labels do to the score. And which of the numbers would you put in a report?
Predict, then run.

In [ ]:
# Q33  1-NN ruler accuracy on the test split: as delivered, after your cleaning, after the reference's.
_ruler_x = {n: measure_image(p, RULER_THUMB)[1].ravel() for n, p in batch_files.items() if is_readable(p)}


def batch_ruler(drop) -> tuple[float, int]:
    """(1-NN accuracy on the test split, number of test photos), with the files in `drop` removed."""
    keep = batch[batch["file"].isin(_ruler_x) & ~batch["file"].isin(set(drop))]
    train, test = keep[keep["split"] == "train"], keep[keep["split"] == "test"]
    if len(train) == 0 or len(test) == 0:
        return float("nan"), len(test)
    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(np.stack([_ruler_x[f] for f in train["file"]]), train["class"])
    predicted = knn.predict(np.stack([_ruler_x[f] for f in test["file"]]))
    return float((predicted == test["class"].to_numpy()).mean()), len(test)


_yours = [Path(f[0]).name for f in findings
          if isinstance(f, (list, tuple)) and len(f) == 2 and isinstance(f[0], str)]
ruler_scores = {"as delivered": batch_ruler([]), "your cleaning": batch_ruler(_yours),
                "reference cleaning": batch_ruler([f for f, _ in _reference])}
RESULTS["challenge_ruler"] = {k: v[0] for k, v in ruler_scores.items()}
for name, (acc, n_test) in ruler_scores.items():
    print(f"{name:19s} ruler accuracy {acc:.1%} on {n_test} test photos")
if not _yours:
    print("(your findings list is empty, so 'your cleaning' is the batch as delivered)")

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(list(ruler_scores), [v[0] for v in ruler_scores.values()],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["horse"], CLASS_COLORS["zebra"]])
ax.axhline(1 / len(CLASSES), color=INK, linestyle="--", linewidth=1, label="chance")
ax.set_ylabel("1-NN ruler accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("The same ruler on three versions of the batch")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

💡 The ruler did not change between the bars; only the data did. Any difference you see is the data's
doing, which is the whole point of data-centric work.

**Worked answer (Q33).** Both directions are possible, and on a test half of about 30 photos one photo
moves the score by about 3 points. The train/test overlap gives the dirty batch a free point: its test copy
finds its own source in training. The wrong labels cost it points: the ruler is graded against a label
that is wrong. Cleaning removes both, so the score can go up or down. The number for a report is the
cleaned one, because it is the only one that measures what you think it measures; and the honest report
also says what was removed and why (section 5's "removed N, kept M").

**Takeaway.** A batch is checked with the same short list every time: does it open, is it a copy (also
across the splits), is it what its mode says, does the label agree with the annotation, does the headcount
agree with the boxes, does the mask agree with its box. Most of these are comparisons between two sources,
and each takes one line once the tools exist. Section 11 runs the list on your own dataset.

## 11. Your project dataset

**Objective:** run the checklist of sections 3 to 6 on your own folder-per-class dataset, and answer the
guiding questions for it the way we answered them for the park.

Everything so far ran on the park's photos. Your project has its own dataset, and it deserves the same
twenty minutes of looking before anybody trains on it. This section packs the classification checklist
into a few cells that read **any folder-per-class dataset**: one sub-folder per class, the images inside.

```
my_dataset/
├── cat/      img_001.jpg, img_002.png, …
├── dog/      …
└── sheep/    …
```

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Point the notebook at your dataset</b><br>
Put your dataset in Colab first: mount Google Drive (<code>from google.colab import drive</code>, then
<code>drive.mount("/content/drive")</code>) or upload a zip in the Files pane and unzip it. Then set
<code>MY_DATASET_DIR</code> to the folder that holds the class folders, e.g.
<code>"/content/drive/MyDrive/project/my_dataset"</code>, and run this section again.<br>
Leave it at <code>None</code> and the cells run on a small sample built from the Play Dataset, so you can
see what the output looks like first. Images may sit in sub-folders of a class folder; other files are
ignored. For a detection or segmentation project, put all images in one folder inside
<code>MY_DATASET_DIR</code>: the class counts then say nothing, but sizes, modes, copies and the split
still do (the box and mask checks are sections 8 and 9).
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
MY_DATASET_DIR = None  # e.g. "/content/drive/MyDrive/project/my_dataset" (one sub-folder per class)

The next cell checks the folder and lists every image with its class (its folder name). If
`MY_DATASET_DIR` is `None`, or the folder cannot be used, it builds the sample in `data/cache/my_dataset/`
instead: a few cleaned photos per class, plus **three planted problems** so the checklist has something to
find (an exact copy, a mirrored copy and a photo saved in grayscale). Very large datasets are sampled down
to `MY_MAX_IMAGES` photos, keeping the class shares, so the checks stay at a minute or two.

In [ ]:
# @title Read the folder, or build the sample { display-mode: "form" }
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
MY_SAMPLE_DIR = DATA_DIR / "cache" / "my_dataset"
MY_SAMPLE_PER_CLASS = 4 if SMOKE else 20  # photos per class in the sample (N-014: fewer in smoke mode)
MY_MAX_IMAGES = 2000  # larger datasets are sampled down to this many photos, stratified by class


def list_folder_per_class(root) -> pd.DataFrame:
    """One row per image file under `root`: columns path, file, class (the top-level folder it sits in)."""
    rows = []
    class_dirs = [p for p in Path(root).iterdir() if p.is_dir() and not p.name.startswith((".", "_"))]
    for class_dir in sorted(class_dirs):
        for f in sorted(class_dir.rglob("*")):
            if f.is_file() and f.suffix.lower() in IMAGE_SUFFIXES and not f.name.startswith("."):
                rows.append({"path": str(f), "file": str(f.relative_to(root)), "class": class_dir.name})
    return pd.DataFrame(rows, columns=["path", "file", "class"])


def build_my_sample(dest: Path, per_class: int, seed: int) -> Path:
    """Folder-per-class copy of a few clean Play Dataset photos, plus an exact copy, a mirror, a grayscale."""
    shutil.rmtree(dest, ignore_errors=True)  # rebuilt from scratch: re-running gives the same sample (N-013)
    rng = random.Random(seed)
    pool = clean_images[clean_images["flags"] == ""]
    picked = {}
    for c in CLASSES:
        of_class = sorted(pool.loc[pool["class"] == c, "path"])
        picked[c] = rng.sample(of_class, min(per_class, len(of_class)))
        (dest / c).mkdir(parents=True, exist_ok=True)
        for p in picked[c]:
            shutil.copyfile(p, dest / c / Path(p).name)
    first = {c: Path(picked[c][0]) for c in CLASSES if picked[c]}
    shutil.copyfile(first["cat"], dest / "cat" / f"{first['cat'].stem}_copy.jpg")
    with Image.open(first["dog"]) as im:
        ImageOps.mirror(im.convert("RGB")).save(dest / "dog" / f"{first['dog'].stem}_b.jpg", quality=92)
    with Image.open(first["bird"]) as im:
        im.convert("L").save(dest / "bird" / first["bird"].name, quality=92)
    return dest


my_source = "sample"
if MY_DATASET_DIR is not None and not SMOKE:
    _root = Path(str(MY_DATASET_DIR)).expanduser()
    if not _root.is_dir():
        print(f"❌ MY_DATASET_DIR = {MY_DATASET_DIR!r} is not a folder. Check the path in the Files pane.")
    elif (_listed := list_folder_per_class(_root)).empty:
        print(f"❌ No images in class sub-folders of {_root}. Expected {_root}/<class>/<image files>.")
    else:
        my_source, my_root, my_files = "yours", _root, _listed
if my_source == "sample":
    if MY_DATASET_DIR is not None:
        print("Running on the Play Dataset sample instead" + (" (smoke mode)." if SMOKE else "."))
    my_root = build_my_sample(MY_SAMPLE_DIR, MY_SAMPLE_PER_CLASS, SEED)
    my_files = list_folder_per_class(my_root)

if len(my_files) > MY_MAX_IMAGES:
    _n_all = len(my_files)
    my_files = pd.concat([g.sample(max(1, round(len(g) * MY_MAX_IMAGES / _n_all)), random_state=SEED)
                          for _, g in my_files.groupby("class")])
    print(f"⚠️ {_n_all} images found: the checks run on a stratified sample of {len(my_files)}.")
my_files = my_files.sort_values(["class", "file"]).reset_index(drop=True)
MY_CLASSES = sorted(my_files["class"].unique())
# Park classes keep their colour; any other class name borrows one from the same palette (L-003).
_palette = list(CLASS_COLORS.values())
MY_COLORS = {c: CLASS_COLORS.get(c, _palette[i % len(_palette)]) for i, c in enumerate(MY_CLASSES)}
print(f"source: {my_source} · {my_root} · {len(my_files)} images in {len(MY_CLASSES)} class folders")

**Q34.** 🟢 Before running the checklist on your dataset, write down what you expect: the largest and the
smallest class, how many sizes and colour modes, and whether you think there are any copies. (On the
sample you already know the planted problems; predict whether the checks will catch all three.)

The first check counts the images per folder, as section 4 did for the park.

In [ ]:
# Check 1: images per class, and the ratio between the largest and the smallest class.
my_counts = my_files["class"].value_counts().reindex(MY_CLASSES)
fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(MY_CLASSES) + 2), 3.2))
ax.bar(MY_CLASSES, my_counts.to_numpy(), color=[MY_COLORS[c] for c in MY_CLASSES])
ax.set_ylabel("images")
ax.set_title(f"Images per class folder ({len(my_files)} in total)")
ax.tick_params(axis="x", rotation=30 if len(MY_CLASSES) > 6 else 0)
plt.tight_layout()
plt.show()
my_imbalance = float(my_counts.max() / my_counts.min())
print(f"largest {my_counts.idxmax()} ({my_counts.max()}) · smallest {my_counts.idxmin()} ({my_counts.min()}) "
      f"· ratio {my_imbalance:.1f} : 1")

💡 A ratio of 2 or 3 to 1 is common and harmless if you watch the small class's recall; at 10 to 1 and
beyond, accuracy hides the small class completely (section 4's ⚠️ Pitfall). Ask whether the *expensive*
class is the small one.

The second check opens every file once: does it open, how large is it, and which colour mode is stored.

In [ ]:
# Check 2: readable? size and colour mode of every file (section 3's measure_image on a small thumbnail).
_my_rows = []
for _, r in my_files.iterrows():
    ok = is_readable(r["path"])
    stats = measure_image(r["path"], thumb=64)[0] if ok else {}
    _my_rows.append({"file": r["file"], "class": r["class"], "readable": ok, "width": stats.get("width"),
                  "height": stats.get("height"), "mode": stats.get("mode", "unreadable")})
my_stats = pd.DataFrame(_my_rows)
_ok = my_stats[my_stats["readable"]]

fig, ax = plt.subplots(figsize=(5.5, 4))
for c in MY_CLASSES:
    sub = _ok[_ok["class"] == c]
    ax.scatter(sub["width"], sub["height"], s=18, alpha=0.7, color=MY_COLORS[c], label=c)
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("Image sizes per class")
ax.legend(frameon=False, fontsize=8, ncol=2 if len(MY_CLASSES) > 6 else 1)
plt.tight_layout()
plt.show()
print(f"unreadable files: {int((~my_stats['readable']).sum())}")
print("colour modes:", " · ".join(f"{m} {n}" for m, n in my_stats["mode"].value_counts().items()))
print(f"distinct sizes: {_ok.groupby(['width', 'height']).ngroups} · "
      f"long side from {int(_ok[['width', 'height']].max(axis=1).min())} "
      f"to {int(_ok[['width', 'height']].max(axis=1).max())} px")
display(my_stats[my_stats["mode"] != "RGB"].head(10))

💡 One tight cluster of sizes means one source; a second cluster, or one class sitting apart from the
others, means a second source, and the model may learn the source instead of the animal (section 3's
detective beat). Every file that is not RGB is listed above: decide whether to convert it or drop it.

The third check is section 5's copy hunt: exact copies by file hash, and near, flipped and rotated copies
by comparing perceptual hashes of every image and its variants, with your `HASH_THRESHOLD` from section 5.

In [ ]:
# Check 3: exact, recompressed, flipped and rotated copies (all pairs, in blocks to keep memory small).
_readable = my_stats.index[my_stats["readable"]].to_numpy()
_my_paths = my_files.loc[_readable, "path"].tolist()
_md5 = [file_md5(p) for p in _my_paths]
_variant_bits = {v: [] for v in VARIANT_NAMES}
for p in _my_paths:
    for v, im in variants(load_thumb(p, THUMB_FOR_HASH)).items():
        _variant_bits[v].append(phash(im))
_variant_bits = {v: hash_bits(h) for v, h in _variant_bits.items()}
_plain = _variant_bits["original"]
_n = len(_my_paths)
_best = np.full((_n, _n), 65, dtype=np.int16)  # smallest distance over the variants, and which variant
_best_v = np.zeros((_n, _n), dtype=np.int8)
for k, v in enumerate(VARIANT_NAMES):
    for start in range(0, _n, 256):
        d = hash_distances(_variant_bits[v][start:start + 256], _plain)
        better = d < _best[start:start + 256]
        _best[start:start + 256][better] = d[better]
        _best_v[start:start + 256][better] = k

_pairs = []
for i, j in zip(*np.nonzero(np.triu(_best <= HASH_THRESHOLD, k=1)), strict=True):
    a, b = _readable[i], _readable[j]
    kind = "exact" if _md5[i] == _md5[j] else KIND_OF_VARIANT[VARIANT_NAMES[_best_v[i, j]]]
    _pairs.append({"a": my_files.at[a, "file"], "b": my_files.at[b, "file"], "kind": kind,
                   "distance": int(_best[i, j]),
                   "same class": my_files.at[a, "class"] == my_files.at[b, "class"],
                   "ia": a, "ib": b})
my_copies = pd.DataFrame(_pairs, columns=["a", "b", "kind", "distance", "same class", "ia", "ib"])
print(f"{len(my_copies)} copy pairs at HASH_THRESHOLD = {HASH_THRESHOLD}:",
      " · ".join(f"{k} {n}" for k, n in my_copies["kind"].value_counts().items()) or "none")
print(f"pairs whose two files sit in different class folders (one label is wrong): "
      f"{int((~my_copies['same class']).sum())}")
display(my_copies.drop(columns=["ia", "ib"]).head(10))

_show = my_copies.head(4)
if len(_show):
    fig, axes = plt.subplots(len(_show), 2, figsize=(5, 2.4 * len(_show)), squeeze=False)
    for row, (_, pr) in zip(axes, _show.iterrows(), strict=True):
        for ax, idx in zip(row, [pr["ia"], pr["ib"]], strict=True):
            ax.imshow(load_thumb(my_files.at[idx, "path"], 256))
            ax.set_title(f"{my_files.at[idx, 'file']}\n{pr['kind']}", fontsize=8)
            frame(ax, MY_COLORS[my_files.at[idx, "class"]])
    plt.tight_layout()
    plt.show()

💡 Every pair is a decision for a human: keep one file of each pair, and look twice at pairs from different
class folders, because one of the two labels is wrong. On a dataset of near-identical shots (a fixed camera,
a microscope) many pairs are genuine different photos: lower `HASH_THRESHOLD` in section 5 and rerun.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Splitting before deduplicating</b><br>
If the two files of a copy pair land on different sides of the split, the test set contains photos the
model has already seen, and the score is too good (section 6). The split below therefore keeps every copy
group together, even before you have removed the copies.
</font></td></tr></table>

The last check suggests a split: it groups the copies, then makes a stratified, group-aware split.

In [ ]:
# Check 4: a split suggestion. Copies form groups that stay on one side; the split is stratified by class.
import warnings

from sklearn.model_selection import StratifiedGroupKFold

_parent = list(range(len(my_files)))  # union-find: every copy pair joins its two files' groups


def _find(i: int) -> int:
    while _parent[i] != i:
        _parent[i] = _parent[_parent[i]]
        i = _parent[i]
    return i


for _, pr in my_copies.iterrows():
    _parent[_find(pr["ia"])] = _find(pr["ib"])
my_groups = np.array([_find(i) for i in range(len(my_files))])

_smallest = int(my_counts.min())
_n_splits = min(5, max(2, _smallest))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a class with fewer photos than folds only triggers a warning
    _splitter = StratifiedGroupKFold(n_splits=_n_splits, shuffle=True, random_state=SEED)
    _train_idx, _test_idx = next(_splitter.split(my_files, my_files["class"], groups=my_groups))
my_split = my_files[["file", "class"]].copy()
my_split["split"] = "train"
my_split.loc[_test_idx, "split"] = "test"
_split_copies = sum(my_split.at[pr["ia"], "split"] != my_split.at[pr["ib"], "split"]
                    for _, pr in my_copies.iterrows())
display(pd.crosstab(my_split["class"], my_split["split"]).reindex(MY_CLASSES))
_test_small = int((my_split["split"] == "test").groupby(my_split["class"]).sum().min())
print(f"test share {len(_test_idx) / len(my_files):.0%} · copy pairs across train/test: {_split_copies}")
if _test_small < 10:
    print(f"💡 the smallest class has only {_test_small} test images: prefer stratified {_n_splits}-fold "
          "cross-validation with the same groups, and report the mean and the spread over the folds.")
else:
    print("💡 a stratified 80/20 hold-out with these groups is enough; keep the test half for the end.")
print("If photos share a camera, a session, a burst or a patient, add that to the groups (section 6).")
_split_csv = DATA_DIR / "cache" / "my_split.csv"
my_split.to_csv(_split_csv, index=False)
print(f"split saved to {_split_csv}")

💡 The split is a suggestion from what the folders show. Only you know what else ties photos together,
such as one camera, one day, one patient; section 6 showed that a split which ignores such groups can
halve the error it reports.

The checklist in one place, so you can paste it into your project notes:

In [ ]:
# The report card for this dataset.
_n_unreadable = int((~my_stats["readable"]).sum())
_n_not_rgb = int(((my_stats["mode"] != "RGB") & my_stats["readable"]).sum())
my_report = [
    ("✅" if my_imbalance < 3 else "⚠️", f"classes: {len(MY_CLASSES)}, imbalance {my_imbalance:.1f} : 1"),
    ("✅" if _n_unreadable == 0 else "⚠️", f"unreadable files: {_n_unreadable}"),
    ("✅" if _n_not_rgb == 0 else "⚠️", f"files not stored as RGB: {_n_not_rgb}"),
    ("✅" if len(my_copies) == 0 else "⚠️", f"copy pairs: {len(my_copies)} "
     f"({int((~my_copies['same class']).sum())} across class folders)"),
    ("✅" if _split_copies == 0 else "⚠️", f"split: {len(_test_idx)} test / {len(_train_idx)} train, "
     f"copy pairs across the split: {_split_copies}"),
]
for mark, line in my_report:
    print(mark, line)
RESULTS["swap_in_ok"] = bool(len(my_files) > 0 and len(my_stats) == len(my_split) == len(my_files))
RESULTS["swap_in"] = {"source": my_source, "n_images": int(len(my_files)), "n_classes": len(MY_CLASSES),
                      "copies": int(len(my_copies)), "not_rgb": _n_not_rgb}

**Worked answer (Q34, on the sample).** Every folder holds the same number of photos, plus the planted
copies in cat and dog, so the ratio is close to 1 : 1; that is a property of the sample, not of the park.
All sizes fall in the Play Dataset's few clusters (long side 640 or 500 px). The grayscale bird shows up
as one `L` file in check 2, and check 3 finds the exact copy of the cat and the mirrored dog as an `exact`
and a `flipped` pair, both inside one folder; the split keeps each pair on one side. If a check missed one
of them, look at the `HASH_THRESHOLD` you set in section 5.

### The guiding questions

These questions are the notebook in short. Each one has the park's answer, and an empty line for yours.

**Q35.** 🟢 Double-click the next cell and answer each guiding question for your project dataset, using
the output of the checklist above and what you know about where the data comes from.

1. **What is the task, and which mistake is the expensive one?**
   - *The park:* classification of who is in front of the camera; the expensive mistake is a missed dog
     in the sheep paddock, so the metric is recall on dog (section 7).
   - *Your dataset:*
2. **Do the images look like what the model will see in use?**
   - *The park:* only partly. COCO has close-ups, sofas and people; a fixed paddock camera sees small
     animals on grass (section 2).
   - *Your dataset:*
3. **How balanced are the classes, and is the expensive class a small one?**
   - *The park:* cat and dog 130 photos each, sheep 59, about 2 : 1. Dog is not rare, but accuracy would
     still hide a weak recall on dog (section 4).
   - *Your dataset:*
4. **Which sizes, aspect ratios and colour modes are there, and what must be resized or converted?**
   - *The park:* nearly every photo has a long side of 640 or 500 px, in landscape and portrait; files
     that are not RGB get converted (sections 3 and 5).
   - *Your dataset:*
5. **Are there copies, flipped or rotated copies, broken files or doubtful labels?**
   - *The park:* a few copies and odd files, removed before splitting (section 5's "removed N, kept M");
     photos with several animals make the label ambiguous (section 4).
   - *Your dataset:*
6. **What ties photos together, and which split keeps those groups apart?**
   - *The park:* COCO has no groups, so a stratified split after deduplication; the trail cameras have
     bursts and locations, so they are split by camera (section 6).
   - *Your dataset:*
7. **What must be fixed before training?**
   - *The park:* drop the copies, convert the files that are not RGB, decide the label of multi-animal
     photos, and resize without distorting the aspect ratio (section 3's 🔀 Callout).
   - *Your dataset:*

**Takeaway.** The checklist takes a few minutes on any folder-per-class dataset; the answers to the
questions are what you bring to next week's training, together with `data/cache/my_split.csv`.

## 12. Wrap-up & next week

**Objective:** look back at the three things you set out to do, see the park's numbers for each, and know
where next week's training starts.

Nothing was trained today, and yet we now know a lot about whether the park's data is fit for its three
jobs. The next cell collects the numbers the earlier sections measured, grouped by the three outcomes from
the title banner.

In [ ]:
# The park's numbers from today, one block per learning outcome.
_photos = RESULTS["images_per_class"]
_boxes = RESULTS["boxes_per_class"]
_pixels = RESULTS["pixels_per_class"]
_big, _small = max(_photos, key=_photos.get), min(_photos, key=_photos.get)
recap = {
    "1 · Inspect and judge the fit": [
        f"{RESULTS['n_images']} photos of {len(CLASSES)} classes, every one measured "
        f"({RESULTS['image_stats_n']} rows of image stats)",
        f"largest class {_big} ({_photos[_big]}), smallest {_small} ({_photos[_small]}): "
        f"{_photos[_big] / _photos[_small]:.1f} : 1",
    ],
    "2 · Find and fix errors": [
        f"removed {RESULTS['n_removed']}, kept {RESULTS['n_clean']} "
        "(copies and broken files, before any split)",
    ],
    "3 · Split, metric, and the three tasks": [
        f"test photos with a burst sibling in train: random split {RESULTS['sibling_rate_random']:.0%}, "
        f"split by camera {RESULTS['sibling_rate_by_camera']:.0%}",
        f"prepared predictions: accuracy {RESULTS['accuracy']:.0%}, "
        f"recall on dog {RESULTS['recall_dog']:.0%}",
        f"bird: {_photos['bird'] / sum(_photos.values()):.0%} of photos, "
        f"{_boxes['bird'] / sum(_boxes.values()):.0%} of boxes, {_pixels['bird']:.0%} of pixels",
    ],
}
for outcome, lines in recap.items():
    print(outcome)
    for line in lines:
        print("   ", line)

💡 Every line is a decision you can now defend: what to fix, how to split, what to measure, and in which
unit (photo, box or pixel) to count. In smoke mode the numbers are tiny; the full run gives the park's real
ones.

### Takeaways

<table width="100%"><tr><td bgcolor="#0B6E79"><font color="#FFFFFF">
<b>You can now:</b>
<ol>
<li><b>Inspect an image dataset qualitatively and quantitatively, and say whether it fits a task.</b><br>
Look at many photos before computing anything; then measure sizes, modes, colours and class counts. The
outliers point at bugs, and a photo the park's camera would never take points at a poor fit.</li>
<li><b>Find and fix common dataset errors before they leak into training.</b><br>
Hash the files for exact copies, hash the pixels (and their flipped and rotated variants) for near copies,
open every file once, and let a human judge the least typical labels and the ugliest masks. Write down
"removed N, kept M".</li>
<li><b>Choose a split and a metric that match the use case, and explain how imbalance, object size and
leakage look different for classification, detection and segmentation.</b><br>
Deduplicate, then split by whatever ties photos together (a camera, a burst). Pick the metric from the
expensive mistake: recall on dog for the paddock alert, counting error for the headcount, IoU per class
for the outlines. Always say what you count: photos, boxes or pixels.</li>
</ol>
</font></td></tr></table>

### Next week: training

Next week the cleaned table and the split go into a model. Karpathy's recipe continues the same way:
first an end-to-end skeleton with a dumb baseline, then a model that overfits, then regularisation.
Two things from today come back straight away.

**Data augmentation.** The flipped, rotated and recompressed copies you hunted in section 5 are what
augmentation makes *on purpose*: new training views of the same photo. Augment only the training set,
and only after the split, or the copies leak into the test set again.
- Neptune.ai, [Data augmentation in Python](https://neptune.ai/blog/data-augmentation-in-python):
  an overview of the techniques and libraries.
- Ultralytics, [YOLO data augmentation](https://docs.ultralytics.com/guides/yolo-data-augmentation/):
  what each augmentation does for detection, where boxes must move with the pixels.

**Tracking experiments (MLOps).** Training means many runs with different settings; a tracking tool
records each run's settings, curves and scores so you can compare them later.
- [Weights & Biases](https://wandb.ai/)
- [TensorBoard](https://www.tensorflow.org/tensorboard)
- [Neptune.ai](https://neptune.ai/)
- [MLflow](https://mlflow.org/)

**Bring along:** your answers to the guiding questions of section 11 and your split
(`data/cache/my_split.csv`, or the one you choose by hand).